# Emotion-aware recommendation over WikiArt ∩ ArtEmis

Two recommendation surfaces for an online art catalogue — a personalised feed and an
item-to-item "More like this" module — built on one frozen corpus of 20,833 paintings that
carries WikiArt's artist and style labels together with ArtEmis's crowd-annotated emotional
responses. Feedback is **implicit and binary** throughout: an item is shown, and it is either
engaged with or it is not.

## List of Contents

| § | Section | Track |
|---|---|---|
| **1** | Setup and data preparation — images, ArtEmis join, per-painting aggregation, **corpus freeze** | both |
| **2** | Synthetic *implicit* feedback and user profiles — generative model, validation, sensitivity sweeps (content ridge vs explicit MF vs implicit ALS) | A |
| **3** | Baselines: Random and Popularity | A |
| **4** | Item-to-item "More like this": B0 Random, B1 Random-within-style | B |
| **5** | Content-based recommendation from metadata (user profiles; B2) | A + B |
| **6** | Bayesian ranking — engagement quality instead of exposure volume | A |
| **7** | Thompson sampling — an *online* experiment | A |
| **8** | Item-kNN — collaborative similarity on both tracks (B3); §8.2 implicit ALS, tuned on the validation split (B4); §8.3 the per-user metadata ridge; §8.4 the ridge–ALS hybrid and the tuning-shape check; §8.5 BPR; §8.6 the coverage frontier | A + B |
| **9** | CLIP ViT-B/32 visual similarity (I1); §9.1 CLIP as a user profile, §9.2 the hybrid, §9.3 within-list diversity, §9.4 the hybrid weight on held-out queries, §9.5 confidence intervals and paired tests on every rung | A + B |

Track **A** = "what should this user see next" (with §2's simulated engagement log).
Track **B** = "what else is like this painting" (real WikiArt / ArtEmis labels only).

## 1. Setup and Data preparation

### Importing

In [1]:
import pandas as pd
#import kagglehub
import io, os, zipfile
from pathlib import Path
from multiprocessing import Pool
from PIL import Image
import re
import hashlib
import numpy as np
from pathlib import Path


def sigmoid(x):
    """Logistic link, written via tanh so it never overflows on large |x|.

    Shared by §2's observation model and §7's bandit environment, which is why it lives
    here rather than in either section."""
    return 0.5 * (1.0 + np.tanh(0.5 * np.asarray(x, dtype=np.float64)))

In [2]:
# Download latest version 31 GB
#path = kagglehub.dataset_download("steubk/wikiart")
#print("Path to dataset files:", path)

In [3]:
# dataset avaialble at https://www.kaggle.com/datasets/steubk/wikiart?select=wclasses.csv
wikiart = pd.read_csv("/Users/diazm/Documents/HSLU/07_SS2026/Recommender/Code/classes.csv")
print(wikiart.shape, wikiart.columns.tolist())
print(wikiart['genre'].value_counts())

(80042, 9) ['filename', 'artist', 'genre', 'description', 'phash', 'width', 'height', 'genre_count', 'subset']
genre
['Impressionism']                                                           12847
['Realism']                                                                 10534
['Romanticism']                                                              6896
['Expressionism']                                                            6280
['Post Impressionism']                                                       6274
                                                                            ...  
['Color Field Painting', 'Color Field Painting', 'Color Field Painting']        1
['Cubism', 'Cubism']                                                            1
['New Realism', 'New Realism']                                                  1
['Pointillism', 'Pointillism']                                                  1
['Pointillism', 'Art Nouveau Modern']                          

### 1.2 Image preparation — resizing out of the archive

`kagglehub` leaves a 31 GB zip (`~/.cache/kagglehub/.../1.archive`) holding 81,444 JPEGs plus
`classes.csv` / `wclasses.csv`. Extracting it as-is needs ~63 GB (archive + output), which is
what exhausted the disk on the first attempt.

Instead we stream each entry out of the zip, downscale it, and never materialise the originals:
**31.4 GB → 2.6 GB, a 12× reduction that discards nothing CLIP consumes.**

Two deliberate choices:

- **Short side → 256, not longest side.** CLIP's own transform is
  `Resize(224)` → `CenterCrop(224)`, and its `Resize` scales the *shorter* edge. Capping the
  longest edge instead would leave the short edge below 224, forcing CLIP to upscale and adding
  blur it never saw in training. 256 leaves headroom above the 224 the model needs.
- **No cropping.** Aspect ratio is preserved and the full canvas kept. Sampled ratios run to
  **6.34:1** — center-cropping such a painting to square discards ~84% of it. Storing uncropped
  means crop strategy (center / squash / pad) stays a choice at inference time rather than
  being baked into the data.

In [4]:
"""
Stream WikiArt images out of the kagglehub archive, downscaled, aspect preserved.
Run once. Resumable: re-running skips files already written, so an interrupted
run costs nothing. Writes via a .part temp + os.replace, so a crash can never
leave a truncated JPEG behind.
"""

ARCHIVE    = Path.home() / ".cache/kagglehub/datasets/steubk/wikiart/1.archive"
IMG_DIR    = Path("wikiart_256")
SHORT_SIDE = 256
QUALITY    = 90
_zf = None

def _init(archive):
    global _zf
    _zf = zipfile.ZipFile(archive)

def _one(args):
    name, out_dir, short_side, quality = args
    dst = Path(out_dir) / name
    try:
        im = Image.open(io.BytesIO(_zf.read(name)))
        im.draft("RGB", None)                    # fast DCT-scaled JPEG decode
        im = im.convert("RGB")
        w, h = im.size
        s = short_side / min(w, h)
        if s < 1.0:                              # downscale only, never upscale
            im = im.resize((max(1, round(w*s)), max(1, round(h*s))), Image.LANCZOS)
        dst.parent.mkdir(parents=True, exist_ok=True)
        tmp = dst.with_suffix(dst.suffix + ".part")
        im.save(tmp, "JPEG", quality=quality, optimize=True)
        os.replace(tmp, dst)                     # atomic
        return True
    except Exception as e:
        print(f"FAIL {name}: {e!r}")
        return False

def build_image_set():
    if not ARCHIVE.exists():
        print(f"archive gone (deleted after extraction) — using existing {IMG_DIR}/")
        return
    IMG_DIR.mkdir(exist_ok=True)
    with zipfile.ZipFile(ARCHIVE) as zf:
        names = [n for n in zf.namelist() if n.lower().endswith((".jpg", ".jpeg", ".png"))]
        for meta in ("classes.csv", "wclasses.csv"):
            (IMG_DIR / meta).write_bytes(zf.read(meta))
    todo = [n for n in names if not (IMG_DIR / n).exists()]      # resumable
    print(f"{len(names):,} images | {len(names)-len(todo):,} done | {len(todo):,} to do")
    if todo:
        jobs = ((n, str(IMG_DIR), SHORT_SIDE, QUALITY) for n in todo)
        with Pool(max(1, (os.cpu_count() or 4) - 2), _init, (ARCHIVE,)) as pool:
            ok = sum(pool.imap_unordered(_one, jobs, chunksize=32))
        print(f"resized {ok:,}")

# NOTE: multiprocessing on macOS/Python 3.13 defaults to *spawn*, so this must be
# guarded — in a notebook the functions above must live in an importable module,
# or be run via the standalone `resize_wikiart.py`.
if __name__ == "__main__":
    build_image_set()

n_img = sum(1 for _ in Path("wikiart_256").rglob("*.jpg"))
print(f"images available: {n_img:,}")

archive gone (deleted after extraction) — using existing wikiart_256/


images available: 81,444


### 1.3 Datasets downloads


In [5]:
# dataset avaialble at https://www.kaggle.com/datasets/samamostafa03/artemis?select=artemis_dataset_release_v0.csv
artemis = pd.read_csv("/Users/diazm/Documents/HSLU/07_SS2026/Recommender/Code/artemis_dataset_release_v0.csv")
print(artemis.shape, artemis.columns.tolist())

(454684, 5) ['art_style', 'painting', 'emotion', 'utterance', 'repetition']


In [6]:
print(artemis['emotion'].value_counts())

emotion
contentment       126134
awe                72927
something else     52962
sadness            49061
amusement          45336
fear               41577
excitement         37636
disgust            22411
anger               6640
Name: count, dtype: int64


In [7]:
print(artemis['painting'].value_counts()) # one row per annotation, ~5.8 per painting

painting
ion-pacea_midnight-sun                                                  58
jackson-pollock_number-17-1949                                          58
a.y.-jackson_spring-caribou-country-1949                                57
childe-hassam_the-flag-outside-her-window-april-aka-boys-marching-by    56
george-morland_thunderstorm                                             56
                                                                        ..
pieter-bruegel-the-elder_go-ye-into-the-emmaus                           5
albrecht-durer_the-small-chariot                                         5
albrecht-durer_sitting-mary-with-child                                   5
jan-van-hemessen_tobias-restores-his-father-s-sight                      5
albrecht-durer_standing-apostle-1508                                     5
Name: count, Length: 80031, dtype: int64


In [8]:
print(artemis.groupby(['art_style','painting']).size().describe())  # annotations per artwork

count    80031.000000
mean         5.681348
std          4.025648
min          5.000000
25%          5.000000
50%          5.000000
75%          5.000000
max         58.000000
dtype: float64


### 1.4 Datasets merged

In [9]:
IMG_DIR  = Path("wikiart_256")
EMOTIONS = ["amusement", "awe", "contentment", "excitement",
            "anger", "disgust", "fear", "sadness", "something else"]

def norm(s: str) -> str:
    """Fold to an ASCII skeleton — kills mojibake bytes and apostrophes alike."""
    return re.sub(r"[^a-z0-9_/.-]", "", s.lower())

# `artemis` is loaded above. Work on a copy under a new name so this cell stays
# idempotent — it filters rows, and rebinding `artemis` would make a second run
# operate on the already-filtered frame.
ann = artemis.copy()
ann["key"] = ann.art_style + "/" + ann.painting + ".jpg"

disk = sorted(str(p.relative_to(IMG_DIR)) for p in IMG_DIR.rglob("*.jpg"))
by_skeleton = {}
for d in disk:
    by_skeleton.setdefault(norm(d), d)
assert len(by_skeleton) == len(disk), "skeleton collision — matching would be ambiguous"

exact = set(ann.key) & set(disk)
ann["file"] = ann.key.map(lambda k: by_skeleton.get(norm(k)))

n_paintings = ann.key.nunique()
print(f"ARTEMIS paintings : {n_paintings:,}")
print(f"  exact match     : {len(exact):,} ({len(exact)/n_paintings*100:.2f}%)")
print(f"  ASCII-recovered : {ann.loc[ann.file.notna(),'key'].nunique() - len(exact):,}")
print(f"  unmatched       : {ann.loc[ann.file.isna(),'key'].nunique():,}")

ann = ann[ann.file.notna()].copy()

ARTEMIS paintings : 80,031
  exact match     : 79,274 (99.05%)
  ASCII-recovered : 756
  unmatched       : 1


### 1.4.1 Per-painting aggregation and the artist-identity fix

ARTEMIS gives ~5.68 annotations per painting, so we collapse it to one row each: the emotion
distribution over the 9 labels, its entropy (how contested the reading is), and the raw
utterances kept as a list for the text side.

The same artist can arrive under two spellings — the mojibake in
classes.csv and the differently-mangled filename. Sorolla alone had 350 works split across
two labels. Because relevance here is the same-artist proxy, leaving that split would score
those works as unrelated to their own author and silently deflate recall. Grouping on the ASCII
skeleton collapses 1,124 → 1,119 artists, with none still split.

In [ ]:
# per-painting emotion distribution ----------------------------------
counts = (ann.groupby(["file", "emotion"]).size()
                 .unstack(fill_value=0)
                 .reindex(columns=EMOTIONS, fill_value=0))
n_ann = counts.sum(axis=1)
props = counts.div(n_ann, axis=0)
props.columns = [f"emo_{c.replace(' ', '_')}" for c in props.columns]

p    = props.to_numpy()
safe = np.where(p > 0, p, 1.0)                  # log(1)=0; masked out anyway
entropy = -(np.where(p > 0, p * np.log(safe), 0.0)).sum(axis=1)

agg = props.copy()
agg["n_annotations"]    = n_ann
agg["dominant_emotion"] = counts.idxmax(axis=1)
agg["emotion_entropy"]  = entropy
agg["utterances"]       = ann.groupby("file").utterance.apply(list)
agg = agg.reset_index()

# ---- catalogue side ------------------------------------------------------
catalog = pd.DataFrame({"file": disk})
catalog["style"] = catalog.file.str.split("/").str[0]
catalog["_n"]    = catalog.file.map(norm)

cls = pd.read_csv(IMG_DIR / "classes.csv")
cls["_n"] = cls.filename.map(norm)
catalog = catalog.merge(
    cls.drop_duplicates("_n")[["_n", "artist", "genre", "description",
                               "phash", "width", "height", "subset"]],
    on="_n", how="left")
catalog = catalog.rename(columns={"description": "title",
                                  "width": "orig_width", "height": "orig_height"})

# classes.csv misses 1,402 images — recover the artist from the filename
missing = catalog.artist.isna()
catalog.loc[missing, "artist"] = (catalog.loc[missing, "file"]
                                  .str.split("/").str[1].str.split("_").str[0]
                                  .str.replace("-", " "))
catalog = catalog.drop(columns="_n")

# ---- canonical artist identity --------------------------
skeleton = catalog.artist.str.lower().str.replace(r"[^a-z0-9]", "", regex=True)
pick = (pd.DataFrame({"sk": skeleton, "artist": catalog.artist,
                      "na": catalog.artist.str.count(r"[^\x00-\x7f]")})
          .groupby(["sk", "artist"], as_index=False)
          .agg(na=("na", "first"), n=("artist", "size"))
          .sort_values(["sk", "na", "n"], ascending=[True, True, False])
          .drop_duplicates("sk").set_index("sk").artist)
before = catalog.artist.nunique()
catalog["artist"] = skeleton.map(pick)
print(f"artist identities merged: {before} -> {catalog.artist.nunique()}")

# ---- final table ---------------------------------------------------------
paintings = catalog.merge(agg, on="file", how="left")
paintings["has_artemis"]   = paintings.n_annotations.notna()
paintings["n_annotations"] = paintings.n_annotations.fillna(0).astype(int)

paintings.to_parquet("merged_paintings.parquet", index=False)
paintings.drop(columns="utterances").to_csv("merged_paintings.csv", index=False)
ann[["file", "art_style", "emotion", "utterance", "repetition"]] \
       .to_parquet("artemis_utterances.parquet", index=False)

print(f"paintings : {len(paintings):,}")
print(f"  with ARTEMIS : {paintings.has_artemis.sum():,} "
      f"({paintings.has_artemis.mean()*100:.2f}%)")
print(f"  image-only   : {(~paintings.has_artemis).sum():,}")
print(f"artists   : {paintings.artist.nunique():,}")
print(f"utterances: {len(ann):,}")

artist identities merged: 1124 -> 1119


paintings : 81,444
  with ARTEMIS : 80,030 (98.26%)
  image-only   : 1,414
artists   : 1,119
utterances: 454,679


#### Sanity checks

Guards on the properties later stages assume. `emotion_entropy` in particular is easy to get
wrong: `np.log(p, where=p>0)` leaves *uninitialised memory* where `p == 0`, and `0 * nan` is
`nan`, so the masked form above is deliberate.

In [11]:
emo_cols = [c for c in paintings.columns if c.startswith("emo_")]
annotated = paintings[paintings.has_artemis]

assert np.allclose(annotated[emo_cols].sum(axis=1), 1.0),        "emotion props must sum to 1"
assert np.isfinite(annotated.emotion_entropy).all(),             "entropy must be finite"
assert paintings.artist.notna().all(),                           "artist is the relevance proxy"
assert paintings.file.is_unique,                                 "one row per painting"
sk = paintings.artist.str.lower().str.replace(r"[^a-z0-9]", "", regex=True)
assert paintings.groupby(sk).artist.nunique().max() == 1,        "artist identity still split"

works = paintings.artist.value_counts()
print(f"artists                : {len(works):,}")
print(f"  with >=2 works       : {(works >= 2).sum():,}")
print(f"  singletons (no same-artist positive possible): {(works == 1).sum()}")
print(f"entropy range          : {annotated.emotion_entropy.min():.3f} – "
      f"{annotated.emotion_entropy.max():.3f}  (ln 9 = {np.log(9):.3f})")
print()
#print("dominant emotion:")
print(paintings.dominant_emotion.value_counts().to_string())

artists                : 1,119
  with >=2 works       : 1,095
  singletons (no same-artist positive possible): 24
entropy range          : -0.000 – 2.164  (ln 9 = 2.197)

dominant_emotion
contentment       28198
awe               15687
amusement         11513
sadness            6728
fear               6507
something else     5093
excitement         3704
disgust            2226
anger               374


### 1.5 Corpus construction

The 81,444-image WikiArt dump is not the evaluation population. Five steps cut it to a frozen
corpus on which the emotion signal is stable, the same-artist relevance proxy behaves, and
ties do not resolve by row order. **The order matters**: ArtEmis coverage is the binding
constraint, so it is applied first.

**1. Artworks with ≥5 ArtEmis annotations.** A 9-dimensional emotion distribution estimated
from one or two annotators is noise. This filter is a *guarantee rather than a reduction*:
ArtEmis sampled at least 5 annotators for every painting it covers, so all 80,030 annotated
works pass and the 1,414 image-only works drop. It is asserted anyway, because it is a
precondition the later stages assume and it would fail silently if the merge changed.

**2. A fixed list of styles.** Restricting to 8 styles brings the corpus to ~20K works. The
list is committed in code below and chosen on three criteria:

- *No mega-style.* Impressionism (12,867) and Realism (10,726) are each large enough to
  dominate a 20K corpus on their own. Both are excluded; the largest surviving style is
  Expressionism at 19.7%.
- *Coverage of all three target emotions.* A corpus is useless here if one of awe,
  contentment or amusement is rare. Amusement is the scarce one — it is the dominant label
  for only 14% of the annotated set — so Naive Art, Pop Art and Art Nouveau are included
  specifically to carry it. The chosen list scores awe 0.163 / contentment 0.249 /
  amusement 0.115 in mean annotator proportion, against 0.097 for a size-optimal list that
  ignored balance.
- *Historical span.* Baroque through Pop Art, so style is a meaningful content feature
  rather than a proxy for one period.

**3. Deduplication, two passes.** WikiArt holds multiple scans of the same painting and
separate panels of polyptychs. Both would be scored as *relevant* by the same-artist proxy in
§4 while telling a browsing user nothing, so they are removed before anything is measured:

- **perceptual hash**, from the `phash` column of `classes.csv`;
- **exact title within artist**, which catches re-scans whose colour balance or crop moves
  the hash.

Two caveats, both handled in code. First, `phash` is missing for 388 of these rows, and a
*missing* hash is not evidence of a duplicate — an unguarded `duplicated()` would collapse all
388 onto one row. Second, this pair of passes is nearly a no-op on this corpus: it removes
**two** pairs and no exact-title pair at all. Both are cross-artist — one image filed under
both Camille Corot and Thomas Cole, and a Hodler / Cole pair that may be a genuine hash
collision rather than a duplicate. Two rows are worth the code anyway, because a cross-artist
duplicate is a guaranteed false *negative* for the relevance proxy, and because the guard is
what lets §4's numbers be read as measuring retrieval rather than scanning artefacts. It also
sets the expectation correctly for anyone reading the §10 note on series collapse: hashing
removes re-scans, and removes nothing at all of the ten-Monet-haystacks problem.

**Deduplication is by hash and title only, never by embedding distance.** Using a method under
evaluation to build the corpus would bias the comparison: near-duplicates are almost always
same-artist, so removing them by CLIP cosine would selectively delete CLIP's easiest hits
before §9 ever runs.

**4. Artists with 10–200 works in the surviving set.** Relevance in the item-to-item
evaluation is the same-artist proxy, which makes this filter an evaluation-validity
requirement rather than a cleanup step. Below 10 works, recall@k is computed against so few
positives that it swings wildly between queries. Above 200, a handful of prolific artists
supply most of the positives for every query and the metric measures those artists rather
than the recommender.

The upper bound is the expensive one, and the cost should be stated plainly: it removes
Van Gogh, Picasso, Dürer, Chagall, Hokusai, Warhol and the other most canonical names in
the corpus. `CAP_PROLIFIC` below switches to subsampling those artists down to 200 works
instead of dropping them, which keeps the same per-artist ceiling while retaining the
canon; the results reported in this project use the drop, as specified.

**5. One seeded shuffle, then freeze.** The surviving rows are shuffled once with
`CORPUS_SEED` and the order is never touched again. WikiArt file paths
are `Style/artist-name_title.jpg`, so sorting by filename groups the corpus by style and then
by artist: before the shuffle the 472 artists occupied just 671 contiguous runs of rows.
Every ranker in §3–§9 resolves tied scores through `argpartition`/`argsort`, which order by
ascending index — so on an artist-sorted corpus, every tie is decided in favour of whichever
candidate happens to sit near the query's own artist block. Ties are not rare: B2's 17
dimensions leave ~11 candidates sharing the rank-10 score of a typical query, and §6's raw
engagement rate ties hundreds of items at rank 10.

The measured cost of leaving it unshuffled turns out to be small on this corpus — B2's NDCG@10
moves in the third decimal — so this is a **guard**, not a correction of a large error. It is
still worth having: without it, a number that ought to depend only on the scoring function
depends silently on the order of a CSV, and nothing in the results would reveal it. After the
shuffle, row order carries no artist signal (20,756 runs over 472 artists), and §3 adds a
seeded draw on top so that tied candidates rotate between queries instead of one item winning
every list.

`item_id` is assigned from this frozen order in §2 and indexes the corpus everywhere
afterwards, so re-running §1.5 with a different seed changes every number downstream.

In [ ]:
# ============================ corpus construction ========================
MIN_ANNOTATIONS = 5
ARTIST_MIN, ARTIST_MAX = 10, 200
CAP_PROLIFIC = False        # True: subsample artists with more than 200 works down to 200 instead of dropping them
CORPUS_SEED  = 20260826     # the freeze shuffle

# Committed style list — 8 styles, Baroque through Pop Art. Changing this changes the
# corpus and invalidates every number downstream, so it is fixed here and nowhere else.
CORPUS_STYLES = [
    "Art_Nouveau_Modern",       # 4,272 works   amusement-carrying, turn of the century
    "Baroque",                  # 4,238         awe-heavy, pre-modern
    "Expressionism",            # 6,636         modern, emotionally contested
    "Naive_Art_Primitivism",    # 2,404         highest amusement outside Pop Art
    "Pop_Art",                  # 1,483         highest amusement in the corpus
    "Post_Impressionism",       # 6,417         contentment-heavy
    "Romanticism",              # 6,901         awe-heavy
    "Symbolism",                # 4,233         awe-heavy
]

# 1. annotation floor          2. committed styles
step1 = paintings[paintings.has_artemis & (paintings.n_annotations >= MIN_ANNOTATIONS)]
step2 = step1[step1["style"].isin(CORPUS_STYLES)].sort_values("file")

# 3. deduplication, hash then title. The notna() guards are load-bearing: `phash` and `title`
#    are missing for 388 rows, and pandas treats NaN == NaN as a duplicate, so an unguarded
#    call would collapse all 388 onto a single row.
dup_hash  = step2.phash.notna() & step2.phash.duplicated()
step3     = step2[~dup_hash]
dup_title = step3.title.notna() & step3.duplicated(["artist", "title"])
step3     = step3[~dup_title]

# 4. artist band — counted AFTER deduplication, so no artist is pushed below the floor by it
works = step3.artist.value_counts()
if CAP_PROLIFIC:
    keep  = works[works >= ARTIST_MIN].index
    corpus = (step3[step3.artist.isin(keep)]
              .groupby("artist", group_keys=False)[step3.columns]
              .apply(lambda g: g.sample(min(len(g), ARTIST_MAX), random_state=0)))
else:
    keep  = works[(works >= ARTIST_MIN) & (works <= ARTIST_MAX)].index
    corpus = step3[step3.artist.isin(keep)]

# 5. freeze: one seeded shuffle, so that row order carries no artist or style information.
#    Ties in every later ranker fall back on this order — see the note above and §3.
corpus = corpus.sample(frac=1, random_state=CORPUS_SEED).reset_index(drop=True)

# ---- guards on the properties the evaluation assumes --------------------
w = corpus.artist.value_counts()
assert corpus.n_annotations.min() >= MIN_ANNOTATIONS,   "emotion distribution not stable"
assert set(corpus["style"]) == set(CORPUS_STYLES),         "style list not honoured"
assert w.min() >= ARTIST_MIN,                           "artist below the recall floor"
assert w.max() <= ARTIST_MAX,                           "prolific artist can dominate metrics"
assert corpus.file.is_unique,                           "one row per painting"
assert not (corpus.phash.notna() & corpus.phash.duplicated()).any(),   "pHash duplicate survived"
assert not (corpus.title.notna() &
            corpus.duplicated(["artist", "title"])).any(),             "title duplicate survived"

corpus.to_parquet("corpus.parquet", index=False)

a = corpus.artist.to_numpy()
print(f"{'WikiArt images':<34}{len(paintings):>8,}")
print(f"{'  >=5 ArtEmis annotations':<34}{len(step1):>8,}  "
      f"(-{len(paintings)-len(step1):,} image-only)")
print(f"{'  in the 8 committed styles':<34}{len(step2):>8,}  (-{len(step1)-len(step2):,})")
print(f"{'  deduplicated':<34}{len(step3):>8,}  "
      f"(-{int(dup_hash.sum()):,} by pHash, -{int(dup_title.sum()):,} by title within artist)")
print(f"{'  artists with 10-200 works':<34}{len(corpus):>8,}  (-{len(step3)-len(corpus):,})")
print(f"\nartists   : {corpus.artist.nunique()}  "
      f"(works/artist: min {w.min()}, median {int(w.median())}, max {w.max()})")
print(f"largest style share : {corpus["style"].value_counts(normalize=True).iloc[0]*100:.1f}%")
print(f"annotations/work    : min {corpus.n_annotations.min()}, "
      f"mean {corpus.n_annotations.mean():.2f}")
print(f"target emotions     : awe {corpus.emo_awe.mean():.3f}  "
      f"contentment {corpus.emo_contentment.mean():.3f}  "
      f"amusement {corpus.emo_amusement.mean():.3f}")
print(f"shuffle check       : {(a[1:] != a[:-1]).sum():,} artist runs over {len(corpus):,} rows "
      f"({corpus.artist.nunique()} artists) — row order carries no artist signal")
print(f"\n{corpus["style"].value_counts().to_string()}")
print(f"\ndominant emotion:\n{corpus.dominant_emotion.value_counts().to_string()}")

WikiArt images                      81,444
  >=5 ArtEmis annotations           80,030  (-1,414 image-only)
  in the 8 committed styles         36,584  (-43,446)
  deduplicated                      36,582  (-2 by pHash, -0 by title within artist)
  artists with 10-200 works         20,833  (-15,749)

artists   : 472  (works/artist: min 10, median 29, max 198)
largest style share : 19.7%
annotations/work    : min 5, mean 5.77
target emotions     : awe 0.163  contentment 0.249  amusement 0.115
shuffle check       : 20,756 artist runs over 20,833 rows (472 artists) — row order carries no artist signal

style
Romanticism              4108
Expressionism            3944
Baroque                  3072
Post_Impressionism       2878
Art_Nouveau_Modern       2101
Symbolism                2094
Naive_Art_Primitivism    1448
Pop_Art                  1188

dominant emotion:
dominant_emotion
contentment       6402
awe               4236
amusement         3524
fear              2023
sadness           18

## 2. Synthetic implicit feedback and user profiles

(Extended version in comparison to the report)

> **Declaration — synthetic data.** Everything in this section is **simulated**. Neither
> WikiArt nor ArtEmis contains a user identifier: the ArtEmis release exposes only
> `art_style, painting, emotion, utterance, repetition`, with no annotator field, so the
> ~455K emotion attributions cannot be grouped into people. Collecting real preference data
> was out of scope, so the impression log, its binary engagement outcomes and the user
> profiles are all drawn from the documented generative model below, for exercise
> purposes. Every accuracy figure reported downstream describes recovery of a known ground
> truth, not performance on real users.

### The feedback is implicit and binary

Each row of the log is one **impression** — a painting the system put in front of a user —
carrying a single bit: did they engage with it or not. That is the signal an art catalogue
can actually collect (a click into the detail page, a save, a long dwell), and it is the
signal type baselines (BPR, implicit ALS, EASE) assume.

Three consequences run through everything downstream:

1. **Missing is not negative.** A painting a user was never shown carries no information at
   all. A painting that was shown and not engaged with is a weak negative, and it is the
   only kind of negative this log contains.
2. **There is no threshold to choose.** Relevance is "engaged", full stop.
3. **§6 and §7 become exactly the models they are named after.** A Beta–Bernoulli posterior
   over (impressions, engagements) is the click-through model rather than
   an analogue of it, and §7's bandit reward is a real Bernoulli draw.

### Why simulate rather than sample uniformly

A simulator is not merely a stand-in for missing data, it buys two things real logs never
give you for free:

1. **A known ground truth.** The true utility of every (user, item) pair is available, so the
   ceiling of the task can be measured. A model scoring 0.60 means something once you know
   the best attainable score and what random gives on the same candidate set.
2. **An unbiased test set.** The logged impressions are missing-not-at-random: users are only
   shown what the system chose to show them, and the policy leans toward what they already
   like. A second, small holdout is generated under uniform exposure. Biased-train /
   unbiased-test is the standard setup for studying popularity bias, and it cannot be built
   from logs you do not own.

### The circularity trap, and how it is handled

If engagement is generated from the same features a model consumes, the winner is decided
before anything runs. Three concrete mitigations, each measured in §2.2–2.4 rather than
asserted:

| Risk | Mitigation | Measured |
|---|---|---|
| Content model is handed the answer key | Utility contains a term `⟨p_u, q_i⟩` over **hidden latent factors**; `q_i` is pure noise, not a function of any observable feature. No content model can reach it; CF can. | content-explainable $R^2 = 0.496 \pm 0.147$ (§2.2) |
| Any model reaches a perfect score | The logistic link carries an irreducible noise scale $\tau$, so engagement stays stochastic given the utility and attainable NDCG is capped below 1. | oracle NDCG@5 = 0.861 vs a random floor of 0.313 (§2.2) |
| The profile *is* the hidden taste vector | The stored profile is self-declared and noisy (~25% misreporting), plus two deliberately uninformative demographics. True parameters go to a separate `ORACLE` file. | §2.3 |

### 2.0 Two decisions, in plain terms

**The catalogue is the all 20,833 works (§1.5 corpus), nothing resampled.**
Every `item_id` below indexes it.

**`N_USERS = 12,000`** Collaborative filtering
learns from co-engagement: two users who were both shown, and both engaged with, the same
painting. What controls whether there are enough of those is not the number of users, nor the
percentage density, but

$$\text{impressions per item} \;=\; \frac{\text{users} \times \text{impressions per user}}{\text{items}}$$

The sweep in §2.4 measured where that number has to sit — matrix factorisation loses to a
content model at 9 per item, ties at 32, and wins at 87 *(measured under the earlier
simulator; the crossovers move slightly under the implicit one, the shape does not)*. With the
corpus fixed at 20,833 works and activity at ~150 impressions per user, the only free variable
left is the number of users:

| users | impressions/item | outcome |
|---|---|---|
| 3,000 | 21.6 | CF cannot work — the earlier configuration, and a dead end |
| **12,000** | **86.4** planned, 102 realised | CF works; ~2.1M interactions, MovieLens-1M scale |

Picking 12,000 keeps one catalogue for the whole project, which is the simplest thing to
carry into the modelling sections: no second item set to explain, no metric that means
something different depending on which table it was computed on. The cost is file size and
runtime, not complexity.

The sweeps in §2.3 still run at 3,000 users on a subsampled catalogue. They are diagnostics
of the generative model, not of the delivered dataset, and re-running them at full scale
would take about an hour without changing a single conclusion.

### 2.1 The generative model

For user *u* and painting *i*, the latent utility is

$$
z_{ui} \;=\;
\underbrace{w_e \,\langle \mathbf{a}_u, \mathbf{e}_i\rangle \, c_i}_{\text{emotion match, damped by disagreement}}
\;+\; w_s\, s_u[\text{style}_i]
\;+\; w_a\, t_u[\text{artist}_i]
\;+\; \underbrace{w_\ell\, \langle \mathbf{p}_u, \mathbf{q}_i\rangle}_{\text{hidden, CF-only}}
\;+\; w_p\,\pi_u\, \widetilde{\log f_i}
$$

An impression is now a Bernoulli trial:

$$
\Pr(y_{ui} = 1) \;=\; \sigma\!\big(\lambda\,(\sigma_u z_{ui} + b_u - c)\big),
\qquad
y_{ui} \sim \mathrm{Bernoulli}(\cdot),
\qquad
\lambda = \frac{1.702}{\tau},\;\; \tau = 0.75 .
$$

Two constants, and both have a job:

- **$c$ — the engagement intercept.** Calibrated by bisection on a 400-user probe so that the
  marginal engagement rate over *uniformly drawn* (user, item) pairs equals `LIKE_RATE`. It is
  fitted rather than hand-set, exactly as the star cut points used to be, so the marginal is
  right by construction instead of by tuning.
- **$\tau$ — the irreducible noise scale.** The factor $1.702/\tau$ is the standard logistic
  approximation to a probit: this model is numerically almost indistinguishable from "add
  Gaussian noise $\mathcal N(0,\tau^2)$ to the utility and threshold at $c$", which is what the
  previous version did. So $\tau$ still means what it always meant — raise it and the task gets
  harder, and the oracle ceiling in §2.2 falls.

The point of writing it this way rather than as threshold-plus-noise is that the observation is
now an explicit Bernoulli draw with a *known probability per pair*. §7 uses that directly: the
bandit environment can return an exact engagement probability for any (visitor, arm) pair
instead of estimating it by repeated sampling.

Term by term, in the utility:

- $\mathbf{a}_u$ — signed affinity over the 9 ArtEmis emotions, drawn per user from a
  segment-specific Dirichlet and centred on $1/9$, so a disliked emotion pushes utility
  *down* rather than merely up less.
- $c_i = 1 - H_i/\ln 9$ — annotator agreement. A painting five annotators unanimously called
  *awe* is a reliable awe stimulus; one splitting across five labels is not, so its emotional
  pull is damped. This is the one place the simulator uses something specific to this
  dataset, and it is why `emotion_entropy` was computed in §1.4.
- $b_u, \sigma_u$ — per-user engagement propensity and sensitivity. Some visitors click on
  nearly anything; others are hard to please and respond only to a sharp utility difference.
- $\pi_u = 0.5 - \text{novelty}_u$ — appetite for the popular over the obscure, itself tied
  to the user's experience level.

**Exposure** is missing-not-at-random by construction:

$$
\Pr(\text{shown } i \mid u) \;\propto\; \exp\!\left(\alpha \,\widetilde{\log f_i} + \beta\, z_{ui}\right)
$$

— popular works are shown more ($\alpha$), and the policy leans toward what the user already
likes ($\beta$). Sampling $n_u$ items without replacement uses the Gumbel top-$k$ trick. Note
that exposure depends on what a user *would* engage with drives what they are
shown, which is precisely the loop that makes the log unusable as an unbiased test set.

In [ ]:
# ============================ configuration ==============================
SEED       = 20260826
N_ITEMS    = None     # None = use the whole §1.5 corpus (20,833 works)
MEAN_SHOWN = 150      # impressions per user, on average
# The one number that decides whether collaborative filtering can work at all is
# impressions per ITEM = N_USERS * MEAN_SHOWN / n_items. The sweep in §2.4 puts the
# threshold near 32 and comfort near 87:
#     3,000 users -> 21.6 per item  (MF loses to the content model)
#    12,000 users -> 86.4 per item  (MF wins)
# Activity is lognormal, so the realised mean is ~177 rather than 150 and the delivered
# dataset lands at ~102 impressions/item and ~2.1M interactions.
N_USERS    = 12_000
K_LATENT   = 16       # hidden CF factors — deliberately not derivable from content
N_UNBIASED = 20       # uniformly-exposed holdout impressions per user
COLD_FRAC  = 0.12     # share of users with <=5 impressions, for the cold-start comparison

# --- the observation model (§2.1) ---------------------------------------
# Impressions are Bernoulli trials. LIKE_RATE is the target marginal engagement rate over
# UNIFORMLY drawn (user, item) pairs; the intercept `c` is calibrated to it by bisection in
# simulate(). The rate observed in the *logged* data is much higher, because exposure is
# biased toward what each user would have engaged with anyway.
LIKE_RATE  = 0.28

W_EMOTION, W_LATENT = 2.2, 0.9          # the two weights §2.3 sweeps
# "noise" is tau, the irreducible noise scale of §2.1. It enters as the slope of the logistic
# link, lambda = 1.702 / tau, which is the standard logistic approximation to a probit — so
# raising it still makes the task harder in exactly the way it used to.
W = dict(style=0.7, artist=0.5, pop=0.6, noise=0.75)
ALPHA_POP, BETA_SELF = 1.3, 0.55        # exposure: popularity bias / self-selection bias

SEGMENTS = ["awe_seeker", "comfort_seeker", "amusement_seeker",
            "omnivore_positive", "melancholic"]
SEG_P    = np.array([0.28, 0.30, 0.22, 0.12, 0.08])
# Dirichlet concentration per segment, in EMOTIONS order:
#                     amus  awe  cont  exci  ange  disg  fear  sadn  else
SEG_ALPHA = {
 "awe_seeker":       [0.6, 4.0, 1.4, 2.0, 0.3, 0.3, 0.9, 0.7, 0.5],
 "comfort_seeker":   [1.2, 1.6, 4.2, 1.1, 0.2, 0.2, 0.3, 0.5, 0.5],
 "amusement_seeker": [4.0, 1.1, 1.5, 1.8, 0.4, 0.6, 0.4, 0.3, 0.7],
 "omnivore_positive":[2.0, 2.0, 2.0, 2.0, 0.5, 0.5, 0.8, 0.8, 0.8],
 "melancholic":      [0.5, 1.6, 0.7, 0.5, 0.8, 0.7, 1.8, 3.6, 0.9],
}
EXP_LEVELS = ["novice", "regular", "expert"]
EMO_COLS   = [f"emo_{e.replace(' ', '_')}" for e in EMOTIONS]

NameError: name 'np' is not defined

In [ ]:
# ======================= catalogue and population ========================
def sample_catalogue(rng, n_items=None):
    """The catalogue is the §1.5 corpus, plus the hidden per-item factors.

    n_items=None uses the whole corpus, which is what everything downstream does. A smaller
    n_items takes a stratified subsample and is used only by the density sweep in §2.3;
    stratifying on dominant_emotion keeps the corpus's real emotional mix, since the
    recommender must *find* the awe/contentment/amusement works among the sad and fearful.
    """
    pool = corpus.reset_index(drop=True)
    if n_items is None or n_items >= len(pool):
        items = pool.copy()
    else:
        take = []
        for _, g in pool.groupby("dominant_emotion", sort=True):
            k = max(1, round(len(g) * n_items / len(pool)))
            take.append(rng.choice(g.index.to_numpy(), min(k, len(g)), replace=False))
        items = pool.loc[np.sort(np.concatenate(take))].reset_index(drop=True)
    items = items.reset_index(drop=True)
    items["item_id"] = np.arange(len(items))
    I = len(items)

    c = dict(items=items, n=I)
    c["E"]    = items[EMO_COLS].to_numpy(np.float32)
    c["conf"] = (1.0 - items.emotion_entropy.to_numpy(np.float32) / np.log(9)).clip(0, 1)
    c["styles"],  c["style_idx"]  = np.unique(items["style"],  return_inverse=True)
    c["artists"], c["artist_idx"] = np.unique(items.artist, return_inverse=True)
    # Hidden item factors: pure noise. Recoverable from co-engagement patterns, invisible to
    # every content feature.
    c["Q"] = rng.standard_normal((I, K_LATENT), dtype=np.float32) / np.sqrt(K_LATENT)
    # Latent "fame": lognormal
    lp = np.log(rng.lognormal(0.0, 1.0, I) * (items.n_annotations.to_numpy() / 5.0) ** 0.5)
    c["logpop_z"] = ((lp - lp.mean()) / lp.std()).astype(np.float32)
    return c


def sample_population(rng, cat, n_users, mean_shown, cold_frac):
    """Hidden user parameters. None of this is ever exposed to a recommender."""
    p = {}
    p["seg"] = rng.choice(len(SEGMENTS), n_users, p=SEG_P)
    alpha    = np.array([SEG_ALPHA[s] for s in SEGMENTS], np.float64)[p["seg"]] * 3.0
    p["U_emo"] = ((np.stack([rng.dirichlet(a) for a in alpha]) - 1/9) * 9).astype(np.float32)

    nS = len(cat["styles"])
    p["U_style"] = rng.normal(0, 0.15, (n_users, nS)).astype(np.float32)
    for u in range(n_users):                        # 3-5 favourite styles
        fav = rng.choice(nS, rng.integers(3, 6), replace=False)
        p["U_style"][u, fav] += rng.normal(0.9, 0.25, len(fav))

    nA  = len(cat["artists"])
    big = np.flatnonzero(np.bincount(cat["artist_idx"], minlength=nA) >= 20)
    p["U_artist"] = np.zeros((n_users, nA), np.float32)
    for u in range(n_users):                        # 2-4 favourites, among well-represented artists
        fav = rng.choice(big, rng.integers(2, 5), replace=False)
        p["U_artist"][u, fav] = rng.normal(0.8, 0.2, len(fav))

    p["P"]      = rng.standard_normal((n_users, K_LATENT), dtype=np.float32)  # hidden CF factors
    p["b_user"] = rng.normal(0, 0.45, n_users).astype(np.float32)             # generosity
    p["s_user"] = np.exp(rng.normal(0, 0.18, n_users)).astype(np.float32)     # scale use
    # experience is drawn first, so the *observable* profile field carries genuine (weak)
    # signal: experts explore the long tail more than novices do
    p["exp_lvl"]   = rng.choice(3, n_users, p=[0.50, 0.35, 0.15])
    novelty        = rng.beta(2 + 1.5 * p["exp_lvl"], 5 - 1.0 * p["exp_lvl"])
    p["pop_taste"] = (0.5 - novelty).astype(np.float32)

    n = np.maximum(5, rng.lognormal(np.log(mean_shown), 0.75, n_users).round()).astype(int)
    cold = rng.random(n_users) < cold_frac
    n[cold] = rng.integers(1, 6, cold.sum())
    p["n_shown"] = np.minimum(n, cat["n"] - N_UNBIASED - 1)
    return p

In [ ]:
# ======================= utility, exposure, logging ======================
def utility(cat, pop, u0, u1, w_emotion, w_latent):
    """True latent utility z for users [u0, u1) over the whole catalogue. Chunked, because
    the full matrix would be n_users x n_items."""
    z  = w_emotion * (pop["U_emo"][u0:u1] @ cat["E"].T) * cat["conf"][None, :]
    z += W["style"]  * pop["U_style"][u0:u1][:, cat["style_idx"]]
    z += W["artist"] * pop["U_artist"][u0:u1][:, cat["artist_idx"]]
    z += w_latent    * (pop["P"][u0:u1] @ cat["Q"].T)
    z += W["pop"]    * pop["pop_taste"][u0:u1, None] * cat["logpop_z"][None, :]
    return z


def calibrate_intercept(m, lam, target, iters=60):
    """Find c such that mean(sigmoid(lam * (m - c))) == target.

    Monotone in c, so bisection converges without a derivative and without scipy. This is
    the implicit-feedback counterpart of the star cut points: the marginal engagement rate
    is fitted to LIKE_RATE rather than hand-tuned, so changing w_emotion or w_latent does
    not silently change how often people engage.
    """
    lo, hi = float(m.min()), float(m.max())
    for _ in range(iters):
        c = 0.5 * (lo + hi)
        if sigmoid(lam * (m - c)).mean() > target:
            lo = c                      # engagement too frequent raises the bar
        else:
            hi = c
    return 0.5 * (lo + hi)


def simulate(w_emotion=W_EMOTION, w_latent=W_LATENT, n_items=N_ITEMS,
             mean_shown=MEAN_SHOWN, n_users=N_USERS, seed=SEED, keep_z=False):
    """One full draw of the world: catalogue, population, MNAR impression log with binary
    engagement outcomes, and a uniformly-exposed holdout."""
    rng = np.random.default_rng(seed)
    cat = sample_catalogue(rng, n_items)
    pop = sample_population(rng, cat, n_users, mean_shown, COLD_FRAC)
    I   = cat["n"]

    # Observation model: y ~ Bernoulli(sigmoid(lam * (s_u z + b_u - c))). lam = 1.702/tau is
    # the logistic approximation to a probit with noise sd tau, so this is the same difficulty
    # as thresholding z + N(0, tau^2), but with an explicit per-pair probability.
    lam  = 1.702 / W["noise"]
    _m   = pop["s_user"][:400, None] * utility(cat, pop, 0, 400, w_emotion, w_latent) \
           + pop["b_user"][:400, None]
    c_engage = calibrate_intercept(_m, lam, LIKE_RATE)
    del _m

    def engage(m):
        """One Bernoulli draw per impression."""
        return (rng.random(m.shape) < sigmoid(lam * (m - c_engage))).astype(np.int8)

    rows, unb = [], []
    for u0 in range(0, n_users, 250):
        u1 = min(u0 + 250, n_users)
        z  = utility(cat, pop, u0, u1, w_emotion, w_latent)
        # Gumbel top-k == sampling k items without replacement from softmax(logit).
        # Exposure depends on z, not on y: what a user would engage with drives what they
        # are shown, which is exactly what makes the log MNAR.
        g  = ALPHA_POP * cat["logpop_z"][None, :] + BETA_SELF * z + rng.gumbel(size=z.shape)
        for k, u in enumerate(range(u0, u1)):
            n    = pop["n_shown"][u]
            pick = np.argpartition(-g[k], n)[:n]
            rows.append((np.full(n, u), pick,
                         engage(pop["s_user"][u] * z[k, pick] + pop["b_user"][u])))
            # uniformly-exposed holdout: exposure ignored, disjoint from what was logged
            m = np.ones(I, bool); m[pick] = False
            ub = rng.choice(np.flatnonzero(m), N_UNBIASED, replace=False)
            unb.append((np.full(N_UNBIASED, u), ub,
                        engage(pop["s_user"][u] * z[k, ub] + pop["b_user"][u]),
                        z[k, ub].copy()))

    def frame(rs, cols):
        parts = list(zip(*rs))[:len(cols)]
        return pd.DataFrame(dict(zip(cols, [np.concatenate(x) for x in parts])))

    events   = frame(rows, ["user_id", "item_id", "engaged"])
    unbiased = frame(unb, ["user_id", "item_id", "engaged"] + (["z"] if keep_z else []))

    # timestamps and a per-user temporal split
    events = events.sample(frac=1, random_state=seed)
    events["_o"] = rng.exponential(1.0, len(events))
    events = events.sort_values(["user_id", "_o"]).drop(columns="_o").reset_index(drop=True)
    # gaps accumulate within each user, not across the whole frame, a global cumsum
    # both overflows the ns range and makes every user's history start where the last
    # user's ended.
    start = pd.Timestamp("2026-01-01") + pd.to_timedelta(rng.integers(0, 120, n_users), "D")
    gap   = pd.Series(rng.exponential(3600 * 6, len(events))) \
              .groupby(events.user_id.to_numpy()).cumsum().to_numpy()
    events["timestamp"] = start.values[events.user_id.values] + pd.to_timedelta(gap, "s")
    q = events.groupby("user_id").cumcount() / events.groupby("user_id").user_id.transform("size")
    cnt = events.groupby("user_id").user_id.transform("size")
    events["split"] = np.where(cnt < 5, "train",
                       np.where(q >= 0.85, "test", np.where(q >= 0.70, "val", "train")))

    # observable content-feature design matrix, reused by the diagnostics
    S1 = np.zeros((I, len(cat["styles"])), np.float32)
    S1[np.arange(I), cat["style_idx"]] = 1
    X = np.column_stack([np.ones(I, np.float32), cat["E"], cat["conf"], S1, cat["logpop_z"]])

    return dict(cat=cat, pop=pop, events=events, unbiased=unbiased, X=X,
                engage_c=c_engage, engage_lam=lam,
                w_emotion=w_emotion, w_latent=w_latent)

In [ ]:
# ======================= the dataset used downstream =====================
sim   = simulate(keep_z=True)
cat, pop = sim["cat"], sim["pop"]
items, events, unbiased = cat["items"], sim["events"], sim["unbiased"]
n_items, rng = cat["n"], np.random.default_rng(SEED + 1)

# ---- observable user profile -------------------------------------------
# What a real platform would actually hold: self-declared, noisy, partly worthless.
# The hidden parameters (U_emo, P, ...) deliberately do not appear here.
noisy_emo  = pop["U_emo"] + rng.normal(0, 0.55, pop["U_emo"].shape).astype(np.float32)
decl_style = np.array([cat["styles"][pop["U_style"][u].argmax()] if rng.random() > 0.25
                       else cat["styles"][rng.integers(len(cat["styles"]))]
                       for u in range(N_USERS)])
users = pd.DataFrame({
    "user_id":          np.arange(N_USERS),
    "declared_emotion": np.array(EMOTIONS)[noisy_emo.argmax(1)],   # informative, ~25% noise
    "declared_style":   decl_style,                                # informative, ~25% noise
    "art_experience":   np.array(EXP_LEVELS)[pop["exp_lvl"]],      # weakly informative
    "age_bracket":      rng.choice(["18-24","25-34","35-49","50-64","65+"], N_USERS),
    "country":          rng.choice(["CH","DE","FR","IT","US","UK","other"], N_USERS),
    "n_impressions":    pop["n_shown"],
    "is_cold_start":    pop["n_shown"] <= 5,
})   # age_bracket and country are pure noise by design as a feature-importance sanity check

print(f"catalogue        : {n_items:,} paintings | {len(cat['styles'])} styles "
      f"| {len(cat['artists']):,} artists")
print(f"impressions      : {len(events):,} logged   |  {len(unbiased):,} uniform holdout")
print(f"engagement rate  : {events.engaged.mean():.3f} logged (MNAR)  "
      f"vs {unbiased.engaged.mean():.3f} uniform  <- the MNAR gap")
print(f"engagement events: {int(events.engaged.sum()):,} positives in the log")
print(f"density          : {100 * len(events) / (N_USERS * n_items):.2f}% "
      f"({len(events) / n_items:.0f} impressions/item)")
print(f"split            : {events.split.value_counts().to_dict()}")
print(pd.Series(np.array(SEGMENTS)[pop["seg"]]).value_counts().to_string())

catalogue        : 20,833 paintings | 8 styles | 472 artists
impressions      : 2,126,031 logged   |  240,000 uniform holdout
engagement rate  : 0.536 logged (MNAR)  vs 0.282 uniform  <- the MNAR gap
engagement events: 1,139,484 positives in the log
density          : 0.85% (102 impressions/item)
split            : {'train': 1494002, 'val': 318818, 'test': 313211}
comfort_seeker       3569
awe_seeker           3313
amusement_seeker     2693
omnivore_positive    1470
melancholic           955


In [ ]:
# ======================= a second uniform holdout, for validation ===================
# §8.2 and §8.3 choose their settings on `val`, whose shape is protocol 1's. To check that a
# choice made on the exposure-biased shape survives on the unbiased one, they also need a
# uniformly-exposed holdout that is NOT the test holdout. This draws one: the same
# construction as `unbiased` inside simulate() — N_UNBIASED items per user, uniform over the
# catalogue, engagement drawn from §2.1's observation model — from its own generator
# (SEED + 2) and disjoint from both the user's log and their test holdout. It is written to
# its own file and read by nothing except the two tuning cells.
rng_v = np.random.default_rng(SEED + 2)
logged_by_user = events.groupby("user_id").item_id.apply(lambda s: s.to_numpy()).to_dict()
test_ub_by_user = unbiased.groupby("user_id").item_id.apply(lambda s: s.to_numpy()).to_dict()
vrows = []
for u0 in range(0, N_USERS, 250):
    u1 = min(u0 + 250, N_USERS)
    z  = utility(cat, pop, u0, u1, W_EMOTION, W_LATENT)
    for k, u in enumerate(range(u0, u1)):
        m = np.ones(n_items, bool)
        m[logged_by_user.get(u, [])] = False
        m[test_ub_by_user[u]] = False
        pick = rng_v.choice(np.flatnonzero(m), N_UNBIASED, replace=False)
        p_eng = sigmoid(sim["engage_lam"] * (pop["s_user"][u] * z[k, pick] + pop["b_user"][u]
                                             - sim["engage_c"]))
        vrows.append((np.full(N_UNBIASED, u), pick, (rng_v.random(N_UNBIASED) < p_eng).astype(np.int8)))
unbiased_val = pd.DataFrame({c: np.concatenate(x) for c, x in
                             zip(["user_id", "item_id", "engaged"], zip(*vrows))})
assert not (unbiased_val.merge(unbiased, on=["user_id", "item_id"]).shape[0]), "overlaps the test holdout"
assert not (unbiased_val.merge(events, on=["user_id", "item_id"]).shape[0]), "overlaps the log"
unbiased_val.to_parquet("synthetic_interactions_unbiased_val.parquet", index=False)
print(f"validation holdout : {len(unbiased_val):,} uniform impressions, "
      f"{unbiased_val.engaged.mean():.3f} engaged (test holdout: {unbiased.engaged.mean():.3f})")


### 2.2 Validating the simulator

1. **Shape** — density, a heavy-tailed activity distribution, a long-tailed exposure
   distribution.
2. **The MNAR gap** — engagement rate on logged impressions vs. on the uniformly-exposed
   holdout. A positive gap *is* the popularity / self-selection bias: the log looks better
   than the world because the policy showed people what they were going to like. It is the
   thing an unbiased test set exists to expose, and `LIKE_RATE` fixes only the second of the
   two numbers, never the first.
3. **Content-explainable share** — per-user $R^2$ of the true utility regressed on the
   *observable* content features (emotion proportions, agreement, style one-hot, log
   popularity). Well below 1 means the content-based model probably cannot solve the task
   alone, which is the quantitative answer to the circularity objection.
4. **Ceiling and floor** — NDCG@5 on the unbiased holdout for the oracle (true, noiseless
   utility), a popularity baseline, and random. Every later model must land between floor and
   ceiling; a model above the ceiling means a leak.

The last two are `asserted`, not merely printed, a mis-tuned simulator should fail loudly.

In [ ]:
# ============================ diagnostics ================================
def ndcg_at(unb, score_fn, k=5, users_subset=None):
    """NDCG@k on the uniformly-exposed holdout. Relevance is a held-out engagement — there is
    no threshold to choose, which is one of the things the implicit signal buys. With 20
    candidates of which ~LIKE_RATE are positives, a random ranker scores ~0.3, so the metric
    keeps a wide dynamic range."""
    out = []
    for u, g in unb.groupby("user_id"):
        if users_subset is not None and u not in users_subset:
            continue
        rel = g.engaged.to_numpy().astype(float)
        if rel.sum() == 0:
            continue
        order = np.argsort(-score_fn(u, g.item_id.to_numpy()))
        disc  = np.log2(np.arange(2, min(k, len(rel)) + 2))
        out.append((rel[order][:k] / disc).sum() / (np.sort(rel)[::-1][:k] / disc).sum())
    return float(np.mean(out))


def content_r2(sim, n_probe=200, seed=0):
    """Per-user R² of the true utility regressed on observable content features."""
    Xp = np.linalg.pinv(sim["X"])                # one pseudo-inverse, reused for every user
    n_users = len(sim["pop"]["n_shown"])
    probe = np.random.default_rng(seed).choice(n_users, n_probe, replace=False)
    Z = np.concatenate([utility(sim["cat"], sim["pop"], u, u + 1,
                                sim["w_emotion"], sim["w_latent"]) for u in probe])
    fit = (Z @ Xp.T) @ sim["X"].T
    r2 = 1 - ((Z - fit) ** 2).sum(1) / ((Z - Z.mean(1, keepdims=True)) ** 2).sum(1)
    return float(r2.mean()), float(r2.std())


print("=" * 64)
n_shown = pop["n_shown"]
c       = np.sort(events.item_id.value_counts().to_numpy())
gini    = 1 - 2 * np.sum(np.cumsum(c) / c.sum()) / len(c) + 1 / len(c)
print(f"impressions/user  : min {n_shown.min()}  median {np.median(n_shown):.0f}  "
      f"max {n_shown.max()}")
print(f"cold-start users  : {(n_shown <= 5).sum():,}")
print(f"items ever shown  : {events.item_id.nunique():,} / {n_items:,} "
      f"({events.item_id.nunique() / n_items * 100:.1f}%)")
print(f"exposure Gini     : {gini:.3f}   (top 1% of items hold "
      f"{c[-int(.01 * len(c)):].sum() / c.sum() * 100:.1f}% of all impressions)")

log_rate, unb_rate = events.engaged.mean(), unbiased.engaged.mean()
print(f"\nengagement rate      logged(MNAR)   uniform-exposure")
print(f"  P(engage)            {log_rate * 100:5.1f}%          {unb_rate * 100:5.1f}%"
      f"   <- MNAR gap {log_rate - unb_rate:+.3f}")
print(f"  target LIKE_RATE     {'':5}            {LIKE_RATE * 100:5.1f}%   "
      f"(the uniform column is what is calibrated)")
assert abs(unb_rate - LIKE_RATE) < 0.03, "intercept calibration drifted — check LIKE_RATE"
assert log_rate > unb_rate + 0.05,       "log is not MNAR — raise ALPHA_POP / BETA_SELF"

m, s = content_r2(sim)
print(f"\ncontent-explainable share of true utility (R², 200 users): {m:.3f} ± {s:.3f}")
print("  -> the remainder is hidden CF structure + irreducible noise, unreachable by any")
print("     content model. The CB/CF comparison is therefore not circular.")

zmap   = {u: g.set_index("item_id").z for u, g in unbiased.groupby("user_id")}
pop_ct = events.item_id.value_counts().reindex(range(n_items), fill_value=0).to_numpy()
oracle = ndcg_at(unbiased, lambda u, i: zmap[u].loc[i].to_numpy())
popbl  = ndcg_at(unbiased, lambda u, i: pop_ct[i].astype(float))
rndbl  = ndcg_at(unbiased, lambda u, i: rng.random(len(i)))
print(f"\nNDCG@5, uniformly-exposed holdout")
print(f"  oracle (true noiseless utility) : {oracle:.3f}   <- ceiling, nothing may exceed it")
print(f"  popularity baseline             : {popbl:.3f}")
print(f"  random                          : {rndbl:.3f}   <- floor")
assert oracle < 0.98,         "simulator too easy — raise W['noise']"
assert popbl < oracle - 0.10, "popularity nearly solves it — lower ALPHA_POP / BETA_SELF"

tgt      = ["emo_awe", "emo_contentment", "emo_amusement"]
engaged  = events[events.engaged == 1].merge(items[["item_id"] + tgt], on="item_id")
engaged["segment"] = np.array(SEGMENTS)[pop["seg"]][engaged.user_id.to_numpy()]
tbl = engaged.groupby("segment")[tgt].mean()
tbl.loc["— catalogue average —"] = items[tgt].mean()
print("\nmean emotion content of ENGAGED impressions, by hidden segment:")
print(tbl.round(3).to_string())

impressions/user  : min 1  median 133  max 2960
cold-start users  : 1,458
items ever shown  : 20,718 / 20,833 (99.4%)
exposure Gini     : 0.659   (top 1% of items hold 14.5% of all impressions)

engagement rate      logged(MNAR)   uniform-exposure
  P(engage)             53.6%           28.2%   <- MNAR gap +0.254
  target LIKE_RATE                       28.0%   (the uniform column is what is calibrated)

content-explainable share of true utility (R², 200 users): 0.496 ± 0.147
  -> the remainder is hidden CF structure + irreducible noise, unreachable by any
     content model. The CB/CF comparison is therefore not circular.



NDCG@5, uniformly-exposed holdout
  oracle (true noiseless utility) : 0.861   <- ceiling, nothing may exceed it
  popularity baseline             : 0.371
  random                          : 0.313   <- floor

mean emotion content of ENGAGED impressions, by hidden segment:
                       emo_awe  emo_contentment  emo_amusement
segment                                                       
amusement_seeker         0.123            0.244          0.304
awe_seeker               0.344            0.259          0.069
comfort_seeker           0.131            0.589          0.065
melancholic              0.147            0.162          0.064
omnivore_positive        0.182            0.324          0.137
— catalogue average —    0.163            0.249          0.115


In [ ]:
# ============================ persist ====================================
users.to_parquet("synthetic_users.parquet", index=False)
events.to_parquet("synthetic_interactions.parquet", index=False)
unbiased.drop(columns="z").to_parquet("synthetic_interactions_unbiased.parquet", index=False)
# synthetic_interactions_unbiased_val.parquet was written by the cell after §2.1's dataset cell
items[["item_id", "file"]].to_parquet("synthetic_items.parquet", index=False)

# Ground truth, kept apart and named so that importing it by accident is hard to do
# quietly. Nothing downstream may read this except the evaluation ceiling in §2.2 and the
# bandit ENVIRONMENT in §7 — engage_c and engage_lam are there so §7 can reconstruct the
# exact engagement probability of any (visitor, arm) pair.
np.savez_compressed("synthetic_ORACLE_do_not_train_on_this.npz",
                    P=pop["P"], Q=cat["Q"], U_emo=pop["U_emo"], U_style=pop["U_style"],
                    U_artist=pop["U_artist"], b_user=pop["b_user"], s_user=pop["s_user"],
                    pop_taste=pop["pop_taste"], logpop_z=cat["logpop_z"],
                    segment=pop["seg"],
                    engage_c=sim["engage_c"], engage_lam=sim["engage_lam"],
                    style_idx=cat["style_idx"], artist_idx=cat["artist_idx"])

print(f"""DATA CARD — synthetic feedback layer
  status      : SYNTHETIC, generated for coursework. Not real user behaviour.
  seed        : {SEED}  (fully reproducible)
  users       : {N_USERS:,}   items: {n_items:,} (the §1.5 corpus)
  density     : {100 * len(events) / (N_USERS * n_items):.2f}%
  signal      : IMPLICIT, binary. One row per impression; `engaged` in {{0, 1}}
  impressions : {len(events):,} logged, MNAR exposure  ->  synthetic_interactions.parquet
              : {len(unbiased):,} uniform exposure      ->  synthetic_interactions_unbiased.parquet
  engagements : {int(events.engaged.sum()):,} logged ({events.engaged.mean():.1%})
              : {int(unbiased.engaged.sum()):,} holdout ({unbiased.engaged.mean():.1%})  <- calibrated to LIKE_RATE={LIKE_RATE}
  profiles    : {len(users):,}, self-declared + noise   ->  synthetic_users.parquet
  ground truth: synthetic_ORACLE_do_not_train_on_this.npz  (evaluation / environment only)
  weights     : w_emotion={W_EMOTION}, w_latent={W_LATENT}, tau={W['noise']}  (sensitivity: §2.3)""")

DATA CARD — synthetic feedback layer
  status      : SYNTHETIC, generated for coursework. Not real user behaviour.
  seed        : 20260826  (fully reproducible)
  users       : 12,000   items: 20,833 (the §1.5 corpus)
  density     : 0.85%
  signal      : IMPLICIT, binary. One row per impression; `engaged` in {0, 1}
  impressions : 2,126,031 logged, MNAR exposure  ->  synthetic_interactions.parquet
              : 240,000 uniform exposure      ->  synthetic_interactions_unbiased.parquet
  engagements : 1,139,484 logged (53.6%)
              : 67,719 holdout (28.2%)  <- calibrated to LIKE_RATE=0.28
  profiles    : 12,000, self-declared + noise   ->  synthetic_users.parquet
  ground truth: synthetic_ORACLE_do_not_train_on_this.npz  (evaluation / environment only)
  weights     : w_emotion=2.2, w_latent=0.9, tau=0.75  (sensitivity: §2.3)


### 2.3 Sensitivity: are the conclusions artifacts of the chosen numbers?

Two of the simulator's constants were *chosen*, not estimated, and both directly control the
headline comparison:

- $w_\ell / w_e$ — how much of a user's taste lives in hidden factors (CF territory) versus
  in content features (CB territory). Set this high and CF wins by construction; set it to
  zero and CB wins by construction.
- **Density** — impressions per item. Matrix factorisation needs co-engagement evidence;
  below some density it cannot work regardless of how much signal is theoretically there.
- **Users** — added after §2.4 found that sweep 2 could not explain
  why implicit ALS loses to the content ridge at every density it measured yet wins on the
  delivered log. Sweep 2 holds users at 3,000 and moves items and impressions per user, so
  "impressions per item" and "users who could have co-engaged" never come apart. Sweep 3
  separates them: the delivered item count and impressions per user, with users at 3,000,
  6,000 and 12,000 — the last point *is* the delivered world — plus one **matched-density
  control**, 3,000 users at 600 impressions each, which has the delivered log's impressions
  per item with a quarter of its users. **Prediction, written before the run:** the per-user
  ridge is flat in users (it cannot borrow across them) and rises with impressions per user,
  so the control is its best point; implicit ALS rises with users and beats the ridge at
  12,000 but *not* on the control, which would show that what CF needs here is users, not
  density.

Reporting a single CB-vs-CF number without sweeping these would be reporting a decision, not
a result. The reference models  used for the sweep — a per-user ridge on the observable
content features, and implicit ALS at §8.2's tuned setting, the
model actually designed for a 0/1 signal.

In [ ]:
# ============ two reference models, used only to probe the data ==========
def fit_content(train, X, n_users, lam=5.0):
    """Per-user ridge on the observable content features — a content-based reference.

    On a binary target this is a linear probability model. It is a diagnostic of the data,
    not a delivered recommender, and staying with least squares keeps it dependency-free
    and directly comparable to the MF reference below."""
    B  = np.zeros((n_users, X.shape[1]), np.float32)
    Im = lam * np.eye(X.shape[1], dtype=np.float32)
    for u, g in train.groupby("user_id"):
        Xi = X[g.item_id.to_numpy()]
        B[u] = np.linalg.solve(Xi.T @ Xi + Im, Xi.T @ g.engaged.to_numpy(np.float32))
    return B


def fit_mf(train, n_items, n_users, f=32, lam=8.0, iters=12, seed=0):
    """Biased MF by ALS over the OBSERVED impressions only.

    PureSVD was tried first and scored well below this, because filling the unobserved
    entries with zeros treats 'not shown' as 'disliked' — which is exactly the MNAR
    assumption the unbiased holdout exists to avoid making.
    """
    r  = np.random.default_rng(seed)
    Pu = r.normal(0, .1, (n_users, f)).astype(np.float32)
    Qi = r.normal(0, .1, (n_items,  f)).astype(np.float32)
    u_, i_ = train.user_id.to_numpy(), train.item_id.to_numpy()
    y  = train.engaged.to_numpy(np.float32)
    mu, bu, bi = y.mean(), np.zeros(n_users, np.float32), np.zeros(n_items, np.float32)
    by_u, by_i = train.groupby("user_id").indices, train.groupby("item_id").indices
    Im = lam * np.eye(f, dtype=np.float32)
    for _ in range(iters):
        for u, ix in by_u.items():
            B = Qi[i_[ix]]
            Pu[u] = np.linalg.solve(B.T @ B + Im, B.T @ (y[ix] - mu - bu[u] - bi[i_[ix]]))
            bu[u] = (y[ix] - mu - bi[i_[ix]] - B @ Pu[u]).sum() / (len(ix) + lam)
        for i, ix in by_i.items():
            A = Pu[u_[ix]]
            Qi[i] = np.linalg.solve(A.T @ A + Im, A.T @ (y[ix] - mu - bu[u_[ix]] - bi[i]))
            bi[i] = (y[ix] - mu - bu[u_[ix]] - A @ Qi[i]).sum() / (len(ix) + lam)
    return lambda u, i: mu + bu[u] + bi[i] + Qi[i] @ Pu[u]


def fit_implicit_als(train, n_items, n_users, f=32, lam=1.0, alpha=10.0, iters=15, seed=0):
    """Implicit-feedback ALS (Lecture 2) — the fix §2.4 asks
    for. Preference p_ui = 1 for an engagement and 0 everywhere else, shown-and-ignored and
    never-shown alike, with confidence 1 + alpha on the positives and 1 on the rest. Unlike
    fit_mf it models every cell rather than only the observed impressions, so 'not engaged' is
    weak evidence of zero preference instead of a missing value, and it has no bias terms to
    absorb the per-item engagement rate. Returns the user and item factors (X, Y); the score
    is their dot product."""
    r = np.random.default_rng(seed)
    pos  = train[train.engaged == 1]
    by_u = {u: g.to_numpy() for u, g in pos.groupby("user_id").item_id}
    by_i = {i: g.to_numpy() for i, g in pos.groupby("item_id").user_id}
    X  = r.normal(0, .01, (n_users, f)).astype(np.float32)
    Y  = r.normal(0, .01, (n_items,  f)).astype(np.float32)
    Im = lam * np.eye(f, dtype=np.float32)
    for _ in range(iters):
        YtY = Y.T @ Y + Im                       # the 'everything is a zero' term, once
        X[:] = 0.0                               # users with no positives sit at the origin
        for u, items in by_u.items():
            Yu = Y[items]
            X[u] = np.linalg.solve(YtY + alpha * (Yu.T @ Yu), (1 + alpha) * Yu.sum(0))
        XtX = X.T @ X + Im
        Y[:] = 0.0
        for i, users in by_i.items():
            Xi = X[users]
            Y[i] = np.linalg.solve(XtX + alpha * (Xi.T @ Xi), (1 + alpha) * Xi.sum(0))
    return X, Y


# §8.2's tuned setting, fixed here so the sweep and the delivered rung are the same model
ALS_SWEEP = dict(f=16, lam=10.0, alpha=5.0)


def probe(sim):
    """Score both reference models on one simulated world."""
    rat, unb, X = sim["events"], sim["unbiased"], sim["X"]
    I, nr = sim["cat"]["n"], sim["pop"]["n_shown"]
    train = rat[rat.split == "train"]
    cold  = set(np.flatnonzero(nr <= 5).tolist())
    warm  = set(np.flatnonzero(nr > 20).tolist())
    n_users = int(rat.user_id.max()) + 1
    B, mf = fit_content(train, X, n_users), fit_mf(train, I, n_users)
    Xa, Ya = fit_implicit_als(train, I, n_users, **ALS_SWEEP)
    als = lambda u, i: Ya[i] @ Xa[u]
    zm    = {u: g.set_index("item_id").z for u, g in unb.groupby("user_id")}
    pc    = train.item_id.value_counts().reindex(range(I), fill_value=0).to_numpy()
    r2, _ = content_r2(sim, n_probe=100)
    return dict(density=100 * len(rat) / (n_users * I), per_item=len(rat) / I, r2=r2,
                oracle=ndcg_at(unb, lambda u, i: zm[u].loc[i].to_numpy()),
                pop=ndcg_at(unb, lambda u, i: pc[i].astype(float)),
                CB=ndcg_at(unb, lambda u, i: X[i] @ B[u]),
                MF=ndcg_at(unb, mf),
                CB_cold=ndcg_at(unb, lambda u, i: X[i] @ B[u], users_subset=cold),
                MF_cold=ndcg_at(unb, mf, users_subset=cold),
                CB_warm=ndcg_at(unb, lambda u, i: X[i] @ B[u], users_subset=warm),
                MF_warm=ndcg_at(unb, mf, users_subset=warm),
                iALS=ndcg_at(unb, als), iALS_cold=ndcg_at(unb, als, users_subset=cold),
                iALS_warm=ndcg_at(unb, als, users_subset=warm))


In [ ]:
# ============ sweep 1: how much taste is hidden from content? ============
# The sweeps are diagnostics of the generative model, not of the delivered dataset, so they
# run on a smaller population and a subsampled catalogue — 9 full simulations at 12,000
# users would take the better part of an hour and would not change any conclusion, because
# what governs the outcome is impressions per item, which sweep 2 varies directly.
SWEEP_USERS, SWEEP_ITEMS = 3_000, 6_000

sweep_w = pd.DataFrame([
    dict(w_latent=w, **probe(simulate(w_latent=w, n_items=SWEEP_ITEMS,
                                      n_users=SWEEP_USERS, keep_z=True)))
    for w in [0.0, 0.45, 0.9, 1.8, 3.2]
])
sweep_w.insert(1, "ratio", sweep_w.w_latent / W_EMOTION)
print(sweep_w.drop(columns=["density", "per_item"]).round(3).to_string(index=False))

 w_latent  ratio    r2  oracle   pop    CB    MF  CB_cold  MF_cold  CB_warm  MF_warm  iALS  iALS_cold  iALS_warm
     0.00  0.000 0.941   0.781 0.376 0.654 0.573    0.392    0.497    0.694    0.585 0.542      0.418      0.561
     0.45  0.205 0.744   0.811 0.374 0.629 0.556    0.388    0.501    0.665    0.564 0.525      0.416      0.542
     0.90  0.409 0.489   0.847 0.352 0.555 0.513    0.380    0.461    0.581    0.521 0.524      0.421      0.539
     1.80  0.818 0.223   0.918 0.314 0.446 0.460    0.345    0.407    0.461    0.468 0.567      0.452      0.584
     3.20  1.455 0.092   0.960 0.291 0.355 0.381    0.312    0.382    0.360    0.381 0.628      0.512      0.645


In [ ]:
# ============ sweep 2: is the log dense enough for CF at all? ============
# Varies catalogue size and activity, i.e. impressions per item. Runtime ~1 min per point.
sweep_d = pd.DataFrame([
    dict(n_items=I, mean_shown=n,
         **probe(simulate(n_items=I, mean_shown=n, n_users=SWEEP_USERS, keep_z=True)))
    for I, n in [(20833, 55), (6000, 55), (6000, 150), (3000, 150)]
])
print(sweep_d[["n_items", "mean_shown", "density", "per_item", "oracle", "pop",
               "CB", "MF", "iALS", "MF_cold", "iALS_cold", "MF_warm", "iALS_warm"]]
      .round(3).to_string(index=False))

 n_items  mean_shown  density  per_item  oracle   pop    CB    MF  iALS  MF_cold  iALS_cold  MF_warm  iALS_warm
   20833          55    0.306     9.177   0.859 0.374 0.527 0.425 0.436    0.425      0.397    0.425      0.444
    6000          55    1.067    32.010   0.854 0.360 0.515 0.439 0.461    0.430      0.403    0.441      0.473
    6000         150    2.900    86.996   0.847 0.352 0.555 0.513 0.524    0.461      0.421    0.521      0.539
    3000         150    5.906   177.169   0.854 0.331 0.565 0.550 0.541    0.477      0.422    0.559      0.557


In [ ]:
# ============ sweep 3: is it density, or is it users? =====================
# Delivered item count and impressions per user; only the user count moves — so impressions
# per item rise with users while the evidence *per user* does not. The last row is the
# matched-density control: the delivered log's impressions per item with a quarter of the
# users. See the prediction in §2.3 above; §2.4 reads the result.
sweep_u = pd.DataFrame([
    dict(n_users=U, mean_shown=n,
         **probe(simulate(n_items=N_ITEMS, mean_shown=n, n_users=U, keep_z=True)))
    for U, n in [(3_000, MEAN_SHOWN), (6_000, MEAN_SHOWN), (12_000, MEAN_SHOWN),
                 (3_000, 4 * MEAN_SHOWN)]
])
print(sweep_u[["n_users", "mean_shown", "per_item", "oracle", "pop", "CB", "MF", "iALS",
               "CB_cold", "iALS_cold", "CB_warm", "iALS_warm"]].round(3).to_string(index=False))


 n_users  mean_shown  per_item  oracle   pop    CB    MF  iALS  CB_cold  iALS_cold  CB_warm  iALS_warm
    3000         150    24.935   0.856 0.370 0.572 0.458 0.494    0.377      0.389    0.600      0.509
    6000         150    50.297   0.862 0.373 0.579 0.491 0.517    0.386      0.404    0.606      0.532
   12000         150   102.051   0.861 0.371 0.573 0.522 0.533    0.377      0.420    0.601      0.549
    3000         600    99.577   0.855 0.355 0.613 0.585 0.544    0.390      0.432    0.645      0.560


### 2.4 What the sweeps establish

**Density alone no longer decides the CB-vs-MF comparison — and that is a finding of the
conversion, not a bug.**

| items | impressions/user | impressions **per item** | CB | MF |
|---|---|---|---|---|
| 20,833 (full corpus) | 55 | 9.2 | **0.527** | 0.425 |
| 6,000 | 55 | 32.0 | **0.515** | 0.439 |
| 6,000 | 150 | 87.0 | **0.555** | 0.513 |
| 3,000 | 150 | 177.2 | **0.565** | 0.550 |

**On binary feedback the ALS matrix factorisation never overtakesa the content ridge at any
density measured**, it closes from 0.10 behind to 0.015 behind, and stops there.

The reason is a mismatch: `fit_mf` is an *explicit-feedback*
model - global mean, user bias, item bias, squared error on the observed cell. Applied to a
0/1 target it spends most of its capacity fitting the per-item engagement rate, which the
item-bias term alone already captures, and the latent factors are left reconstructing
Bernoulli noise. The content ridge suffers less because a linear probability model on a binary
target is a much less lossy thing than a rating-prediction model on one.

**This does not mean collaborative filtering fails on this data.** §8's item-kNN — a method
actually designed for binary co-occurrence — beats the content model on the unbiased holdout
(NDCG@10 0.5717 against 0.5187) and beats *everything else* on both protocols. The correct
reading is that the *reference* model in this diagnostic is now mismatched to the signal, not
that the signal is absent. Replacing `fit_mf` with implicit ALS or BPR is the right fix, and it is on the list in §12.

**With the matched reference model.** Implicit ALS, at the
setting §8.2 later chooses on the validation split, joins both sweeps as `iALS`. It repairs
part of the story and complicates the rest:

| items | impressions per item | CB | explicit MF | **implicit ALS** |
|---|---|---|---|---|
| 20,833 | 9.2 | **0.527** | 0.425 | 0.436 |
| 6,000 | 32.0 | **0.515** | 0.439 | 0.461 |
| 6,000 | 87.0 | **0.555** | 0.513 | 0.524 |
| 3,000 | 177.2 | **0.565** | 0.550 | 0.541 |

The implicit model is ahead of the explicit one at every density but the last — so the
mismatch described above is real — and it *still* does not overtake the content ridge at any
density in the sweep. Yet the same model at the same setting beats that ridge's delivered
counterpart by a wide margin on the full log (§8.2: 0.5984 against 0.5187 on P2). The
first explanation was *users*: the sweep holds users at 3,000 while
the delivered log has 12,000, and co-engagement evidence should scale with the number of users
who could have co-engaged. Sweep 3 was added to test that, with its prediction pre-registered
in §2.3, and **the prediction failed where it mattered**:

| users | impressions/user | per item | CB (ridge) | explicit MF | **implicit ALS** |
|---|---|---|---|---|---|
| 3,000 | 150 | 24.9 | **0.572** | 0.458 | 0.494 |
| 6,000 | 150 | 50.3 | **0.579** | 0.491 | 0.517 |
| 12,000 *(the delivered world)* | 150 | 102.1 | **0.573** | 0.522 | 0.533 |
| 3,000 *(matched-density control)* | 600 | 99.6 | **0.613** | 0.585 | 0.544 |

The ridge is flat in users and best on the control, as predicted; implicit ALS rises with users,
as predicted; but at 12,000 users it *still* trails the ridge by 0.04, and the control — the
same impressions per item with a quarter of the users — scores *higher* than the 12,000-user
row (0.544 against 0.533). Per-item density matters at least as much as user count, and the
user-count explanation is withdrawn.

What the third row shows instead is the actual answer, scored by the sweep's reference models: on the same users, the same impressions and the
same holdout, a per-user **ridge** on the observable metadata beats implicit ALS (0.573 against
0.533 at NDCG@5), while in §8.2 the *delivered* content model loses to the same ALS by a wide
margin (0.5187 against 0.5984 at NDCG@10 on the same holdout). The two content models are not
the same model. §5's is a profile — the sum of the feature vectors of the items a user engaged
with, ranked by cosine; the sweep's ridge fits per-user coefficients from the user's impressions with the
non-engagements as zeros, so it uses the negatives too. **The behaviour-beats-metadata result
of §8 is a result against §5's content model, and the sweep says a stronger content model on
the same metadata may reverse it. It reversed the reading:
0.5972 against ALS's 0.5984 on the unbiased holdout, a tie, at nine times the coverage (§5.0).
The density warning below survives untouched.

On the weight sweep the implicit model behaves as a CF model should — it *rises* with the
hidden weight (0.542 → 0.628 across the sweep) while the ridge falls (0.654 → 0.355), and the
crossover moves from ~0.8 to about **0.6**, still above the operating point of 0.41. And on cold
users it is the worst* of the three (0.42 against the explicit model's 0.46 at the operating
point): a user with no training engagement sits at the origin of the factor space and scores
every item zero, whereas the explicit model's item-bias term hands that user a global quality
ranking.

The density warning itself still stands, and is worth stating as a warning rather than a
result. 

The delivered dataset sits at **102 impressions per item**.

**The weight sweep moves the crossover, and moves it a long way.**

| $w_\ell/w_e$ | content-explainable $R^2$ | CB | MF |
|---|---|---|---|
| 0.00 | 0.941 | **0.654** | 0.573 |
| 0.21 | 0.744 | **0.629** | 0.556 |
| 0.41 ← used | 0.489 | **0.555** | 0.513 |
| 0.82 | 0.223 | 0.446 | **0.460** |
| 1.46 | 0.092 | 0.355 | **0.381** |

The crossover sat near $w_\ell/w_e \approx 0.3$ under star ratings; it now sits near **0.8**,
so the operating point used downstream (0.41) is on the *content* side of it rather than just
past it. The honest way to report the CB-vs-CF comparison is therefore still as a curve, but
the curve has moved, and the direction of the move is the same mismatch described above.

At the operating point the per-user ridge collapses on the ≤5-impression cohort
(0.380, against 0.581 on active users) while MF degrades far less (0.461 against 0.521) — so
MF is *ahead* of the content model on exactly the users where content-based methods are
supposed to win. The cause is structural: the ridge fits 39
coefficients per user from that user's own impressions alone and cannot borrow across users,
whereas MF's item-bias term is a global quality prior that needs no user history at all.

The implication for the modelling that follows is concrete: a content-based
recommender that actually helps cold users has to key off the **declared profile**
(`declared_emotion`, `declared_style`) or off similarity to other users' fitted profiles. 

### 2.5 What may and may not be concluded from this

**Supported.** That the models behave as theory
predicts under a known generative process — that matrix factorisation recovers hidden
factors no content feature exposes, that a per-user content regression collapses on
cold-start users where a biased MF does not, that training on MNAR logs and testing on the uniform holdout moves
specific metrics in specific directions. Relative ordering of methods, the gap to the
measured ceiling, and the sensitivity of both to the simulator's free parameters.

**Not supported.** Any absolute claim about real visitors. The numbers characterise this
simulator. Where a conclusion depends on a stipulated weight or on catalogue density, §2.3 shows how
far it can be pushed before it reverses.

**Not tested at all.** Whether the recommended paintings actually evoke awe, contentment or
amusement in a human being. ArtEmis supplies a crowd-annotated *proxy* for the emotional
response to each painting, and the simulated user reacts to that proxy by construction.
Closing that loop needs a study with real participants.

### 2.6 One flat table

A table joins the pieces into a single dataframe — one row per impression, carrying both
the painting's content and the user's profile — and saves it as `interactions.parquet`.

Two joins are needed because `item_id` is a positional index into the frozen §1.5 corpus
rather than a key anything else carries: `synthetic_items.parquet` is the `item_id ↔ file`
bridge, and `file` is what joins back to the painting's content.

The table can be used for exploration and plots. For training, the separate files are more appropriate.

In [ ]:
# ---- the four pieces ----------------------------------------------------
corpus_flat = pd.read_parquet("corpus.parquet").drop(columns="utterances")
bridge         = pd.read_parquet("synthetic_items.parquet")   # item_id <-> file
events         = pd.read_parquet("synthetic_interactions.parquet")
users          = pd.read_parquet("synthetic_users.parquet")

# ---- one flat table -----------------------------------------------------
data = (events
        .merge(bridge,         on="item_id")    # add the file name
        .merge(corpus_flat,    on="file")       # add the painting's content
        .merge(users,          on="user_id"))   # add the user's profile

data = data[["user_id", "declared_emotion", "declared_style", "art_experience",
             "is_cold_start", "item_id", "file", "artist", "style", "genre",
             "dominant_emotion", "emo_awe", "emo_contentment", "emo_amusement",
             "emotion_entropy", "engaged", "timestamp", "split"]]

data.to_parquet("interactions.parquet", index=False)
print(f"{len(data):,} rows x {data.shape[1]} columns  ->  interactions.parquet")
data.head()

2,126,031 rows x 18 columns  ->  interactions.parquet


,user_id,declared_emotion,declared_style,art_experience,is_cold_start,item_id,file,artist,style,genre,dominant_emotion,emo_awe,emo_contentment,emo_amusement,emotion_entropy,engaged,timestamp,split
0,0,sadness,Expressionism,regular,True,3344,Expressionism/ilka-gedo_spring-1973.jpg,ilka gedo,Expressionism,['Expressionism'],awe,0.333333,0.166667,0.166667,1.329661,1,2026-01-27 01:35:23.355597671,train
1,0,sadness,Expressionism,regular,True,15632,Romanticism/henry-raeburn_portrait-of-lucius-o...,henry raeburn,Romanticism,['Romanticism'],awe,0.200000,0.000000,0.000000,1.609438,0,2026-01-27 03:55:42.480419922,train
2,1,awe,Expressionism,novice,False,20497,Expressionism/georges-braque_the-salon-1944.jpg,georges braque,Expressionism,NaN,something else,0.000000,0.166667,0.083333,1.791759,0,2026-01-19 05:02:32.025407651,train
3,1,awe,Expressionism,novice,False,4371,Expressionism/franz-marc_two-cats-blue-and-yel...,franz marc,Expressionism,['Expressionism'],amusement,0.166667,0.000000,0.333333,1.560710,0,2026-01-19 06:02:56.353579652,train
4,1,awe,Expressionism,novice,False,8016,Romanticism/eugene-delacroix_apollo-slays-pyth...,eugene delacroix,Romanticism,['Romanticism'],awe,1.000000,0.000000,0.000000,-0.000000,1,2026-01-19 19:03:14.936961895,train


In [ ]:
# ---- a look at the contents ---------------------------------------------
print("impressions per split :", data.split.value_counts().to_dict())
print("engagement rate       :", f"{data.engaged.mean():.3f}")

print("\nengagement rate by the user's declared emotion and the painting's dominant emotion")
print(data.pivot_table(index="declared_emotion", columns="dominant_emotion",
                       values="engaged", aggfunc="mean")
          [["awe", "contentment", "amusement", "sadness"]].round(3).to_string())

print("\ntop 5 artists by number of impressions received")
print(data.artist.value_counts().head().to_string())

impressions per split : {'train': 1494002, 'val': 318818, 'test': 313211}
engagement rate       : 0.536

engagement rate by the user's declared emotion and the painting's dominant emotion
dominant_emotion    awe  contentment  amusement  sadness
declared_emotion                                        
amusement         0.361        0.431      0.686    0.253
anger             0.364        0.397      0.309    0.451
awe               0.714        0.472      0.354    0.320
contentment       0.549        0.848      0.452    0.329
disgust           0.469        0.444      0.405    0.379
excitement        0.490        0.453      0.448    0.264
fear              0.392        0.320      0.295    0.515
sadness           0.391        0.316      0.288    0.771
something else    0.420        0.436      0.455    0.395

top 5 artists by number of impressions received
artist
karl bryullov               20877
koloman moser               19946
sir lawrence alma tadema    19679
maurice utrillo            

## 3. Baselines: Random and Popularity

Same protocol as the Day 1 Block 3 lab, with three changes forced by this dataset.

The log becomes an implicit top-$K$ task, previously-shown items are removed from the
candidate pool, each method returns a top-$K$ list from the remaining catalogue, and we
report ranking metrics alongside coverage and novelty.

### What changes, and why

**1. No leave-last-positive-out step is needed.** §2 already wrote a per-user temporal
`split` column, cut at the 0.70 / 0.85 quantiles of each user's own timeline, so no training
event can follow a test event by construction.

**2. There is no relevance threshold at all.** This dataset
records an engagement bit per impression, so relevance is simply `engaged == 1` in the test
split. 

**3. Two evaluation protocols, not one.** The code holds a uniformly-exposed sample (§2), which lets us ask the
same question free of the exposure bias that produced the log. The two are reported
separately and are **not comparable to each other** — different candidate sets — but the
*gap between them* is the measurement that matters.

### Seeds

Two, deliberately separate:

- `SEED = 20260826` (§2) decides which world exists — catalogue, users, impressions,
  engagements.
- `SEED_EVAL = 42` (below) decides only what the Random recommender draws.

Random is seeded *per user*
(`default_rng(SEED_EVAL + user_id)`), so one user's list does not
depend on how many users were scored before them.

In [25]:
# valuation setup ===========================
import math

SEED_EVAL = 42     # only the Random recommender uses this — see the note above
SEED_TIE  = 101    # resolves tied scores; offset per user / per query, see top_k
K         = 10
# No relevance threshold: the feedback is already binary, so a held-out `engaged == 1` IS
# the positive. This is the free parameter the implicit conversion removed.

events_all  = pd.read_parquet("synthetic_interactions.parquet")
bridge      = pd.read_parquet("synthetic_items.parquet")
corpus_meta = pd.read_parquet("corpus.parquet")[["file", "artist", "style", "title",
                                                 "dominant_emotion"]]
titles      = bridge.merge(corpus_meta, on="file").set_index("item_id")
n_items     = len(bridge)
n_users     = int(events_all.user_id.max()) + 1   # derived here, so §3-§5 run without §2

# Baselines are fitted on `train` only, so every later model sees the same fitting data.
# Candidates exclude everything the user was SHOWN in train OR val — seen is seen, whether
# or not they engaged. Never-shown items stay in the pool: missing is not negative.
train = events_all[events_all.split == "train"]
seen  = events_all[events_all.split != "test"]

seen_by_user = seen.groupby("user_id").item_id.apply(lambda s: s.to_numpy()).to_dict()
test_positives = (events_all[(events_all.split == "test") & (events_all.engaged == 1)]
                  .groupby("user_id").item_id.apply(lambda s: set(s.to_numpy())).to_dict())
eval_users = np.array(sorted(test_positives))

# item popularity = impressions received, and a smoothed probability for the novelty metric
popularity  = train.item_id.value_counts().reindex(range(n_items), fill_value=0).to_numpy(float)
item_probability = (popularity + 1) / (popularity.sum() + n_items)

print(f"catalogue          : {n_items:,} items")
print(f"training events    : {len(train):,} impressions, "
      f"{int(train.engaged.sum()):,} engagements ({train.engaged.mean():.1%})")
print(f"evaluation users   : {len(eval_users):,} / {events_all.user_id.nunique():,} "
      f"(the rest engaged with nothing in test)")
print(f"held-out positives : {np.mean([len(v) for v in test_positives.values()]):.1f} per user")

# the leakage guard the lab performs after splitting — ours was asserted at generation,
# so this only re-checks that nothing was reshuffled since
assert all(len(np.intersect1d(seen_by_user[u], list(test_positives[u]))) == 0
           for u in eval_users[:200]), "a test positive is also in the seen set"
print("candidate-exclusion check passed.")

catalogue          : 20,833 items
training events    : 1,494,002 impressions, 800,979 engagements (53.6%)
evaluation users   : 10,448 / 12,000 (the rest engaged with nothing in test)
held-out positives : 16.1 per user
candidate-exclusion check passed.


In [ ]:
# The recommenders ===========================
def top_k(scores, seen_items, k=K, tie_seed=0):
    """Rank the catalogue by `scores`, drop anything the user has already been shown, return the
    best k item_ids. argpartition avoids sorting all 20,833 items for every user.

    Ties are resolved by a seeded draw, never by row order. They are common: B2's
    17-dimensional score leaves ~11 candidates sharing the rank-10 score of a typical query, and
    §6's raw engagement rate ties items in bulk, since anything engaged 2-of-2 scores exactly
    1.000. argpartition and argsort both order tied values by ascending index. §1.5 shuffled the
    corpus so that index order carries no artist or style signal, which removes the inflation
    risk; this adds the second half of the rule, so tied candidates rotate between users and
    queries instead of one low-index item winning every list and quietly capping coverage.

    Two places need it: the boundary (which of the items tied at rank k get in) and the order
    within the returned list (which of the tied items ranks higher, which NDCG and MRR see).
    """
    s = scores.astype(np.float64, copy=True)
    s[seen_items] = -np.inf
    idx = np.argpartition(-s, k)[:k]
    cut = s[idx].min()                                   # the score at rank k
    n_at_cut = int((s[idx] == cut).sum())
    tied = np.flatnonzero(s == cut)                      # every candidate sharing it
    rng_t = np.random.default_rng(SEED_TIE + tie_seed)
    if len(tied) > n_at_cut:                             # contested boundary: draw the seats
        idx = np.concatenate([idx[s[idx] > cut],
                              rng_t.choice(tied, n_at_cut, replace=False)])
    rng_t.shuffle(idx)                                   # random order among equal scores 
    return idx[np.argsort(-s[idx], kind="stable")]       #  which a stable sort preserves


def ties_at_rank_k(scores, k=K):
    """§1.5's tie diagnostic: how many candidates share the score of the item at rank k.
    Reported next to the tie-heavy methods (B2 in §5, the quality rankers in §6), because a
    method whose rank-10 score is shared by thousands of paintings is closer to 'random within
    a score bucket' than to a ranker, and that is itself worth reporting."""
    kth = np.partition(scores, -k)[-k]
    return int((scores == kth).sum())


def recommend_random(user_id, k=K):
    """Baseline A — a reproducible random draw of unseen items."""
    user_rng = np.random.default_rng(SEED_EVAL + user_id)
    return top_k(user_rng.random(n_items), seen_by_user[user_id], k, tie_seed=int(user_id))


def recommend_popularity(user_id, k=K):
    """Baseline B — the most-shown unseen items. The bar Random exists to sit under."""
    return top_k(popularity, seen_by_user[user_id], k, tie_seed=int(user_id))


# basic checks
_u = int(eval_users[0])
assert len(recommend_random(_u)) == K
assert not set(recommend_random(_u)) & set(seen_by_user[_u].tolist())
assert (recommend_random(_u) == recommend_random(_u)).all(), "Random must be reproducible"
# the tie-break must be deterministic per user and must not depend on catalogue row order
_flat = np.zeros(n_items); _flat[:50] = 1.0
assert (top_k(_flat, np.array([0]), tie_seed=7) == top_k(_flat, np.array([0]), tie_seed=7)).all()
assert set(top_k(_flat, np.array([0]), tie_seed=7)) != set(range(1, K + 1)), \
       "tied scores still resolving by row order"
print("Recommendation functions passed the basic checks.")

Recommendation functions passed the basic checks.


In [27]:
# Metrics ====================================
def evaluate_user(ranked_items, targets, k=K):
    """Ranking metrics for one user. Unlike the lab there are several held-out positives,
    so Recall@K and Hit Rate@K differ and the ideal DCG is not 1."""
    hits  = np.isin(ranked_items[:k], list(targets)).astype(float)
    disc  = np.log2(np.arange(2, k + 2))
    idcg  = (1.0 / disc[:min(len(targets), k)]).sum()
    found = np.flatnonzero(hits)
    return {
        "hit_rate":  float(hits.any()),
        "precision": hits.sum() / k,
        "recall":    hits.sum() / len(targets),
        "mrr":       0.0 if len(found) == 0 else 1.0 / (found[0] + 1),
        "ndcg":      (hits / disc).sum() / idcg,
        "average_popularity": float(popularity[ranked_items[:k]].mean()),
        "novelty_bits":       float((-np.log2(item_probability[ranked_items[:k]])).mean()),
    }


def catalogue_coverage(by_user):
    recommended = {int(i) for ranked in by_user.values() for i in ranked}
    return len(recommended) / n_items, len(recommended)


# sanity checks on the metric itself
_r = np.array([7, 3, 5, 9])
assert evaluate_user(_r, {7}, k=4)["mrr"] == 1.0
assert evaluate_user(_r, {3}, k=4)["mrr"] == 0.5
assert evaluate_user(_r, {1}, k=4)["mrr"] == 0.0
assert evaluate_user(_r, {7, 3}, k=4)["recall"] == 1.0
print("Metric checks passed.")

Metric checks passed.


In [28]:
# Protocol 1: full-catalogue ranking ==================
RECOMMENDERS = {"Random": recommend_random, "Popularity": recommend_popularity}

recommendation_lists, per_user_rows = {}, []
for name, recommend in RECOMMENDERS.items():
    recommendation_lists[name] = {}
    for user_id in eval_users:
        ranked = recommend(user_id)
        recommendation_lists[name][user_id] = ranked
        per_user_rows.append({"algorithm": name, "user_id": user_id,
                              **evaluate_user(ranked, test_positives[user_id])})

per_user_metrics = pd.DataFrame(per_user_rows)

# A registry, so later baselines and models append to one comparison table instead of
# each printing its own. Re-running a method overwrites its row rather than duplicating it.
RESULTS = {}

def record(table):
    RESULTS.update({r["algorithm"]: r for r in table.to_dict("records")})
    return pd.DataFrame(RESULTS.values()).sort_values(f"NDCG@{K}", ascending=False)

results = (per_user_metrics.groupby("algorithm")
           .agg(**{f"Hit Rate@{K}":  ("hit_rate", "mean"),
                   f"Precision@{K}": ("precision", "mean"),
                   f"Recall@{K}":    ("recall", "mean"),
                   f"MRR@{K}":       ("mrr", "mean"),
                   f"NDCG@{K}":      ("ndcg", "mean"),
                   "Average popularity": ("average_popularity", "mean"),
                   "Novelty (bits)":     ("novelty_bits", "mean")})
           .reset_index())

coverage = pd.DataFrame([{"algorithm": n, "Catalogue coverage": catalogue_coverage(b)[0],
                          "Unique items recommended": catalogue_coverage(b)[1]}
                         for n, b in recommendation_lists.items()])
results = results.merge(coverage, on="algorithm")
record(results).round(4)

,algorithm,Hit Rate@10,Precision@10,Recall@10,MRR@10,NDCG@10,Average popularity,Novelty (bits),Catalogue coverage,Unique items recommended
0,Popularity,0.2998,0.0396,0.0264,0.1313,0.0488,2070.2538,9.5421,0.0026,54
1,Random,0.0075,0.0008,0.0004,0.0020,0.0008,69.5521,15.6187,0.9930,20688


In [29]:
# Protocol 2: the unbiased holdout ====================
# Re-rank the 20 uniformly-exposed candidates per user. Exposure played no part in choosing
# them, so popularity gets no free ride here. Not comparable to the table above — a
# different candidate set — but the gap between the two IS the popularity bias.
unbiased = pd.read_parquet("synthetic_interactions_unbiased.parquet")

def evaluate_reranking(score_fn, k=K, frame=False):
    """Re-rank each user's uniformly-exposed candidates with `score_fn(user_id, candidates)`.
    Returns the mean over users; with frame=True the per-user frame, which is what §9.5
    bootstraps and pairs. Coverage is deliberately absent — the candidate set is fixed per
    user, so it would measure the holdout, not the recommender (§7.6)."""
    rows = []
    for user_id, g in unbiased.groupby("user_id"):
        candidates = g.item_id.to_numpy()
        relevant   = g.engaged.to_numpy().astype(float)
        if relevant.sum() == 0:
            continue
        order = np.argsort(-score_fn(user_id, candidates))
        disc  = np.log2(np.arange(2, min(k, len(candidates)) + 2))
        hits  = relevant[order][:k]
        found = np.flatnonzero(hits)
        rows.append({"user_id":   int(user_id),
                     "hit_rate":  float(hits.any()),
                     "precision": hits.sum() / k,
                     "recall":    hits.sum() / relevant.sum(),
                     "mrr":       0.0 if len(found) == 0 else 1.0 / (found[0] + 1),
                     "ndcg":      (hits / disc).sum() / (np.sort(relevant)[::-1][:k] / disc).sum(),
                     "average_popularity": float(popularity[candidates[order][:k]].mean())})
    table = pd.DataFrame(rows).set_index("user_id")
    return table if frame else table.mean()

unbiased_results = pd.DataFrame({
    "Random":     evaluate_reranking(
                      lambda u, c: np.random.default_rng(SEED_EVAL + u).random(len(c))),
    "Popularity": evaluate_reranking(lambda u, c: popularity[c]),
}).T
print(f"{len(unbiased):,} holdout impressions, "
      f"{unbiased.groupby('user_id').size().iloc[0]} candidates/user, "
      f"{unbiased.engaged.mean():.1%} engaged\n")
unbiased_results.round(4)


240,000 holdout impressions, 20 candidates/user, 28.2% engaged



,hit_rate,precision,recall,mrr,ndcg,average_popularity
Random,0.9303,0.2895,0.4992,0.4932,0.4015,69.4848
Popularity,0.9445,0.3180,0.5530,0.5545,0.4552,125.6273


In [30]:
# The protocol-2 ceiling, at the same K as every table ===============
# §2.2 reports the oracle at NDCG@5; §3-§8 all report NDCG@10. Without this cell the ladder
# has no top rung on its own scale, and it is tempting to compare a @10 result against the
# @5 ceiling — which flatters every method. The oracle ranks by true noiseless utility, so
# nothing may exceed it; a method that does has a leak.
with np.load("synthetic_ORACLE_do_not_train_on_this.npz") as _z:      # materialise once: a
    _O = {k: _z[k] for k in ("U_emo", "U_style", "U_artist", "P", "Q",  # lazy .npz decompresses
                             "pop_taste", "logpop_z", "style_idx", "artist_idx")}  # per access

_meta = (bridge.merge(pd.read_parquet("corpus.parquet"), on="file")
               .sort_values("item_id").reset_index(drop=True))
_E    = _meta[[c for c in _meta.columns if c.startswith("emo_")]].to_numpy(np.float32)
_conf = (1 - _meta.emotion_entropy.to_numpy(np.float32) / np.log(9)).clip(0, 1)

_u = unbiased.sort_values("user_id", kind="stable")
_n = _u.groupby("user_id").size().iloc[0]
_U = _u.user_id.to_numpy().reshape(-1, _n)[:, 0]
_I = _u.item_id.to_numpy().reshape(-1, _n)
_R = _u.engaged.to_numpy().astype(float).reshape(-1, _n)

# §2.1's latent utility. s_user and b_user are omitted: both are monotone in the score, so
# neither can change a per-user ranking.
_Z  = 2.2 * np.einsum("uk,uik->ui", _O["U_emo"][_U], _E[_I]) * _conf[_I]
_Z += 0.7 * np.take_along_axis(_O["U_style"][_U],  _O["style_idx"][_I],  axis=1)
_Z += 0.5 * np.take_along_axis(_O["U_artist"][_U], _O["artist_idx"][_I], axis=1)
_Z += 0.9 * np.einsum("uf,uif->ui", _O["P"][_U], _O["Q"][_I])
_Z += 0.6 * _O["pop_taste"][_U][:, None] * _O["logpop_z"][_I]

_disc  = np.log2(np.arange(2, min(K, _n) + 2))
_hits  = np.take_along_axis(_R, np.argsort(-_Z, axis=1), axis=1)[:, :K]
_ideal = np.sort(_R, axis=1)[:, ::-1][:, :K]
_keep  = _R.sum(1) > 0
ORACLE_NDCG_P2 = float(((_hits / _disc).sum(1)[_keep] / (_ideal / _disc).sum(1)[_keep]).mean())

print(f"users scored                      : {_keep.sum():,}")
print(f"ORACLE NDCG@{K}, unbiased holdout   : {ORACLE_NDCG_P2:.4f}   <- ceiling for every P2 number")
print(f"random floor (measured above)     : 0.4015")
print(f"headroom the ladder has to close  : {ORACLE_NDCG_P2 - 0.4015:.4f}")

users scored                      : 11,695
ORACLE NDCG@10, unbiased holdout   : 0.8919   <- ceiling for every P2 number
random floor (measured above)     : 0.4015
headroom the ladder has to close  : 0.4904


In [31]:
# Inspect one user's recommendations ==================
def show_recommendations(user_id, n=5):
    target = test_positives[user_id]
    print(f"User {user_id} — {len(target)} held-out engaged paintings in test\n")
    rows = []
    for name, by_user in recommendation_lists.items():
        for rank, item_id in enumerate(by_user[user_id][:n], start=1):
            meta = titles.loc[item_id]
            rows.append({"algorithm": name, "rank": rank, "artist": meta.artist,
                         "style": meta.style, "emotion": meta.dominant_emotion,
                         "impressions": int(popularity[item_id]),
                         "is_target": item_id in target})
    return pd.DataFrame(rows)

show_recommendations(int(eval_users[0]))

User 1 — 24 held-out engaged paintings in test



,algorithm,rank,artist,style,emotion,impressions,is_target
0,Random,1,james tissot,Symbolism,awe,87,False
1,Random,2,koloman moser,Art_Nouveau_Modern,contentment,4,False
2,Random,3,arnold bã¶cklin,Romanticism,contentment,12,False
3,Random,4,panayiotis tetsis,Post_Impressionism,contentment,17,False
4,Random,5,john everett millais,Romanticism,amusement,23,False
5,Popularity,1,frank johnston,Post_Impressionism,contentment,2513,False
6,Popularity,2,henri rousseau,Naive_Art_Primitivism,contentment,2488,False
7,Popularity,3,johannes vermeer,Baroque,contentment,2314,False
8,Popularity,4,george frederick watts,Romanticism,awe,2223,False
9,Popularity,5,aldemir martins,Naive_Art_Primitivism,amusement,2058,False


### 3.1 What to notice

**Random is not a competitor, it is a floor.** Its NDCG@10 of 0.0008 against Popularity's
0.0488 says the catalogue is large and blind guessing does not work — nothing more. Its
value is that any model failing to clear it is broken, and that it sets the *upper* bound on
coverage and novelty that better models will trade away.

**Popularity's 61× ranking advantage is largely an artifact of how the log was made.** The
same comparison on the uniformly-exposed holdout collapses to 1.13× (NDCG@10 0.4552 against
Random's 0.4015). Exposure in the log is proportional to popularity by construction (§2.1), so
the logged test set rewards a recommender for predicting *what the system showed people*
rather than what they would have engaged with. This is the single most important reason the
unbiased holdout exists, and it is not visible from the first table alone.

**Popularity reaches 54 items.** Out of 20,833 — a catalogue coverage of 0.26%, against
Random's 99.3%. For an art-discovery product whose stated purpose is exploratory browsing,
a recommender that shows everyone the same 54 paintings has failed at the task even while
winning the accuracy metric. Novelty tells the same story in bits: 9.5 for Popularity
against 15.6 for Random. Its recommendations average 2,070 impressions each; Random's average
70, which is the catalogue mean.

**Where the ceiling is, and a caution about comparing to it.** §2.2 measured the best
attainable **NDCG@5** on the unbiased holdout at 0.861, against 0.371 for popularity and 0.313
for random *at the same k*. The protocol-2 table above reports **NDCG@10**, which is a
different normalisation on the same 20 candidates and runs higher for every method. The two
should not be read against each other directly — the ceiling is quoted here to establish that
the task has a known solvable structure and a measurable headroom, not to compute a percentage.

**Hit Rate is very high on protocol 2 for every method, including Random (0.93).** That is not
a bug: with 20 candidates, ~28% of them positives, and K = 10, almost every user has *some*
positive in a random top-10. On this protocol Hit Rate carries almost no information and NDCG
does the work — a reminder that the metric has to suit the candidate set, not just the task.

## 4. Item-to-item "More like this"

A second, independent evaluation track. §3 asks *what should this user see next*; this section
asks *what else is like this painting* — the "More like this" module from §1, where there is no
user at all, only a painting on screen.

The two tracks share a catalogue and nothing else. They have different queries, different
relevance definitions, different candidate pools and different metrics, so **their numbers are
never comparable** and they get separate results tables.

### Protocol

| | |
|---|---|
| query | one painting from the corpus |
| candidates | every other corpus painting (20,832) |
| relevance | **same artist** as the query |
| positives per query | 9 to 197 (the artist band from §1.5), median 67 |

The median of 66 rather than the corpus's median of 29 works per artist: queries are sampled uniformly over *paintings*, so a 198-work artist is drawn 20 times as often as a 10-work one. The query set is length-biased toward prolific artists by construction, which is worth remembering when reading per-query averages.
| metrics | Recall@K, Precision@K, MRR@K, NDCG@K, plus coverage and novelty |

The same-artist proxy is the reason §1.5 restricted artists to 10–200 works. Below 10, recall
is computed against so few positives that it swings wildly between queries; above 200, a few
prolific artists supply most of the positives for every query and the metric measures them
rather than the recommender.

**It is a proxy** Two Monets are labelled relevant to each other; a Monet
and a visually near-identical Sisley are not. It rewards recovering authorship, which
correlates with visual and affective similarity without being the same thing. Anything the
metric says should be read with that in mind.

**Queries are sampled, not exhaustive.** 2,000 of 20,833, drawn with a fixed seed. Scoring
every painting against every other is 434M pairs per method and buys no precision that
matters at three decimal places.

### The baselines

- **B0 — Random.** Uniform over the catalogue. The floor.
- **B1 — Random within the same style.** Uniform over paintings sharing the query's style,
  falling back to the rest of the catalogue if that runs short. This is the first baseline
  that uses any content signal, and it is a deliberately weak one: style is a single
  categorical field, already in the metadata, requiring no image and no model.

B1 is worth measuring because of two facts about the corpus: **68.9%** of artists work in a
single style, and **88.2%** of an artist's other works share the style of any given query —
while a style holds only 12.5% of the catalogue on average. A large gain from B1 therefore says something
about how the corpus is built, not about the recommender being clever.

In [32]:
# Item-to-item evaluation setup ======================
N_QUERIES = 2_000
SEED_I2I  = 7

# item_id -> style / artist, aligned to the catalogue index used everywhere in §3-§4
meta = (bridge.merge(pd.read_parquet("corpus.parquet")[["file", "artist", "style"]], on="file")
              .sort_values("item_id").reset_index(drop=True))
item_style  = meta["style"].to_numpy()
item_artist = meta.artist.to_numpy()

# positives = the artist's other works
by_artist = {a: g.item_id.to_numpy() for a, g in meta.groupby("artist")}

rng_i2i  = np.random.default_rng(SEED_I2I)
query_ids = np.sort(rng_i2i.choice(n_items, N_QUERIES, replace=False))

n_pos = np.array([len(by_artist[item_artist[q]]) - 1 for q in query_ids])
print(f"queries            : {N_QUERIES:,} of {n_items:,}")
print(f"positives/query    : min {n_pos.min()}, median {int(np.median(n_pos))}, max {n_pos.max()}")
print(f"styles             : {len(np.unique(item_style))}")
print(f"a style holds      : {pd.Series(item_style).value_counts().mean()/n_items:.1%} "
      f"of the catalogue on average")
assert n_pos.min() >= 1, "every query needs at least one same-artist positive"

queries            : 2,000 of 20,833
positives/query    : min 9, median 67, max 197
styles             : 8
a style holds      : 12.5% of the catalogue on average


In [33]:
# The two baselines ==================================
def recommend_i2i_random(query_id, k=K):
    """B0 — uniform over the catalogue, the query itself excluded."""
    rng_q = np.random.default_rng(SEED_EVAL + query_id)
    return top_k(rng_q.random(n_items), np.array([query_id]), k,
                 tie_seed=int(query_id))


def recommend_i2i_random_same_style(query_id, k=K):
    """B1 — uniform within the query's style. Adding 1.0 to same-style scores puts every
    same-style painting above every other one, so the fallback is automatic when a style
    holds fewer than k items."""
    rng_q  = np.random.default_rng(SEED_EVAL + query_id)
    scores = rng_q.random(n_items)
    scores[item_style == item_style[query_id]] += 1.0
    return top_k(scores, np.array([query_id]), k, tie_seed=int(query_id))


_q = int(query_ids[0])
assert _q not in recommend_i2i_random(_q), "the query must not recommend itself"
assert (item_style[recommend_i2i_random_same_style(_q)] == item_style[_q]).all(), \
       "B1 must return same-style items when the style is large enough"
print("Item-to-item recommenders passed the basic checks.")

Item-to-item recommenders passed the basic checks.


In [34]:
# Metrics and the run ================================
def evaluate_query(ranked_items, query_id, k=K):
    positives = set(by_artist[item_artist[query_id]].tolist()) - {query_id}
    hits  = np.isin(ranked_items[:k], list(positives)).astype(float)
    disc  = np.log2(np.arange(2, k + 2))
    idcg  = (1.0 / disc[:min(len(positives), k)]).sum()
    found = np.flatnonzero(hits)
    return {
        "hit_rate":  float(hits.any()),
        "precision": hits.sum() / k,
        "recall":    hits.sum() / len(positives),
        "mrr":       0.0 if len(found) == 0 else 1.0 / (found[0] + 1),
        "ndcg":      (hits / disc).sum() / idcg,
        "same_style_share": float((item_style[ranked_items[:k]] == item_style[query_id]).mean()),
        "novelty_bits":     float((-np.log2(item_probability[ranked_items[:k]])).mean()),
    }


I2I_RECOMMENDERS = {"B0 Random": recommend_i2i_random,
                    "B1 Random within style": recommend_i2i_random_same_style}

i2i_lists, i2i_rows = {}, []
for name, recommend in I2I_RECOMMENDERS.items():
    i2i_lists[name] = {}
    for q in query_ids:
        ranked = recommend(int(q))
        i2i_lists[name][int(q)] = ranked
        i2i_rows.append({"algorithm": name, "query_id": int(q),
                         **evaluate_query(ranked, int(q))})

per_query_metrics = pd.DataFrame(i2i_rows)
RESULTS_I2I = {}

def record_i2i(table):
    RESULTS_I2I.update({r["algorithm"]: r for r in table.to_dict("records")})
    return pd.DataFrame(RESULTS_I2I.values()).sort_values(f"Recall@{K}", ascending=False)

i2i_results = (per_query_metrics.groupby("algorithm")
               .agg(**{f"Hit Rate@{K}":  ("hit_rate", "mean"),
                       f"Precision@{K}": ("precision", "mean"),
                       f"Recall@{K}":    ("recall", "mean"),
                       f"MRR@{K}":       ("mrr", "mean"),
                       f"NDCG@{K}":      ("ndcg", "mean"),
                       "Same-style share": ("same_style_share", "mean"),
                       "Novelty (bits)":   ("novelty_bits", "mean")})
               .reset_index())
i2i_cov = pd.DataFrame([{"algorithm": n,
                         "Catalogue coverage": len({int(i) for r in b.values() for i in r}) / n_items}
                        for n, b in i2i_lists.items()])
record_i2i(i2i_results.merge(i2i_cov, on="algorithm")).round(4)

,algorithm,Hit Rate@10,Precision@10,Recall@10,MRR@10,NDCG@10,Same-style share,Novelty (bits),Catalogue coverage
1,B1 Random within style,0.2240,0.0261,0.0035,0.0710,0.0262,1.0000,15.6036,0.6162
0,B0 Random,0.0405,0.0041,0.0005,0.0127,0.0042,0.1498,15.6152,0.6145


In [35]:
# Inspect one query ==================================
def show_similar(query_id, n=5):
    q = titles.loc[query_id]
    print(f"Query: {q.artist} — {q.style} — {q.dominant_emotion}")
    print(f"       {q.title if pd.notna(q.title) else q.name}")
    print(f"       {len(by_artist[item_artist[query_id]]) - 1} same-artist positives exist\n")
    rows = []
    for name, by_query in i2i_lists.items():
        for rank, item_id in enumerate(by_query[query_id][:n], start=1):
            m = titles.loc[item_id]
            rows.append({"algorithm": name, "rank": rank, "artist": m.artist,
                         "style": m.style, "emotion": m.dominant_emotion,
                         "same_artist": m.artist == q.artist,
                         "same_style":  m.style == q.style})
    return pd.DataFrame(rows)

show_similar(int(query_ids[0]))

Query: frantisek kupka — Expressionism — awe
       the-song-of-songs
       50 same-artist positives exist



,algorithm,rank,artist,style,emotion,same_artist,same_style
0,B0 Random,1,gustave moreau,Romanticism,awe,False,False
1,B0 Random,2,mikhail nesterov,Art_Nouveau_Modern,sadness,False,False
2,B0 Random,3,henri rousseau,Post_Impressionism,contentment,False,False
3,B0 Random,4,anthony van dyck,Baroque,sadness,False,False
4,B0 Random,5,hiro yamagata,Pop_Art,excitement,False,False
5,B1 Random within style,1,viorel marginean,Expressionism,fear,False,True
6,B1 Random within style,2,oskar kokoschka,Expressionism,contentment,False,True
7,B1 Random within style,3,albert bloch,Expressionism,sadness,False,True
8,B1 Random within style,4,otto dix,Expressionism,fear,False,True
9,B1 Random within style,5,candido portinari,Expressionism,amusement,False,True


## 5. Content-based recommendation

$$
\text{IDF}(g)=\log\frac{|I|+1}{|\{i:g\in i\}|+1}+1
\qquad
\mathbf{u}=P\,W
\qquad
s(u,i)=\cos(\mathbf{u},\mathbf{w}_i)
$$

The feedback here is binary, so the lab's graded preference mapping collapses to a weight of
1 per engaged item: a user's profile is the IDF-weighted **mean** of the paintings they
engaged with in the training split. One fewer arbitrary constant than the graded version.

### The features

| block | dims | treatment |
|---|---|---|
| emotion proportions | 9 | **used raw** — already a distribution, IDF would meaningless |
| style | 8 | one-hot, IDF-weighted |
| artist | 472 | one-hot, IDF-weighted — **§3 only**, see below |

`genre` is dropped. WikiArt stores a style list in that field rather than a subject taxonomy:
for every corpus row where it is present (98.1%; the other 1.9% are missing outright) it
contains the row's own style label, so it would add dimensions and no information. Real genre
codes (11 classes) exist in `wclasses.csv`, which §1.2 copies into `wikiart_256/` but nothing
currently loads.

### Artist is a feature in §3 and a leak in §4

The item-to-item track defines relevance as *same artist*. Putting artist into the content
vector there would hand the model the label — it would score every same-artist painting at
maximum similarity and report a near-perfect result that means nothing. So two feature
matrices are built:

- `W_user` (489 dims, artist included) — §3, where relevance is a user's held-out
  engagements and artist is a legitimate taste signal.
- `W_i2i` (17 dims, artist excluded) — §4, where artist is the target.

### What this is not

Metadata only — no pixels. This is the *metadata* content baseline that a visual model has to
beat, and with 17 dimensions on the item-to-item side it is a deliberately modest bar. The
visual method (I1, CLIP ViT-B/32 cosine) is specified and coded in §9 and has not been run;
its row in §4's table is empty on purpose.

In [36]:
# Content features ===================================
def one_hot(values):
    """Categorical column -> binary matrix, the lab's genre_matrix equivalent."""
    categories, index = np.unique(values, return_inverse=True)
    M = np.zeros((len(values), len(categories)), np.float32)
    M[np.arange(len(values)), index] = 1.0
    return M

def idf(M):
    """Inverse document frequency, exactly as in the lab."""
    document_frequency = (M > 0).sum(axis=0)
    return np.log((len(M) + 1) / (document_frequency + 1)) + 1.0

def l2_normalize(M):
    norms = np.linalg.norm(M, axis=1, keepdims=True)
    return np.divide(M, norms, out=np.zeros_like(M), where=norms > 0)

content_meta = (bridge.merge(pd.read_parquet("corpus.parquet"), on="file")
                      .sort_values("item_id").reset_index(drop=True))
EMO_FEATURES = [c for c in content_meta.columns if c.startswith("emo_")]

emotion_block = content_meta[EMO_FEATURES].to_numpy(np.float32)   # raw: already a distribution
style_block   = one_hot(content_meta["style"].to_numpy())
artist_block  = one_hot(content_meta.artist.to_numpy())

# artist is a taste signal for §3 but the LABEL for §4 — never put it in W_i2i
W_user = np.hstack([emotion_block, style_block * idf(style_block),
                    artist_block * idf(artist_block)]).astype(np.float32)
W_i2i  = np.hstack([emotion_block, style_block * idf(style_block)]).astype(np.float32)

W_user_n, W_i2i_n = l2_normalize(W_user), l2_normalize(W_i2i)
print(f"W_user {W_user.shape}  (emotions 9 + styles 8 + artists {artist_block.shape[1]})")
print(f"W_i2i  {W_i2i.shape}  (artist excluded — it is the item-to-item relevance label)")

W_user (20833, 489)  (emotions 9 + styles 8 + artists 472)
W_i2i  (20833, 17)  (artist excluded — it is the item-to-item relevance label)


In [37]:
# User profiles (§3) =================================
# profiles = P @ W_user, but P would be 12,000 x 20,833 (1 GB) if materialised in full.
# Sorting the impressions by user makes each chunk's slice of P small enough to build densely,
# which keeps the lab's matrix formulation without the memory cost.
positive_train = events_all[(events_all.split == "train") & (events_all.engaged == 1)]
preference = np.ones(len(positive_train), np.float32)   # binary feedback: one weight, 1.0

order = np.argsort(positive_train.user_id.to_numpy(), kind="stable")
u_sorted = positive_train.user_id.to_numpy()[order]
i_sorted = positive_train.item_id.to_numpy()[order]
p_sorted = preference[order]

CHUNK = 500
user_profiles = np.zeros((n_users, W_user.shape[1]), np.float32)
edges = np.searchsorted(u_sorted, np.arange(0, n_users + 1, CHUNK))
for lo, hi in zip(edges[:-1], edges[1:]):
    if lo == hi:
        continue
    users = u_sorted[lo:hi]
    first = users[0]
    P = np.zeros((users[-1] - first + 1, n_items), np.float32)
    P[users - first, i_sorted[lo:hi]] = p_sorted[lo:hi]
    user_profiles[first:users[-1] + 1] = P @ W_user

user_profiles_n = l2_normalize(user_profiles)
print(f"user profiles: {user_profiles_n.shape}, "
      f"{(np.abs(user_profiles_n).sum(1) > 0).sum():,} non-empty")

user profiles: (12000, 489), 11,722 non-empty


In [38]:
# §3: content-based top-K ============================
def content_top_k_for_all(users, k=K, chunk=500):
    """Score users in blocks — one 12,000 x 20,833 score matrix would be 1 GB."""
    lists = {}
    for start in range(0, len(users), chunk):
        block  = users[start:start + chunk]
        scores = user_profiles_n[block] @ W_user_n.T
        for row, user_id in enumerate(block):
            lists[int(user_id)] = top_k(scores[row], seen_by_user[user_id], k,
                                        tie_seed=int(user_id))
    return lists


def evaluate_and_record(name, lists):
    rows = [{"algorithm": name, **evaluate_user(r, test_positives[u])}
            for u, r in lists.items()]
    table = (pd.DataFrame(rows).groupby("algorithm")
             .agg(**{f"Hit Rate@{K}":  ("hit_rate", "mean"),
                     f"Precision@{K}": ("precision", "mean"),
                     f"Recall@{K}":    ("recall", "mean"),
                     f"MRR@{K}":       ("mrr", "mean"),
                     f"NDCG@{K}":      ("ndcg", "mean"),
                     "Average popularity": ("average_popularity", "mean"),
                     "Novelty (bits)":     ("novelty_bits", "mean")}).reset_index())
    cov, uniq = catalogue_coverage(lists)
    table["Catalogue coverage"] = cov
    table["Unique items recommended"] = uniq
    return record(table)

content_lists = content_top_k_for_all(eval_users)
recommendation_lists["Content (metadata)"] = content_lists
evaluate_and_record("Content (metadata)", content_lists).round(4)

,algorithm,Hit Rate@10,Precision@10,Recall@10,MRR@10,NDCG@10,Average popularity,Novelty (bits),Catalogue coverage,Unique items recommended
0,Popularity,0.2998,0.0396,0.0264,0.1313,0.0488,2070.2538,9.5421,0.0026,54
2,Content (metadata),0.0703,0.0080,0.0049,0.0256,0.0092,120.6574,14.7265,0.5159,10747
1,Random,0.0075,0.0008,0.0004,0.0020,0.0008,69.5521,15.6187,0.9930,20688


In [39]:
# §4: content-based "More like this" =================
# Only the 2,000 sampled queries are scored, so the 20,833 x 20,833 similarity matrix
# (1.7 GB) is never built.
def content_i2i_lists(queries, k=K):
    scores = W_i2i_n[queries] @ W_i2i_n.T          # 2,000 x 20,833
    scores[np.arange(len(queries)), queries] = -np.inf          # never return the query
    # §1.5's tie diagnostic — B2 has 17 dimensions and ties in bulk, so how it breaks ties
    # is part of its ranking function rather than an implementation detail
    ties = (scores == np.partition(scores, -k, axis=1)[:, -k][:, None]).sum(1)
    print(f"B2 candidates tied at rank {k}: mean {ties.mean():,.0f}, "
          f"median {np.median(ties):,.0f}, max {ties.max():,}")
    return {int(q): top_k(scores[row], np.array([q]), k, tie_seed=int(q))
            for row, q in enumerate(queries)}


def evaluate_and_record_i2i(name, lists):
    rows = [{"algorithm": name, **evaluate_query(r, q)} for q, r in lists.items()]
    table = (pd.DataFrame(rows).groupby("algorithm")
             .agg(**{f"Hit Rate@{K}":  ("hit_rate", "mean"),
                     f"Precision@{K}": ("precision", "mean"),
                     f"Recall@{K}":    ("recall", "mean"),
                     f"MRR@{K}":       ("mrr", "mean"),
                     f"NDCG@{K}":      ("ndcg", "mean"),
                     "Same-style share": ("same_style_share", "mean"),
                     "Novelty (bits)":   ("novelty_bits", "mean")}).reset_index())
    table["Catalogue coverage"] = len({int(i) for r in lists.values() for i in r}) / n_items
    return record_i2i(table)

i2i_content = content_i2i_lists(query_ids)
i2i_lists["B2 Content (emotion+style)"] = i2i_content
evaluate_and_record_i2i("B2 Content (emotion+style)", i2i_content).round(4)

B2 candidates tied at rank 10: mean 11, median 7, max 86


,algorithm,Hit Rate@10,Precision@10,Recall@10,MRR@10,NDCG@10,Same-style share,Novelty (bits),Catalogue coverage
2,B2 Content (emotion+style),0.2960,0.0434,0.0059,0.1022,0.0431,1.0000,15.5456,0.5915
1,B1 Random within style,0.2240,0.0261,0.0035,0.0710,0.0262,1.0000,15.6036,0.6162
0,B0 Random,0.0405,0.0041,0.0005,0.0127,0.0042,0.1498,15.6152,0.6145


In [40]:
# Content on the unbiased holdout (protocol 2) ==============
# Popularity won protocol 1 partly because exposure in the log is proportional to
# popularity by construction. Here exposure played no part in choosing the candidates.
unbiased_results = pd.DataFrame({
    "Random":     evaluate_reranking(
                      lambda u, c: np.random.default_rng(SEED_EVAL + u).random(len(c))),
    "Popularity": evaluate_reranking(lambda u, c: popularity[c]),
    "Content (metadata)": evaluate_reranking(
                      lambda u, c: user_profiles_n[u] @ W_user_n[c].T),
}).T
unbiased_results.round(4)

,hit_rate,precision,recall,mrr,ndcg,average_popularity
Random,0.9303,0.2895,0.4992,0.4932,0.4015,69.4848
Popularity,0.9445,0.3180,0.5530,0.5545,0.4552,125.6273
Content (metadata),0.9666,0.3551,0.6263,0.6065,0.5187,72.2733


### 5.1 Results relevance

**The two protocols disagree.**

| §3, NDCG@10 | logged test (MNAR) | unbiased holdout |
|---|---|---|
| Popularity | **0.0488** | 0.4552 |
| Content (metadata) | 0.0092 | **0.5187** |
| Random | 0.0008 | 0.4015 |

On the logged test set Popularity beats content 5.3×. On the uniformly-exposed holdout
content beats Popularity, and by a wider margin than Popularity beats Random (+0.064 against
+0.054). Same models, same catalogue, opposite conclusion.

The reason is built into §2.1: exposure in the log is proportional to popularity, so the
logged test set partly rewards a model for predicting *what the system showed people*.
Content scores a painting from its features alone and never sees an impression count, so it
gains nothing from that and is penalised by it. Reporting only the first column would have
concluded that content-based recommendation does not work here. It would have been measuring
the logging policy.

**Coverage behaves as it should.** Content reaches 10,747 of 20,833 paintings (51.6%) against
Popularity's 54 (0.26%) and Random's 20,688 (99.3%), with novelty of 14.7 bits against 9.5 and
15.6, and an average recommended popularity of 121 impressions against Popularity's 2,070. It
concentrates on what a user plausibly likes rather than on what everyone has already seen —
which is the behaviour an exploratory art-browsing product needs, and the reason the first
table's ranking should not drive the design decision on its own.

**On the item-to-item track, the emotion features carry the whole gain.**

| §4 | NDCG@10 | Hit Rate@10 | same-style share |
|---|---|---|---|
| B2 Content (emotion+style) | **0.0431** | 0.2960 | 1.000 |
| B1 Random within style | 0.0262 | 0.2240 | 1.000 |
| B0 Random | 0.0042 | 0.0405 | 0.150 |

B2 beats B1 by 1.65×, and both return same-style items 100% of the time. So the entire gain
comes from the 9 ArtEmis emotion proportions re-ordering paintings *within* a style — the
style component contributes nothing B1 did not already have. This is the most direct evidence
so far that the affective signal does measurable work on this task.

Track B uses no feedback data, so the implicit conversion should not have touched them — and every accuracy figure here
came back bit-identical to the star-rating run (Precision, Recall, MRR, NDCG, Hit Rate,
same-style share, coverage all unchanged to four decimals). The only Track B number that moved
was novelty, 15.5493 → 15.5456, which is the one metric computed from the impression log.

### 5.2 The annotator noise floor — split-half reliability

§10.E says the emotion vectors pool ~5.8 crowd annotators and that the proposal's split-half
reliability check was never run. This cell runs it. Every painting's annotations are dealt at
random into two halves (`SEED_SPLIT_HALF`); each half yields its own emotion vector. Three
things are reported:

- the **per-emotion split-half correlation** across paintings, with its Spearman–Brown
  correction to the full annotator count — the reliability of each emotion proportion as a
  measurement;
- the **per-painting agreement** between the two half-vectors — cosine, and whether the
  dominant emotion even agrees;
- **B2 re-run on either half alone**, style untouched, next to B2 on all annotators and B1.

The share of the B1 → B2 gain that survives on half the evidence
is the share that annotator sampling cannot be responsible for; and the overlap between the two
half-annotator B2 lists says whether the *lists* are stable, which is a different question from
whether the *metric* is.


In [41]:
# §5.2 — split-half reliability of the emotion vectors, and what it costs B2 ==========
# Every painting's ~5.8 annotations are dealt at random into two halves. Each half gives its
# own emotion vector; how well the two agree is the annotator-sampling noise floor, and
# re-running B2 on either half alone says how much of the B1 -> B2 gain survives with half
# the evidence. Style is unchanged in both variants, so only the emotion block moves.
SEED_SPLIT_HALF = 3

ann_emo = pd.read_parquet("artemis_utterances.parquet")[["file", "emotion"]]
ann_emo = ann_emo[ann_emo.file.isin(content_meta.file)]
ann_emo = ann_emo.sample(frac=1, random_state=SEED_SPLIT_HALF).reset_index(drop=True)
ann_emo["half"] = ann_emo.groupby("file").cumcount() % 2      # balanced within painting

def emotion_block_from(rows):
    """Per-painting emotion proportions in corpus (item_id) order, as §1.4.1 builds them."""
    counts = (rows.groupby(["file", "emotion"]).size().unstack(fill_value=0)
                  .reindex(columns=EMOTIONS, fill_value=0)
                  .reindex(content_meta.file, fill_value=0))
    n = counts.sum(axis=1).to_numpy()
    return counts.to_numpy(np.float32) / np.maximum(n, 1)[:, None], n

E_full, n_full = emotion_block_from(ann_emo)
E_a, n_a = emotion_block_from(ann_emo[ann_emo.half == 0])
E_b, n_b = emotion_block_from(ann_emo[ann_emo.half == 1])
assert np.allclose(E_full, emotion_block, atol=1e-5), "rebuilt emotion block differs from §1.4.1's"
both = (n_a > 0) & (n_b > 0)
print(f"annotations per painting: median {int(np.median(n_full))}, "
      f"halves of {int(np.median(n_a))} and {int(np.median(n_b))}; "
      f"{both.sum():,} / {n_items:,} paintings have both halves")

# ---- 1. reliability of the emotion vector itself ---------------------------------------
rel = pd.DataFrame({"share of annotations": E_full.mean(0)}, index=EMOTIONS)
r_half = np.array([np.corrcoef(E_a[both, k], E_b[both, k])[0, 1] for k in range(len(EMOTIONS))])
rel["split-half r"] = r_half
rel["Spearman-Brown (full n)"] = 2 * r_half / (1 + r_half)
cos_ab = (l2_normalize(E_a[both]) * l2_normalize(E_b[both])).sum(1)
dom_agree = (E_a[both].argmax(1) == E_b[both].argmax(1)).mean()
print(f"\nhalf-vs-half agreement per painting: mean cosine {cos_ab.mean():.3f}, "
      f"dominant emotion agrees on {dom_agree:.1%}\n")
display(rel.round(3))

# ---- 2. B2 on half the annotators ------------------------------------------------------
def i2i_lists_from(W, queries=query_ids, k=K):
    Wn = l2_normalize(W.astype(np.float32))
    scores = Wn[queries] @ Wn.T
    scores[np.arange(len(queries)), queries] = -np.inf
    return {int(q): top_k(scores[row], np.array([q]), k, tie_seed=int(q))
            for row, q in enumerate(queries)}

style_idf = style_block * idf(style_block)
variants = {"B2, all annotators":  i2i_content,
            "B2, half A only":     i2i_lists_from(np.hstack([E_a, style_idf])),
            "B2, half B only":     i2i_lists_from(np.hstack([E_b, style_idf])),
            "B1 (style only)":     i2i_lists["B1 Random within style"]}
def overlap(l1, l2):
    return np.mean([len(set(l1[q][:K]) & set(l2[q][:K])) / K for q in l1])
rows = []
for name, lists in variants.items():
    m = pd.DataFrame([evaluate_query(r, q) for q, r in lists.items()]).mean()
    rows.append({"variant": name, f"NDCG@{K}": m.ndcg, f"Precision@{K}": m.precision,
                 "top-10 overlap with full B2": overlap(lists, i2i_content)})
sh_table = pd.DataFrame(rows).set_index("variant")
half_ab_overlap = overlap(variants["B2, half A only"], variants["B2, half B only"])
print(f"\ntop-10 overlap between the two half-annotator B2 lists: {half_ab_overlap:.3f}")
b1 = sh_table.loc["B1 (style only)", f"NDCG@{K}"]; full = sh_table.loc["B2, all annotators", f"NDCG@{K}"]
half_mean = sh_table.loc[["B2, half A only", "B2, half B only"], f"NDCG@{K}"].mean()
print(f"B1 -> B2 gain: {full - b1:.4f} with all annotators, {half_mean - b1:.4f} with half "
      f"({(half_mean - b1) / (full - b1):.0%} of it survives on half the evidence)\n")
display(sh_table.round(4))


annotations per painting: median 5, halves of 3 and 2; 20,833 / 20,833 paintings have both halves

half-vs-half agreement per painting: mean cosine 0.469, dominant emotion agrees on 35.1%



,share of annotations,split-half r,Spearman-Brown (full n)
amusement,0.115,0.311,0.475
awe,0.163,0.207,0.343
contentment,0.249,0.331,0.498
excitement,0.084,0.163,0.281
anger,0.013,0.090,0.165
disgust,0.051,0.178,0.303
fear,0.105,0.378,0.548
sadness,0.112,0.359,0.528
something else,0.110,0.070,0.131



top-10 overlap between the two half-annotator B2 lists: 0.008
B1 -> B2 gain: 0.0170 with all annotators, 0.0117 with half (69% of it survives on half the evidence)



,NDCG@10,Precision@10,top-10 overlap with full B2
variant,,,
"B2, all annotators",0.0431,0.0433,1.0000
"B2, half A only",0.0370,0.0365,0.0767
"B2, half B only",0.0386,0.0386,0.0346
B1 (style only),0.0262,0.0261,0.0044


## 6. Bayesian ranking: engagement quality instead of exposure volume

With implicit binary feedback per painting: *successes = engagements,
trials = impressions*:

$$
p_i \sim \operatorname{Beta}(1,1),
\qquad
p_i \mid \text{data} \sim \operatorname{Beta}(1+s_i,\; 1+f_i)
$$

where $s_i$ is the number of impressions of painting *i* that were engaged with and $f_i$ the
number that were not. It is the click-through model.

### A different recommender, not a variant of Popularity

§3's Popularity baseline ranks by *how often a painting was shown*, which in this dataset is
exposure. Exposure and engagement quality are close to independent here, and §6's output
prints the correlation between the two — a low value is what makes the two rankings genuinely
different lists rather than reorderings of the same one.

Thousands of paintings have fewer than 10 training impressions, and a couple of hundred have
none at all. Ranking by the raw engagement rate puts paintings with one or two impressions at
the top:

| ranked by | what lands on top |
|---|---|
| raw rate | 2 impressions, both engaged → 1.000 |
| posterior mean | many impressions, nearly all engaged |
| lower 5% bound | many impressions, nearly all engaged — and penalised for thin evidence |

Three rankers are evaluated as follows:

- **Raw rate** $s/n$ — the naive estimate, kept to show the trap rather than as a serious
  method.
- **Posterior mean** $\frac{1+s}{2+n}$ — shrinks thin evidence toward the prior.
- **Lower 5% credible bound** — conservative; rewards quality *and* evidence, at the cost of
  making new-item discovery harder.

In [ ]:
# Beta-Bernoulli posteriors
BETA_PRIOR_A, BETA_PRIOR_B = 1.0, 1.0     # weak uniform prior
N_POSTERIOR_DRAWS = 2_000

# trials = impressions, successes = engagements.
# `engaged` is int8 to keep the parquet small; cast before summing so the aggregate cannot
# overflow on a heavily-exposed painting (pandas 3 happens to upcast, pandas 2 does not).
counts = (train.assign(engaged=train.engaged.astype(np.int32)).groupby("item_id").engaged
          .agg(trials="size", successes="sum")
          .reindex(range(n_items), fill_value=0))

alpha_post = BETA_PRIOR_A + counts.successes.to_numpy(float)
beta_post  = BETA_PRIOR_B + (counts.trials - counts.successes).to_numpy(float)

raw_rate       = np.where(counts.trials > 0,
                          counts.successes / counts.trials.clip(lower=1), 0.0)
posterior_mean = alpha_post / (alpha_post + beta_post)

# lower 5% credible bound by posterior sampling
rng_beta = np.random.default_rng(SEED_EVAL)
lower_bound = np.empty(n_items)
for start in range(0, n_items, 4_000):
    stop = min(start + 4_000, n_items)
    draws = rng_beta.beta(alpha_post[start:stop, None], beta_post[start:stop, None],
                          size=(stop - start, N_POSTERIOR_DRAWS))
    lower_bound[start:stop] = np.quantile(draws, 0.05, axis=1)

assert (lower_bound <= posterior_mean + 1e-9).all(), "bound must sit below the mean"
print(f"items with <10 training impressions: {(counts.trials < 10).sum():,} / {n_items:,}")
print(f"items with no training impressions : {(counts.trials == 0).sum():,} "
      f"(posterior mean = {posterior_mean[counts.trials.to_numpy() == 0][0]:.2f}, the prior)")
print(f"corr(impression count, posterior mean) = "
      f"{np.corrcoef(counts.trials[counts.trials > 0], posterior_mean[counts.trials > 0])[0,1]:.3f}"
      f"   <- low means quality and exposure are different rankings")

pd.DataFrame({"trials": counts.trials, "successes": counts.successes,
              "raw_rate": raw_rate, "posterior_mean": posterior_mean,
              "lower_bound": lower_bound}).nlargest(5, "raw_rate").round(3)

items with <10 training impressions: 4,377 / 20,833
items with no training impressions : 209 (posterior mean = 0.50, the prior)
corr(impression count, posterior mean) = 0.190   <- low means quality and exposure are different rankings


,trials,successes,raw_rate,posterior_mean,lower_bound
item_id,,,,,
18,1,1,1.0,0.667,0.233
72,1,1,1.0,0.667,0.214
124,5,5,1.0,0.857,0.607
221,2,2,1.0,0.750,0.356
383,2,2,1.0,0.750,0.373


In [43]:
# Evaluate the three rankers
# Non-personalized, like Popularity: one ranking, the same for every user apart from the
# per-user "already shown" exclusion.
BAYES_SCORES = {
    "Quality (raw rate)":       raw_rate,
    "Quality (posterior mean)": posterior_mean,
    "Quality (lower bound)":    lower_bound,
}

for name, scores in BAYES_SCORES.items():
    lists = {int(u): top_k(scores, seen_by_user[u], tie_seed=int(u)) for u in eval_users}
    recommendation_lists[name] = lists
    print(f"{name:<26} candidates tied at rank {K}: {ties_at_rank_k(scores):,}")
    table = evaluate_and_record(name, lists)

table.round(4)

Quality (raw rate)         candidates tied at rank 10: 331


Quality (posterior mean)   candidates tied at rank 10: 1


Quality (lower bound)      candidates tied at rank 10: 1


,algorithm,Hit Rate@10,Precision@10,Recall@10,MRR@10,NDCG@10,Average popularity,Novelty (bits),Catalogue coverage,Unique items recommended
0,Popularity,0.2998,0.0396,0.0264,0.1313,0.0488,2070.2538,9.5421,0.0026,54
2,Content (metadata),0.0703,0.0080,0.0049,0.0256,0.0092,120.6574,14.7265,0.5159,10747
5,Quality (lower bound),0.0322,0.0035,0.0018,0.0079,0.0033,167.3336,13.3468,0.0012,24
4,Quality (posterior mean),0.0218,0.0023,0.0012,0.0053,0.0021,113.9964,13.9806,0.0009,19
1,Random,0.0075,0.0008,0.0004,0.0020,0.0008,69.5521,15.6187,0.9930,20688
3,Quality (raw rate),0.0007,0.0001,0.0001,0.0002,0.0001,3.4118,18.7958,0.0159,331


In [44]:
# And on the unbiased holdout
unbiased_results = pd.DataFrame({
    "Random":     evaluate_reranking(
                      lambda u, c: np.random.default_rng(SEED_EVAL + u).random(len(c))),
    "Popularity": evaluate_reranking(lambda u, c: popularity[c]),
    "Content (metadata)": evaluate_reranking(
                      lambda u, c: user_profiles_n[u] @ W_user_n[c].T),
    **{name: evaluate_reranking(lambda u, c, s=scores: s[c])
       for name, scores in BAYES_SCORES.items()},
}).T
unbiased_results.round(4)

,hit_rate,precision,recall,mrr,ndcg,average_popularity
Random,0.9303,0.2895,0.4992,0.4932,0.4015,69.4848
Popularity,0.9445,0.3180,0.5530,0.5545,0.4552,125.6273
Content (metadata),0.9666,0.3551,0.6263,0.6065,0.5187,72.2733
Quality (raw rate),0.9614,0.3576,0.6246,0.6072,0.5244,88.0667
Quality (posterior mean),0.9609,0.3565,0.6224,0.6343,0.5317,85.8072
Quality (lower bound),0.9601,0.3581,0.6250,0.6446,0.5385,107.2878


### 6.1 What to notice

**The tiny-sample trap is a *selection* effect, and the two protocols make that visible.**

| NDCG@10 | full-catalogue retrieval (logged) | re-ranking 20 candidates (unbiased) |
|---|---|---|
| Quality (lower bound) | 0.0033 | **0.5385** |
| Quality (posterior mean) | 0.0021 | 0.5317 |
| Quality (raw rate) | **0.0001** | 0.5244 |
| Content (metadata) | 0.0092 | 0.5187 |
| Popularity | **0.0488** | 0.4552 |
| Random | 0.0008 | 0.4015 |

The raw rate scores *below Random* when retrieving from the full catalogue — 0.0001 against
0.0008 — while doing perfectly respectably at re-ranking. Taking the argmax over 20,833 posteriors 
systematically surfaces the noisiest estimates, because a painting shown twice and engaged with 
twice scores exactly 1.000 and nothing shown 200 times ever will. 
Re-ranking 20 candidates never
selects on that extreme, so the same broken estimator looks fine there.
the estimator.

331 candidates share the raw
rate's score at rank 10 — every painting engaged 2-of-2 sits at 1.000 — against exactly **1**
for both shrunk estimators. The raw rate is not ranking the top of its list at all; it is
drawing from a bucket, and §3's seeded tie-break is what decides which ten a user sees.

**Shrinkage helps, and more shrinkage helps more.** On the unbiased holdout the ordering is
lower bound (0.5385) > posterior mean (0.5317) > raw rate (0.5244), measured on this corpus.

**Quality beats exposure once exposure stops being rewarded.** Popularity leads the logged
column by 5.3× over content and 15× over the quality rankers, and comes *last but one* on the
unbiased column. Ranking by how often a painting was shown is a good predictor of what the
logging policy showed and a poor predictor of what people engage with — and the measured
correlation between impression count and posterior engagement quality is only **0.190**, which
is why the two produce different lists rather than reorderings of one another.

**Personalization is not yet earning its keep — changes are visibile in §8** The best
method in this table is a ranker with no user model at all: the same list for everyone. That
is a property of the simulator worth naming — §2 gives paintings large differences in average
appeal, and those item main effects are strong relative to the user–item interaction terms. A
personalized model has to beat 0.5385 to justify its complexity, and content (0.5187) does not.

**Coverage collapses further than Popularity's.** The posterior-mean and lower-bound rankers
reach 19 and 24 paintings of 20,833. The raw rate reaches 331 — not because it explores, but
because 331 items sit at exactly 1.000 and §3's seeded tie-break rotates which ten of them each
user is shown. Non-personalized ranking concentrates by construction; ranking on quality rather
than exposure does not fix that, it just changes which handful of paintings everyone sees.

## 7. Thompson sampling: an *online* experiment

§2's generative model produces an engagement probability for *any* (user, painting) pair, so it can
serve as a live
environment rather than a dataset. The bandit chooses, the simulator responds, and the loop
closes within a pure online experiment.


### Setup

The Beta–Bernoulli bandit, with the true click probabilities coming from the
simulator instead of being hardcoded:

| | |
|---|---|
| arms | 500 paintings sampled from the corpus |
| round | a random visitor arrives, the policy shows one painting |
| reward | a Bernoulli draw from §2.1's engagement probability for that (visitor, arm) pair |
| hidden truth | each arm's true engagement rate, from §2's utility — never shown to the policy |

The reward function is reconstructed from `synthetic_ORACLE_do_not_train_on_this.npz`, which
is legitimate here: the oracle is the *environment*, not training data. Because §2.1's
observation model is an explicit Bernoulli draw with a known probability, the reconstruction
is **exact** — the environment returns the true per-pair engagement probability rather than
a Monte-Carlo estimate of it, and each arm's true rate is computed in closed form instead of
by repeated sampling.

### Policies, and the variable prior

- **Random** — the floor.
- **Greedy** — always show the arm with the highest posterior mean.
- **Thompson** — sample one plausible rate per arm, show the argmax.

Both learners are run under two priors:

- `Beta(1,1)` — mean 0.5. But the median arm's true rate
  sits near `LIKE_RATE`, so 0.5 is wildly optimistic — and optimism is itself an exploration
  mechanism, which is the point the comparison below makes.
- `Beta(1,3)` — mean 0.25, roughly calibrated to the corpus.

In [45]:
# The environment, from the simulator
N_ARMS, SEED_BANDIT = 500, 11
oracle = np.load("synthetic_ORACLE_do_not_train_on_this.npz")   # the ENVIRONMENT, not training data

emotion_props = content_meta[EMO_FEATURES].to_numpy(np.float32)
agreement = (1 - content_meta.emotion_entropy.to_numpy(np.float32) / np.log(9)).clip(0, 1)

rng_pool = np.random.default_rng(SEED_BANDIT)
ARMS = np.sort(rng_pool.choice(n_items, N_ARMS, replace=False))

# Pre-compute every visitor's latent utility for every arm once, so a round costs one
# uniform draw instead of a matrix product. This is §2.1's utility, reassembled.
Z  = 2.2 * (oracle["U_emo"] @ emotion_props[ARMS].T) * agreement[ARMS]
Z += 0.7 * oracle["U_style"][:, oracle["style_idx"][ARMS]]
Z += 0.5 * oracle["U_artist"][:, oracle["artist_idx"][ARMS]]
Z += 0.9 * (oracle["P"] @ oracle["Q"][ARMS].T)
Z += 0.6 * oracle["pop_taste"][:, None] * oracle["logpop_z"][ARMS]
Z = (oracle["s_user"][:, None] * Z + oracle["b_user"][:, None]).astype(np.float32)

# §2.1's observation model, applied to the arms. Because engagement is an explicit Bernoulli
# draw with a known probability, the environment is EXACT here: no Monte-Carlo estimate of
# each arm's true rate is needed, and a round is a single comparison against P_ENGAGE.
ENGAGE_C   = float(oracle["engage_c"])
ENGAGE_LAM = float(oracle["engage_lam"])
P_ENGAGE   = sigmoid(ENGAGE_LAM * (Z - ENGAGE_C)).astype(np.float32)   # visitors x arms
true_ctr   = P_ENGAGE.mean(axis=0)                                     # hidden ground truth

print(f"arms: {N_ARMS}   rounds draw from {Z.shape[0]:,} visitors")
print(f"true engagement rate per arm: min {true_ctr.min():.3f}, "
      f"median {np.median(true_ctr):.3f}, max {true_ctr.max():.3f}")
print(f"arms above the Beta(1,1) prior mean of 0.5: {(true_ctr > 0.5).sum()} of {N_ARMS}")

arms: 500   rounds draw from 12,000 visitors
true engagement rate per arm: min 0.068, median 0.262, max 0.554
arms above the Beta(1,1) prior mean of 0.5: 23 of 500


In [46]:
# The three policies
def run_bandit(policy, prior=(1.0, 1.0), n_rounds=20_000, seed=1):
    """One pass of the tutorial's loop: choose an arm, observe a reward, update that arm."""
    rng_b = np.random.default_rng(seed)
    alpha = np.full(N_ARMS, prior[0], float)
    beta_ = np.full(N_ARMS, prior[1], float)
    visitors = rng_b.integers(0, Z.shape[0], n_rounds)
    rewards  = np.zeros(n_rounds, int)
    chosen   = np.zeros(n_rounds, int)

    for t in range(n_rounds):
        if policy == "Random":
            arm = int(rng_b.integers(N_ARMS))
        elif policy == "Greedy":
            arm = int(np.argmax(alpha / (alpha + beta_) + rng_b.random(N_ARMS) * 1e-9))
        else:                                              # Thompson
            arm = int(np.argmax(rng_b.beta(alpha, beta_)))

        # the environment answers with a real Bernoulli draw from §2.1's observation model
        reward = int(rng_b.random() < P_ENGAGE[visitors[t], arm])
        alpha[arm] += reward
        beta_[arm] += 1 - reward
        rewards[t], chosen[t] = reward, arm

    pulls = np.bincount(chosen, minlength=N_ARMS)
    return {"cumulative engagement rate": rewards.mean(),
            "final 10% rate":     rewards[-n_rounds // 10:].mean(),
            "arms ever shown":    int((pulls > 0).sum()),
            "top arm share":      pulls.max() / n_rounds,
            "true rate of that arm": true_ctr[pulls.argmax()]}


rows = []
for n_rounds in (20_000, 100_000):
    for policy, prior in [("Random", (1, 1)), ("Greedy", (1, 1)), ("Thompson", (1, 1)),
                          ("Greedy", (1, 3)), ("Thompson", (1, 3))]:
        rows.append({"rounds": n_rounds, "policy": policy,
                     "prior": f"Beta{prior}", **run_bandit(policy, prior, n_rounds)})

bandit_results = pd.DataFrame(rows)
print(f"oracle: best arm has a true engagement rate of {true_ctr.max():.3f}\n")
bandit_results.round(3)

oracle: best arm has a true engagement rate of 0.554



,rounds,policy,prior,cumulative engagement rate,final 10% rate,arms ever shown,top arm share,true rate of that arm
0,20000,Random,"Beta(1, 1)",0.278,0.257,500,0.003,0.324
1,20000,Greedy,"Beta(1, 1)",0.527,0.540,399,0.933,0.534
2,20000,Thompson,"Beta(1, 1)",0.384,0.468,500,0.036,0.538
3,20000,Greedy,"Beta(1, 3)",0.396,0.392,4,0.999,0.397
4,20000,Thompson,"Beta(1, 3)",0.410,0.450,500,0.048,0.528
5,100000,Random,"Beta(1, 1)",0.279,0.282,500,0.003,0.290
6,100000,Greedy,"Beta(1, 1)",0.535,0.533,14,1.000,0.538
7,100000,Thompson,"Beta(1, 1)",0.483,0.523,500,0.181,0.554
8,100000,Greedy,"Beta(1, 3)",0.552,0.551,3,1.000,0.554
9,100000,Thompson,"Beta(1, 3)",0.491,0.522,500,0.127,0.546


In [ ]:
# Learning curves
# Output: table about the engagement rate in each fifth of the run.
def learning_curve(policy, prior, n_rounds=100_000, bins=5, seed=1):
    rng_b = np.random.default_rng(seed)
    alpha = np.full(N_ARMS, prior[0], float); beta_ = np.full(N_ARMS, prior[1], float)
    visitors = rng_b.integers(0, Z.shape[0], n_rounds); rewards = np.zeros(n_rounds, int)
    for t in range(n_rounds):
        if policy == "Greedy":
            arm = int(np.argmax(alpha / (alpha + beta_) + rng_b.random(N_ARMS) * 1e-9))
        else:
            arm = int(np.argmax(rng_b.beta(alpha, beta_)))
        reward = int(rng_b.random() < P_ENGAGE[visitors[t], arm])
        alpha[arm] += reward; beta_[arm] += 1 - reward; rewards[t] = reward
    return [rewards[i * n_rounds // bins:(i + 1) * n_rounds // bins].mean() for i in range(bins)]

curves = pd.DataFrame(
    {f"{p} Beta{pr}": learning_curve(p, pr)
     for p, pr in [("Greedy", (1, 1)), ("Thompson", (1, 1)),
                   ("Greedy", (1, 3)), ("Thompson", (1, 3))]},
    index=[f"rounds {i*20}-{(i+1)*20}%" for i in range(5)])
curves.round(3)

,"Greedy Beta(1, 1)","Thompson Beta(1, 1)","Greedy Beta(1, 3)","Thompson Beta(1, 3)"
rounds 0-20%,0.536,0.382,0.552,0.409
rounds 20-40%,0.529,0.470,0.547,0.488
rounds 40-60%,0.534,0.509,0.552,0.514
rounds 60-80%,0.539,0.524,0.557,0.524
rounds 80-100%,0.535,0.529,0.553,0.521


### 7.1 Results relevance

**Greedy wins on cumulative reward here.**

| 100,000 rounds | cumulative engagement | final 10% | arms shown | top arm's share | that arm's true rate |
|---|---|---|---|---|---|
| Greedy Beta(1,3) | **0.552** | 0.551 | **3** | 100% | 0.554 |
| Greedy Beta(1,1) | 0.535 | 0.533 | 14 | 100% | 0.538 |
| Thompson Beta(1,3) | 0.491 | 0.522 | **500** | 13% | 0.546 |
| Thompson Beta(1,1) | 0.483 | 0.523 | **500** | 18% | 0.554 |
| Random | 0.279 | 0.282 | 500 | 0.3% | — |

Compare greedy under
`Beta(1,3)` at the two horizons: at 20,000 rounds it locked onto 4 arms and scored **0.396**;
at 100,000 rounds it locked onto 3 and scored **0.552**. Same policy, same prior, same seed.
The difference is entirely which arm it happened to commit to before its posterior stopped
moving. One run found the best arm in the pool (0.554); the other found a mediocre one (0.397)
and never looked again. Thompson's result barely moves across priors or horizons
(0.483 / 0.491 at 100k), because its exploration comes from posterior uncertainty rather than
from the prior happening to be optimistic.

**Thompson pays an exploration cost that shrinks with the horizon.** Cumulative 0.483 against
greedy's 0.535 under the same prior, but the final-10% rates are 0.523 and 0.533 — nearly
converged, while still keeping all 500 arms alive. The learning-curve across fifths of the run: 
Thompson goes 0.382 → 0.470 → 0.509 → 0.524 → 0.529 while
greedy is flat at ~0.535 from the first fifth, because it stopped learning almost immediately.

§3's Popularity baseline reached 54
of 20,833 paintings; §6's quality rankers reached 19–24. Greedy here reproduces exactly that
failure — **three arms out of 500, one of them taking 100% of impressions** — and it does so
*while winning the accuracy metric*. Thompson gives up ~1 percentage point of final reward to
keep the entire catalogue in circulation.

## 8. Item-kNN — collaborative similarity, on both tracks

This is the first model that learns from *behaviour* rather than
metadata, and it serves both tracks (A and B) from one similarity matrix:

- **§3** — score a user's candidates from the items they already liked.
- **§4** — the neighbours of a painting *are* the "More like this" list. This is the
  collaborative answer to the question B1 and B2 answered from content.

### Three adaptations
**1. Similarity is computed on binary positives, which halves the work.** On binary vectors the cosine
numerator *is* the overlap count, so one matrix product yields both:

$$
n_{ij}=|U_i \cap U_j|,\qquad
s_{ij}=\frac{n_{ij}}{\sqrt{|U_i||U_j|}},\qquad
s'_{ij}=\frac{n_{ij}}{n_{ij}+\lambda}\,s_{ij}
$$

**2. The full 20,833 × 20,833 similarity matrix is never built**. Items are processed in blocks of 1,000 and only the
top 50 neighbours per item are kept. The full pass takes about 12 seconds.

**3. The shrinkage and the hyperparameters.** The median
painting here has few training engagements per item , so
overlaps are much thinner:

| min overlap | median partners | items with none |
|---|---|---|
| 2 | 783 | 5.7% |
| 5 | 43 | 25.7% |
| 10 | 0 | 51.3% |

`MIN_ITEM_OVERLAP = 2` is kept, but `SHRINKAGE` is limited to **5**.

### The backfill is deliberately *not* popularity

The backfill here is the Beta posterior mean from §6: evidence-aware, and nearly orthogonal to exposure.

Unsupported candidates are always ranked *below* every evidence-backed
one, so a hit can be attributed to the model or to the fallback. §8.1 reports that split.

In [48]:
# Item-item neighbourhoods
SHRINKAGE, MIN_ITEM_OVERLAP, ITEM_NEIGHBORS = 5.0, 2, 50
BLOCK = 1_000

positives = events_all[(events_all.split == "train") & (events_all.engaged == 1)]
# group once: scanning the multi-million-row frame per user would dominate everything below.
# The weight is 1.0 for every positive — binary feedback has no strength to carry.
positives_by_user = {u: (g.item_id.to_numpy(), np.ones(len(g), np.float32))
                     for u, g in positives.groupby("user_id")}

M = np.zeros((n_items, n_users), np.float32)          # binary positives, items x users
M[positives.item_id.to_numpy(), positives.user_id.to_numpy()] = 1.0
item_counts = M.sum(axis=1)

neighbour_idx = np.zeros((n_items, ITEM_NEIGHBORS), np.int32)
neighbour_sim = np.zeros((n_items, ITEM_NEIGHBORS), np.float32)
for start in range(0, n_items, BLOCK):
    stop = min(start + BLOCK, n_items)
    overlap = M[start:stop] @ M.T                     # binary -> this IS the cosine numerator
    denom = np.sqrt(item_counts[start:stop, None] * item_counts[None, :])
    denom[denom == 0] = 1.0
    sim = (overlap / denom) * (overlap / (overlap + SHRINKAGE))    # cosine x reliability
    sim[overlap < MIN_ITEM_OVERLAP] = 0.0
    sim[np.arange(stop - start), np.arange(start, stop)] = 0.0     # no self-similarity
    top = np.argpartition(-sim, ITEM_NEIGHBORS, axis=1)[:, :ITEM_NEIGHBORS]
    rows = np.arange(stop - start)[:, None]
    order = np.argsort(-sim[rows, top], axis=1)
    neighbour_idx[start:stop] = top[rows, order]
    neighbour_sim[start:stop] = sim[rows, top[rows, order]]
del M

n_neighbours = (neighbour_sim > 0).sum(axis=1)
print(f"items with >=1 neighbour : {(n_neighbours > 0).sum():,} / {n_items:,} "
      f"({(n_neighbours > 0).mean():.1%})")
print(f"items with >={K} neighbours: {(n_neighbours >= K).sum():,} "
      f"({(n_neighbours >= K).mean():.1%})")
print(f"mean top-1 similarity     : {neighbour_sim[:, 0].mean():.3f}")

items with >=1 neighbour : 18,375 / 20,833 (88.2%)
items with >=10 neighbours: 16,914 (81.2%)
mean top-1 similarity     : 0.059


In [ ]:
# The two scoring variants
# weighted sum : s(u,i) = sum_j p_uj s'_ij          -- preserves how much evidence supports i
# normalized   : divided by the supporting similarity mass -- average preference strength
BACKFILL = posterior_mean          # §6, not popularity — see the note above

def item_knn_lists(user_id, variant, k=K):
    pref = np.zeros(n_items, np.float32)
    items, weights = positives_by_user.get(user_id, (np.array([], int), np.array([])))
    pref[items] = weights

    gathered = pref[neighbour_idx]                       # (n_items, ITEM_NEIGHBORS)
    weighted_sum = (neighbour_sim * gathered).sum(axis=1)
    support_mass = (neighbour_sim * (gathered > 0)).sum(axis=1)
    contributors = ((gathered > 0) & (neighbour_sim > 0)).sum(axis=1)

    primary = weighted_sum if variant == "sum" else np.divide(
        weighted_sum, support_mass, out=np.zeros_like(weighted_sum), where=support_mass > 0)

    # evidence-backed scores always outrank the backfill
    scores = np.where(contributors >= 1, primary + 1.0, BACKFILL)
    return top_k(scores, seen_by_user[user_id], k, tie_seed=int(user_id)), contributors


knn_support = {}
for variant, name in [("sum", "Item-kNN (weighted sum)"), ("norm", "Item-kNN (normalized)")]:
    lists, supported_targets, backed_hits, hits = {}, [], 0, 0
    for u in eval_users:
        ranked, contributors = item_knn_lists(int(u), variant)
        lists[int(u)] = ranked
        targets = test_positives[u]
        supported_targets.append(np.mean([contributors[t] >= 1 for t in targets]))
        hit = np.isin(ranked, list(targets))
        hits += hit.any()
        backed_hits += bool((hit & (contributors[ranked] >= 1)).any())
    recommendation_lists[name] = lists
    knn_support[name] = {"Target support rate": float(np.mean(supported_targets)),
                         f"Hit@{K}": hits / len(eval_users),
                         f"Evidence-backed Hit@{K}": backed_hits / len(eval_users)}
    table = evaluate_and_record(name, lists)

print(pd.DataFrame(knn_support).T.round(4).to_string(), "\n")
table.round(4)

                         Target support rate  Hit@10  Evidence-backed Hit@10
Item-kNN (weighted sum)               0.8085  0.4453                  0.4453
Item-kNN (normalized)                 0.8085  0.0127                  0.0127 



,algorithm,Hit Rate@10,Precision@10,Recall@10,MRR@10,NDCG@10,Average popularity,Novelty (bits),Catalogue coverage,Unique items recommended
6,Item-kNN (weighted sum),0.4453,0.0776,0.0489,0.2432,0.0986,995.2536,10.8584,0.0736,1533
0,Popularity,0.2998,0.0396,0.0264,0.1313,0.0488,2070.2538,9.5421,0.0026,54
2,Content (metadata),0.0703,0.0080,0.0049,0.0256,0.0092,120.6574,14.7265,0.5159,10747
5,Quality (lower bound),0.0322,0.0035,0.0018,0.0079,0.0033,167.3336,13.3468,0.0012,24
4,Quality (posterior mean),0.0218,0.0023,0.0012,0.0053,0.0021,113.9964,13.9806,0.0009,19
7,Item-kNN (normalized),0.0127,0.0013,0.0010,0.0041,0.0015,94.5352,14.9596,0.8555,17823
1,Random,0.0075,0.0008,0.0004,0.0020,0.0008,69.5521,15.6187,0.9930,20688
3,Quality (raw rate),0.0007,0.0001,0.0001,0.0002,0.0001,3.4118,18.7958,0.0159,331


In [50]:
# §4: neighbours as "More like this"
# For a query, its item neighbours are the recommendation. Where a query has fewer than K
# neighbours, the remainder is filled the same way B1 does — random within the same style —
# so the comparison against B1/B2 stays like-for-like.
def item_knn_i2i_lists(queries, k=K):
    lists = {}
    for q in queries:
        q = int(q)
        rng_q = np.random.default_rng(SEED_EVAL + q)
        scores = rng_q.random(n_items) * 0.1                       # backfill floor
        scores[item_style == item_style[q]] += 0.1                 # ... within style, as B1
        supported = neighbour_sim[q] > 0
        scores[neighbour_idx[q][supported]] = 1.0 + neighbour_sim[q][supported]
        lists[q] = top_k(scores, np.array([q]), k, tie_seed=q)
    return lists

i2i_knn = item_knn_i2i_lists(query_ids)
i2i_lists["B3 Item-kNN (collaborative)"] = i2i_knn
evaluate_and_record_i2i("B3 Item-kNN (collaborative)", i2i_knn).round(4)

,algorithm,Hit Rate@10,Precision@10,Recall@10,MRR@10,NDCG@10,Same-style share,Novelty (bits),Catalogue coverage
2,B2 Content (emotion+style),0.2960,0.0434,0.0059,0.1022,0.0431,1.0000,15.5456,0.5915
1,B1 Random within style,0.2240,0.0261,0.0035,0.0710,0.0262,1.0000,15.6036,0.6162
3,B3 Item-kNN (collaborative),0.0860,0.0095,0.0012,0.0288,0.0100,0.3554,13.2179,0.4255
0,B0 Random,0.0405,0.0041,0.0005,0.0127,0.0042,0.1498,15.6152,0.6145


In [51]:
# Both models on the unbiased holdout
def knn_scores_for(user_id, candidates, variant="sum"):
    pref = np.zeros(n_items, np.float32)
    items, weights = positives_by_user.get(user_id, (np.array([], int), np.array([])))
    pref[items] = weights
    gathered = pref[neighbour_idx[candidates]]
    sims = neighbour_sim[candidates]
    weighted_sum = (sims * gathered).sum(axis=1)
    contributors = ((gathered > 0) & (sims > 0)).sum(axis=1)
    return np.where(contributors >= 1, weighted_sum + 1.0, BACKFILL[candidates])

unbiased_results = pd.DataFrame({
    "Random":     evaluate_reranking(
                      lambda u, c: np.random.default_rng(SEED_EVAL + u).random(len(c))),
    "Popularity": evaluate_reranking(lambda u, c: popularity[c]),
    "Content (metadata)": evaluate_reranking(
                      lambda u, c: user_profiles_n[u] @ W_user_n[c].T),
    "Quality (lower bound)": evaluate_reranking(lambda u, c: lower_bound[c]),
    "Item-kNN (weighted sum)": evaluate_reranking(knn_scores_for),
}).T
unbiased_results.round(4)

,hit_rate,precision,recall,mrr,ndcg,average_popularity
Random,0.9303,0.2895,0.4992,0.4932,0.4015,69.4848
Popularity,0.9445,0.3180,0.5530,0.5545,0.4552,125.6273
Content (metadata),0.9666,0.3551,0.6263,0.6065,0.5187,72.2733
Quality (lower bound),0.9601,0.3581,0.6250,0.6446,0.5385,107.2878
Item-kNN (weighted sum),0.9656,0.3660,0.6417,0.7061,0.5717,99.3154


### 8.1 Results relevance

**Item-kNN is the first method to win *both* protocols, and it is not close.**

| NDCG@10 | full-catalogue retrieval (logged) | re-ranking 20 candidates (unbiased) |
|---|---|---|
| **Item-kNN (weighted sum)** | **0.0986** | **0.5717** |
| Quality (lower bound) | 0.0033 | 0.5385 |
| Content (metadata) | 0.0092 | 0.5187 |
| Popularity | 0.0488 | 0.4552 |
| Random | 0.0008 | 0.4015 |

Every other method in this notebook wins one protocol and loses the other. Item-kNN doubles
Popularity on the logged set (0.0986 against 0.0488) *and* beats the best non-personalized
ranker on the unbiased holdout (0.5717 against 0.5385). That is the answer to **RQ3**: the
co-engagement signal recovers preference structure that no content feature can reach — which
is exactly what §2.1 built into the data, in the form of a 16-dimensional latent term that
content explains none of and §2.2 measured at $R^2 = 0.496$ overall.

Personalization finally earns its keep:
0.5717 against the 0.5385 that a single global ranking achieves.

**Every hit is evidence-backed.** Hit@10 and evidence-backed Hit@10 are both 0.4453 — not one
hit in 10,448 users came from the §6 posterior-mean backfill. 80.9% of held-out positives have
at least one neighbourhood contributor behind them. Had the backfill been popularity,that separation would have been impossible to check and the exposure bias §2
exists to detect would have been injected straight back into the scores.

**The normalization variant collapses.** Dividing by the
supporting similarity mass drops NDCG@10 from 0.0986 to 0.0015 — below Random. Normalizing
turns "many neighbours support this, weakly" into the same score as "one neighbour supports
this, strongly", so an item with a single thin connection to the user's history outranks one
with fifty. On sparse binary data the amount of evidence *is* most of the signal, and the
weighted sum keeps it. Its coverage tells the same story from the other side: 17,823 items
(85.6%) against the weighted sum's 1,533 (7.4%) — it is spreading recommendations over the
whole catalogue because its scores are close to meaningless.

**Coverage is the cost.** The weighted sum reaches 1,533 paintings and 10.9
bits of novelty, against content's 10,747 and 14.7. It is far better than Popularity's 54, but
it is a concentrating recommender: it recommends what people already engage with, which is
what collaborative filtering does. In a product this is the argument for hybridising it with
§5's content model rather than shipping it alone.

**On Track B it loses badly.** B3 reaches NDCG@10 0.0100 against
B2's 0.0431 and B1's 0.0262 — worse than picking randomly within the query's style. Its
same-style share is 0.355, against 1.000 for both content baselines. Co-engagement similarity
does not recover *authorship*: two paintings get co-engaged because they appeal to the same
kind of visitor, and a visitor's taste spans artists. That is the second half of **RQ4**, and
the answer is no.

### 8.2 Implicit ALS — the same signal, a latent-factor estimator

Item-kNN and implicit ALS see exactly the same thing: the binary train engagement matrix.
Item-kNN estimates item–item similarity directly and ranks by neighbourhood evidence; implicit
ALS factorises the whole matrix — every cell, with
"not engaged" as weak evidence of zero preference at confidence 1 and each engagement at
confidence $1+\alpha$ — into $f$-dimensional user and item factors, and ranks by their dot
product. This steps incldues no new information, a different
estimator of it. The question is whether a global factorisation recovers §2.1's hidden factor
better than 50 nearest neighbours do. `fit_implicit_als` is defined in §2.3, where the same
model is the collaborative reference in the sensitivity sweeps.

**The first tuned method, and the use of the validation split.** $(f, \lambda, \alpha)$
come from a fixed 36-point grid, scored by NDCG@10 against each user's validation engagements
with the full catalogue as candidates and the user's train impressions excluded — protocol 1's
shape, on val instead of test. Every fit uses train alone, like every other rung, so the chosen
model needs no refit and sees the same data as item-kNN. It is then scored on P1, P2 and, as
**B4**, on Track B by cosine between item factors: B3's question with the other estimator.


In [52]:
# §8.2 — implicit ALS: tune on the validation split, then fit on train alone ============
# The first tuned method on Track A and the first use of `val`. Every fit below uses train
# only; val supplies the targets. Protocol-1 shape: rank the full catalogue with the user's
# train impressions excluded, score the top-10 against their validation engagements.
SEED_ALS = 5
val           = events_all[events_all.split == "val"]
val_positives = (val[val.engaged == 1].groupby("user_id").item_id
                 .apply(lambda s: set(s.to_numpy())).to_dict())
val_users     = np.array(sorted(val_positives))
train_shown   = train.groupby("user_id").item_id.apply(lambda s: s.to_numpy()).to_dict()

unbiased_val = pd.read_parquet("synthetic_interactions_unbiased_val.parquet")   # §2's second draw

def val_ndcg(scores_for, k=K, chunk=500):
    """Protocol-1 shape on `val`: `scores_for(block_of_users)` returns their full-catalogue
    score rows; train impressions are excluded and the top-k scored against val engagements."""
    disc, total = np.log2(np.arange(2, k + 2)), 0.0
    for start in range(0, len(val_users), chunk):
        blk    = val_users[start:start + chunk]
        scores = scores_for(blk)
        for row, u in enumerate(blk):
            scores[row, train_shown[u]] = -np.inf
        top = np.argpartition(-scores, k, axis=1)[:, :k]
        for row, u in enumerate(blk):
            ranked = top[row][np.argsort(-scores[row, top[row]])]
            hits   = np.isin(ranked, list(val_positives[u])).astype(float)
            total += (hits / disc).sum() / (1 / disc[:min(len(val_positives[u]), k)]).sum()
    return total / len(val_users)

def rerank_ndcg(frame, score_fn, k=K):
    """Protocol-2 shape on a uniformly-exposed holdout: `evaluate_reranking`'s loop on any
    frame, mean NDCG only."""
    out = []
    for user_id, g in frame.groupby("user_id"):
        candidates = g.item_id.to_numpy()
        relevant   = g.engaged.to_numpy().astype(float)
        if relevant.sum() == 0:
            continue
        order = np.argsort(-score_fn(user_id, candidates))
        disc  = np.log2(np.arange(2, min(k, len(candidates)) + 2))
        hits  = relevant[order][:k]
        out.append((hits / disc).sum() / (np.sort(relevant)[::-1][:k] / disc).sum())
    return float(np.mean(out))

ALS_GRID = [(f, lam, alpha) for f in (8, 16, 32)          # f = 64 was dominated under both shapes
                            for lam in (0.1, 1.0, 10.0)      # and is dropped for f = 8; alpha = 40
                            for alpha in (2.0, 5.0, 10.0, 20.0, 40.0)]   # extends the P2-shape edge
als_grid, als_best, als_best_p2 = [], (-1.0, None), (-1.0, None)
for f, lam, alpha in ALS_GRID:
    X, Y = fit_implicit_als(train, n_items, n_users, f=f, lam=lam, alpha=alpha, seed=SEED_ALS)
    s1 = val_ndcg(lambda blk, X=X, Y=Y: X[blk] @ Y.T)                       # P1 shape
    s2 = rerank_ndcg(unbiased_val, lambda u, c, X=X, Y=Y: Y[c] @ X[u])      # P2 shape
    als_grid.append({"f": f, "lambda": lam, "alpha": alpha,
                     "val NDCG@10, P1 shape": s1, "val NDCG@10, P2 shape": s2})
    if s1 > als_best[0]:
        als_best = (s1, (f, lam, alpha, X, Y))
    if s2 > als_best_p2[0]:
        als_best_p2 = (s2, (f, lam, alpha, X, Y))
als_grid = pd.DataFrame(als_grid)
ALS_F, ALS_LAM, ALS_ALPHA, X_als, Y_als = als_best[1]                    # the rung: P1 shape
ALS_F_P2, ALS_LAM_P2, ALS_ALPHA_P2, X_als_p2, Y_als_p2 = als_best_p2[1]  # the sensitivity check
print(f"validation users : {len(val_users):,}   grid : {len(ALS_GRID)} fits, train only")
print(f"chosen           : f={ALS_F}, lambda={ALS_LAM}, alpha={ALS_ALPHA} on the P1 shape "
      f"(val NDCG@{K} {als_best[0]:.4f})")
print(f"P2 shape would choose f={ALS_F_P2}, lambda={ALS_LAM_P2}, alpha={ALS_ALPHA_P2} "
      f"(val NDCG@{K} {als_best_p2[0]:.4f})\n")
display(als_grid.pivot_table(index=["f", "lambda"], columns="alpha",
                             values="val NDCG@10, P1 shape").round(4))
print("\nthe same grid under the protocol-2 shape\n")
display(als_grid.pivot_table(index=["f", "lambda"], columns="alpha",
                             values="val NDCG@10, P2 shape").round(4))


validation users : 10,619   grid : 36 fits, train only
chosen           : f=16, lambda=10.0, alpha=5.0   (val NDCG@10 0.1068)



alpha        2.0     5.0     10.0    20.0
f  lambda                                
16 0.1     0.1058  0.1061  0.1030  0.0960
   1.0     0.1058  0.1062  0.1029  0.0961
   10.0    0.1063  0.1068  0.1037  0.0966
32 0.1     0.1015  0.1049  0.1034  0.0934
   1.0     0.1017  0.1051  0.1034  0.0935
   10.0    0.1023  0.1057  0.1041  0.0945
64 0.1     0.0836  0.0892  0.0876  0.0788
   1.0     0.0838  0.0894  0.0876  0.0789
   10.0    0.0861  0.0910  0.0891  0.0800

In [53]:
# §8.2 — the chosen model on protocol 1, protocol 2 and Track B ========================
def als_lists_for_all(users, k=K, chunk=500):
    lists = {}
    for start in range(0, len(users), chunk):
        blk    = users[start:start + chunk]
        scores = X_als[blk] @ Y_als.T
        for row, u in enumerate(blk):
            lists[int(u)] = top_k(scores[row], seen_by_user[u], k, tie_seed=int(u))
    return lists

als_lists = als_lists_for_all(eval_users)
recommendation_lists["Implicit ALS"] = als_lists
display(evaluate_and_record("Implicit ALS", als_lists).round(4))

# Track B — B4: cosine between item factors. Like B3 it sees only the synthetic co-engagement
# log, so it asks B3's question with the other estimator: does a global factorisation recover
# authorship any better than 50 nearest neighbours do?
Y_als_n = l2_normalize(Y_als)
def als_i2i_lists(queries, k=K):
    scores = Y_als_n[queries] @ Y_als_n.T
    scores[np.arange(len(queries)), queries] = -np.inf
    return {int(q): top_k(scores[row], np.array([q]), k, tie_seed=int(q))
            for row, q in enumerate(queries)}

i2i_als = als_i2i_lists(query_ids)
i2i_lists["B4 Implicit ALS (latent factors)"] = i2i_als
display(evaluate_and_record_i2i("B4 Implicit ALS (latent factors)", i2i_als).round(4))

# protocol 2, next to the rungs it is to be read against
unbiased_results = pd.DataFrame({
    "Content (metadata)":      evaluate_reranking(lambda u, c: user_profiles_n[u] @ W_user_n[c].T),
    "Quality (lower bound)":   evaluate_reranking(lambda u, c: lower_bound[c]),
    "Item-kNN (weighted sum)": evaluate_reranking(knn_scores_for),
    "Implicit ALS":            evaluate_reranking(lambda u, c: Y_als[c] @ X_als[u]),
    **({f"Implicit ALS, f={ALS_F_P2}, lambda={ALS_LAM_P2:g}, alpha={ALS_ALPHA_P2:g} (P2-shape choice)":
        evaluate_reranking(lambda u, c: Y_als_p2[c] @ X_als_p2[u])}
       if (ALS_F_P2, ALS_LAM_P2, ALS_ALPHA_P2) != (ALS_F, ALS_LAM, ALS_ALPHA) else {}),
}).T
unbiased_results.round(4)


,algorithm,Hit Rate@10,Precision@10,Recall@10,MRR@10,NDCG@10,Average popularity,Novelty (bits),Catalogue coverage,Unique items recommended
8,Implicit ALS,0.5199,0.0951,0.0615,0.2891,0.1206,1212.6448,10.4576,0.0322,671
6,Item-kNN (weighted sum),0.4453,0.0776,0.0489,0.2432,0.0986,995.2536,10.8584,0.0736,1533
0,Popularity,0.2998,0.0396,0.0264,0.1313,0.0488,2070.2538,9.5421,0.0026,54
2,Content (metadata),0.0703,0.0080,0.0049,0.0256,0.0092,120.6574,14.7265,0.5159,10747
5,Quality (lower bound),0.0322,0.0035,0.0018,0.0079,0.0033,167.3336,13.3468,0.0012,24
4,Quality (posterior mean),0.0218,0.0023,0.0012,0.0053,0.0021,113.9964,13.9806,0.0009,19
7,Item-kNN (normalized),0.0127,0.0013,0.0010,0.0041,0.0015,94.5352,14.9596,0.8555,17823
1,Random,0.0075,0.0008,0.0004,0.0020,0.0008,69.5521,15.6187,0.9930,20688
3,Quality (raw rate),0.0007,0.0001,0.0001,0.0002,0.0001,3.4118,18.7958,0.0159,331


,algorithm,Hit Rate@10,Precision@10,Recall@10,MRR@10,NDCG@10,Same-style share,Novelty (bits),Catalogue coverage
2,B2 Content (emotion+style),0.2960,0.0434,0.0059,0.1022,0.0431,1.0000,15.5456,0.5915
1,B1 Random within style,0.2240,0.0261,0.0035,0.0710,0.0262,1.0000,15.6036,0.6162
3,B3 Item-kNN (collaborative),0.0860,0.0095,0.0012,0.0288,0.0100,0.3554,13.2179,0.4255
4,B4 Implicit ALS (latent factors),0.0455,0.0050,0.0006,0.0140,0.0049,0.1708,15.0551,0.5859
0,B0 Random,0.0405,0.0041,0.0005,0.0127,0.0042,0.1498,15.6152,0.6145


,hit_rate,precision,recall,mrr,ndcg,average_popularity
Content (metadata),0.9666,0.3551,0.6263,0.6065,0.5187,72.2733
Quality (lower bound),0.9601,0.3581,0.6250,0.6446,0.5385,107.2878
Item-kNN (weighted sum),0.9656,0.3660,0.6417,0.7061,0.5717,99.3154
Implicit ALS,0.9726,0.3800,0.6741,0.7220,0.5984,94.7926


### 8.3 Rung 2b — a per-user ridge on the same metadata as rung 2

§2.3's per-user content ridge beat
implicit ALS at NDCG@5, while §5's content profile lost to it at NDCG@10. The two are different
estimators of the same metadata. The profile sums the feature vectors of a user's engagements
and ranks by cosine — positives only, so it can never learn that a user *avoids* a style. The
ridge fits per-user coefficients on the user's train impressions with engaged 0/1 as the target,
so every painting the user was shown and ignored is evidence too.

This cell runs that ridge on the delivered log as **rung 2b**: `W_user`'s 489 columns
(emotion, style and artist, the same features as rung 2), standardised, plus an intercept; λ
chosen on the validation split from a fixed grid of eleven values with §8.2's evaluator; users with
fewer impressions than columns solved in the dual. Nothing else changes — same `top_k`, same
tie-breaking, same protocols — so the whole gap to rung 2 is the estimator and the negatives,
and the comparison with rungs 6 and 6b is the one sweep 3 asked for.


In [54]:
# §8.3 — rung 2b: a per-user ridge on the same metadata as rung 2 =====================
# §2.4's sweep 3 says a content model that learns from the non-engagements too may beat the
# behavioural rungs. This is §2.3's fit_content on the delivered log: rung 2's W_user (489
# columns), standardised, plus an intercept; per-user least squares on the user's train
# impressions with engaged 0/1 as the target; lambda chosen on val exactly as §8.2 chose the
# ALS setting. Users with fewer impressions than columns — most of them — are solved in the
# dual (n_u x n_u instead of 490 x 490), which is what makes 12,000 fits take seconds.
W_std   = (W_user - W_user.mean(0)) / np.where(W_user.std(0) > 0, W_user.std(0), 1.0)
W_ridge = np.hstack([W_std, np.ones((n_items, 1), np.float32)]).astype(np.float32)
D_RIDGE = W_ridge.shape[1]
train_by_user = {u: (g.item_id.to_numpy(), g.engaged.to_numpy(np.float32))
                 for u, g in train.groupby("user_id")}

def fit_ridge_users(lam):
    B = np.zeros((n_users, D_RIDGE), np.float32)
    for u, (items, y) in train_by_user.items():
        Xu = W_ridge[items]
        if len(items) < D_RIDGE:                                  # dual form
            B[u] = Xu.T @ np.linalg.solve(Xu @ Xu.T + lam * np.eye(len(items), dtype=np.float32), y)
        else:                                                     # primal form
            B[u] = np.linalg.solve(Xu.T @ Xu + lam * np.eye(D_RIDGE, dtype=np.float32), Xu.T @ y)
    return B

RIDGE_GRID = [1.0, 3.0, 10.0, 30.0, 100.0, 300.0, 1_000.0, 3_000.0, 10_000.0, 30_000.0, 100_000.0]
ridge_grid, ridge_best, ridge_best_p2 = [], (-1.0, None), (-1.0, None)
for lam in RIDGE_GRID:
    B = fit_ridge_users(lam)
    s1 = val_ndcg(lambda blk, B=B: B[blk] @ W_ridge.T)             # §8.2's evaluator, P1 shape
    s2 = rerank_ndcg(unbiased_val, lambda u, c, B=B: W_ridge[c] @ B[u])   # P2 shape
    ridge_grid.append({"lambda": lam, "val NDCG@10, P1 shape": s1, "val NDCG@10, P2 shape": s2})
    if s1 > ridge_best[0]:
        ridge_best = (s1, (lam, B))
    if s2 > ridge_best_p2[0]:
        ridge_best_p2 = (s2, (lam, B))
RIDGE_LAM, B_ridge = ridge_best[1]                               # the rung: P1-shape choice
RIDGE_LAM_P2, B_ridge_p2 = ridge_best_p2[1]                       # the sensitivity check
print(f"users fitted     : {len(train_by_user):,}   (dual form for "
      f"{sum(len(v[0]) < D_RIDGE for v in train_by_user.values()):,} of them)")
print(f"chosen           : lambda={RIDGE_LAM} on the P1 shape (val NDCG@{K} {ridge_best[0]:.4f}; "
      f"ALS chose {als_best[0]:.4f} on the same targets)")
print(f"P2 shape would choose lambda={RIDGE_LAM_P2} (val NDCG@{K} {ridge_best_p2[0]:.4f})\n")
display(pd.DataFrame(ridge_grid).set_index("lambda").T.round(4))

def ridge_lists_for_all(users, k=K, chunk=500):
    lists = {}
    for start in range(0, len(users), chunk):
        blk    = users[start:start + chunk]
        scores = B_ridge[blk] @ W_ridge.T
        for row, u in enumerate(blk):
            lists[int(u)] = top_k(scores[row], seen_by_user[u], k, tie_seed=int(u))
    return lists

ridge_lists = ridge_lists_for_all(eval_users)
recommendation_lists["Content (ridge)"] = ridge_lists
display(evaluate_and_record("Content (ridge)", ridge_lists).round(4))

unbiased_results = pd.DataFrame({
    "Content (metadata)":      evaluate_reranking(lambda u, c: user_profiles_n[u] @ W_user_n[c].T),
    "Content (ridge)":         evaluate_reranking(lambda u, c: W_ridge[c] @ B_ridge[u]),
    **({f"Content (ridge), lambda={RIDGE_LAM_P2:g} (P2-shape choice)":
        evaluate_reranking(lambda u, c: W_ridge[c] @ B_ridge_p2[u])} if RIDGE_LAM_P2 != RIDGE_LAM else {}),
    "Item-kNN (weighted sum)": evaluate_reranking(knn_scores_for),
    "Implicit ALS":            evaluate_reranking(lambda u, c: Y_als[c] @ X_als[u]),
}).T
unbiased_results.round(4)


users fitted     : 12,000   (dual form for 11,765 of them)
chosen           : lambda=1000.0   (val NDCG@10 0.0076; ALS chose 0.1068 on the same targets)



lambda,1.0,3.0,10.0,30.0,100.0,300.0,1000.0,3000.0
val NDCG@10,0.0038,0.0044,0.0055,0.0065,0.0073,0.0075,0.0076,0.0066


,algorithm,Hit Rate@10,Precision@10,Recall@10,MRR@10,NDCG@10,Average popularity,Novelty (bits),Catalogue coverage,Unique items recommended
8,Implicit ALS,0.5199,0.0951,0.0615,0.2891,0.1206,1212.6448,10.4576,0.0322,671
6,Item-kNN (weighted sum),0.4453,0.0776,0.0489,0.2432,0.0986,995.2536,10.8584,0.0736,1533
0,Popularity,0.2998,0.0396,0.0264,0.1313,0.0488,2070.2538,9.5421,0.0026,54
2,Content (metadata),0.0703,0.0080,0.0049,0.0256,0.0092,120.6574,14.7265,0.5159,10747
9,Content (ridge),0.0621,0.0070,0.0034,0.0215,0.0076,101.6772,15.0887,0.2844,5925
5,Quality (lower bound),0.0322,0.0035,0.0018,0.0079,0.0033,167.3336,13.3468,0.0012,24
4,Quality (posterior mean),0.0218,0.0023,0.0012,0.0053,0.0021,113.9964,13.9806,0.0009,19
7,Item-kNN (normalized),0.0127,0.0013,0.0010,0.0041,0.0015,94.5352,14.9596,0.8555,17823
1,Random,0.0075,0.0008,0.0004,0.0020,0.0008,69.5521,15.6187,0.9930,20688
3,Quality (raw rate),0.0007,0.0001,0.0001,0.0002,0.0001,3.4118,18.7958,0.0159,331


,hit_rate,precision,recall,mrr,ndcg,average_popularity
Content (metadata),0.9666,0.3551,0.6263,0.6065,0.5187,72.2733
Content (ridge),0.9783,0.3988,0.7093,0.6621,0.5972,74.3206
Item-kNN (weighted sum),0.9656,0.3660,0.6417,0.7061,0.5717,99.3154
Implicit ALS,0.9726,0.3800,0.6741,0.7220,0.5984,94.7926


### 8.4 Rung 7 — rung 2b and rung 6b blended

§9.5 says the ridge and implicit ALS tie on protocol 2 (p = 0.58), and they fail differently:
the ridge cannot see §2.1's hidden factor, ALS cannot see a user with no engagements and
concentrates on 671 paintings. The argument for hybridisation applies exactly as it
did on Track B (§9.2). Each score is z-scored per user over the whole catalogue so the two live
on one scale, then blended convexly. α is chosen on `val` under protocol 1's shape, like every
other setting, and the protocol-2 shape is reported beside it; if the two
disagree, the P2-shape choice is evaluated too, so the cost of the committed tuning protocol is
visible rather than assumed.


In [ ]:
# §8.4 — rung 7: rung 2b and rung 6b blended ============================================
# The two tie on protocol 2 and fail differently: the ridge cannot see the hidden factor, ALS
# cannot see a user with no engagements. Each score is z-scored per user over the whole
# catalogue so the two share a scale, then blended convexly; alpha is chosen on `val` under
# protocol 1's shape, like every other setting, and the protocol-2 shape is reported beside it.
def zscore_rows(M):
    return (M - M.mean(1, keepdims=True)) / (M.std(1, keepdims=True) + 1e-9)

def hybrid_block(users, alpha):
    """Blended full-catalogue scores for a block of users."""
    return (alpha * zscore_rows(B_ridge[users] @ W_ridge.T)
            + (1 - alpha) * zscore_rows(X_als[users] @ Y_als.T))

def hybrid_score_fn(alpha):
    return lambda u, c: hybrid_block(np.array([u]), alpha)[0, c]

def holdout_blocks(frame, chunk=500):
    """A uniform holdout as (relevance, z_ridge, z_ALS) matrices of users x candidates, the two
    z-scored score blocks gathered once, so that an alpha sweep costs one weighted sum per
    alpha instead of a per-user pass."""
    f = frame.sort_values("user_id", kind="stable")
    n = f.groupby("user_id").size().iloc[0]
    U = f.user_id.to_numpy().reshape(-1, n)[:, 0]
    I = f.item_id.to_numpy().reshape(-1, n)
    R = f.engaged.to_numpy().astype(float).reshape(-1, n)
    Zr, Za = np.empty_like(R), np.empty_like(R)
    for s in range(0, len(U), chunk):
        blk = U[s:s + chunk]
        Zr[s:s + chunk] = np.take_along_axis(zscore_rows(B_ridge[blk] @ W_ridge.T), I[s:s + chunk], axis=1)
        Za[s:s + chunk] = np.take_along_axis(zscore_rows(X_als[blk] @ Y_als.T),   I[s:s + chunk], axis=1)
    return R, Zr, Za

def ndcg_blocks(R, S, k=K):
    """rerank_ndcg on matrices: same tie rule (argsort), same truncated ideal, users with no
    positive dropped."""
    keep  = R.sum(1) > 0
    disc  = np.log2(np.arange(2, min(k, R.shape[1]) + 2))
    hits  = np.take_along_axis(R, np.argsort(-S, axis=1), axis=1)[:, :k]
    ideal = np.sort(R, axis=1)[:, ::-1][:, :k]
    return float(((hits / disc).sum(1)[keep] / (ideal / disc).sum(1)[keep]).mean())

R_v, Zr_v, Za_v = holdout_blocks(unbiased_val)
assert abs(ndcg_blocks(R_v, Zr_v) - rerank_ndcg(unbiased_val, hybrid_score_fn(1.0))) < 1e-9, \
       "the vectorised holdout scorer disagrees with the per-user one"

HYBRID_GRID = np.round(np.arange(0.0, 1.0001, 0.1), 1)
hyb_grid = pd.DataFrame([{"alpha (ridge weight)": a,
                          "val NDCG@10, P1 shape": val_ndcg(lambda blk, a=a: hybrid_block(blk, a)),
                          "val NDCG@10, P2 shape": ndcg_blocks(R_v, a * Zr_v + (1 - a) * Za_v)}
                         for a in HYBRID_GRID]).set_index("alpha (ridge weight)")
HYB_ALPHA    = float(hyb_grid["val NDCG@10, P1 shape"].idxmax())
HYB_ALPHA_P2 = float(hyb_grid["val NDCG@10, P2 shape"].idxmax())
print(f"alpha chosen on the P1 shape: {HYB_ALPHA:.1f}   (P2 shape would choose {HYB_ALPHA_P2:.1f})\n")
display(hyb_grid.T.round(4))

def hybrid_lists_for_all(users, alpha, k=K, chunk=500):
    lists = {}
    for start in range(0, len(users), chunk):
        blk = users[start:start + chunk]
        scores = hybrid_block(blk, alpha)
        for row, u in enumerate(blk):
            lists[int(u)] = top_k(scores[row], seen_by_user[u], k, tie_seed=int(u))
    return lists

hybrid_lists = hybrid_lists_for_all(eval_users, HYB_ALPHA)
recommendation_lists["Hybrid (ridge+ALS)"] = hybrid_lists
display(evaluate_and_record("Hybrid (ridge+ALS)", hybrid_lists).round(4))

unbiased_results = pd.DataFrame({
    "Content (ridge)":      evaluate_reranking(lambda u, c: W_ridge[c] @ B_ridge[u]),
    "Implicit ALS":         evaluate_reranking(lambda u, c: Y_als[c] @ X_als[u]),
    "Hybrid (ridge+ALS)":   evaluate_reranking(hybrid_score_fn(HYB_ALPHA)),
    **({f"Hybrid, alpha={HYB_ALPHA_P2:.1f} (P2-shape choice)": evaluate_reranking(hybrid_score_fn(HYB_ALPHA_P2))}
       if HYB_ALPHA_P2 != HYB_ALPHA else {}),
}).T
unbiased_results.round(4)


### 8.5 Rung 6c — BPR, the pairwise estimator of the same signal

Implicit ALS fits the factor model by weighted least squares on every cell of the matrix; BPR fits the same model to a ranking objective — for each
engagement and a sampled non-engagement, push the engaged item above it. Same signal, same
16 factors (both tuning shapes chose them in §8.2), third estimator. It is here as the due
diligence §12 asked for rather than as an expected gain, and it is tuned like everything else:
a small grid on `val`, both shapes reported, the P1-shape choice kept.


In [ ]:
# §8.5 — rung 6c: BPR, the pairwise estimator of the same signal ========================
# Bayesian Personalised Ranking: the same factor
# model as §8.2, fitted to a different objective — for every engagement (u, i) and a sampled
# item j the user did not engage with, push x_ui above x_uj through a logistic loss. Mini-batch
# SGD in numpy: one epoch is one pass over the shuffled positives, each with a fresh uniform
# negative. f = 16 is fixed, since both tuning shapes chose it for ALS; the learning rate and
# the regularisation come from a small grid on `val`, under both shapes, like everything else.
SEED_BPR = 6

def fit_bpr(train, n_items, n_users, f=16, lr=0.05, reg=0.01, epochs=20, batch=4096, seed=SEED_BPR):
    r   = np.random.default_rng(seed)
    pos = train[train.engaged == 1]
    pu, pi = pos.user_id.to_numpy(), pos.item_id.to_numpy()
    X = r.normal(0, 0.05, (n_users, f)).astype(np.float32)
    Y = r.normal(0, 0.05, (n_items, f)).astype(np.float32)
    for _ in range(epochs):
        order = r.permutation(len(pu))
        for s in range(0, len(order), batch):
            b  = order[s:s + batch]
            u, i = pu[b], pi[b]
            j  = r.integers(0, n_items, len(b))                 # uniform negative sampling
            xu, yi, yj = X[u], Y[i], Y[j]
            g  = (1.0 / (1.0 + np.exp((xu * (yi - yj)).sum(1))))[:, None]   # sigmoid(-x_uij)
            dX = g * (yi - yj) - reg * xu
            dI = g * xu - reg * yi
            dJ = -g * xu - reg * yj
            np.add.at(X, u, lr * dX)                            # duplicates within a batch add
            np.add.at(Y, i, lr * dI)
            np.add.at(Y, j, lr * dJ)
    return X, Y

BPR_GRID = [(lr, reg) for lr in (0.02, 0.05) for reg in (0.01, 0.1)]
bpr_grid, bpr_best, bpr_best_p2 = [], (-1.0, None), (-1.0, None)
for lr, reg in BPR_GRID:
    X, Y = fit_bpr(train, n_items, n_users, lr=lr, reg=reg)
    s1 = val_ndcg(lambda blk, X=X, Y=Y: X[blk] @ Y.T)
    s2 = rerank_ndcg(unbiased_val, lambda u, c, X=X, Y=Y: Y[c] @ X[u])
    bpr_grid.append({"lr": lr, "reg": reg, "val NDCG@10, P1 shape": s1, "val NDCG@10, P2 shape": s2})
    if s1 > bpr_best[0]:
        bpr_best = (s1, (lr, reg, X, Y))
    if s2 > bpr_best_p2[0]:
        bpr_best_p2 = (s2, (lr, reg, X, Y))
BPR_LR, BPR_REG, X_bpr, Y_bpr = bpr_best[1]
BPR_LR_P2, BPR_REG_P2, X_bpr_p2, Y_bpr_p2 = bpr_best_p2[1]
print(f"grid : {len(BPR_GRID)} fits, f=16, 20 epochs each, train only")
print(f"chosen : lr={BPR_LR}, reg={BPR_REG} on the P1 shape (val NDCG@{K} {bpr_best[0]:.4f}; "
      f"ALS chose {als_best[0]:.4f})")
print(f"P2 shape would choose lr={BPR_LR_P2}, reg={BPR_REG_P2} (val NDCG@{K} {bpr_best_p2[0]:.4f}; "
      f"ALS's P2-shape best {als_best_p2[0]:.4f})\n")
display(pd.DataFrame(bpr_grid).set_index(["lr", "reg"]).round(4))

def factor_lists_for_all(Xf, Yf, users, k=K, chunk=500):
    lists = {}
    for start in range(0, len(users), chunk):
        blk    = users[start:start + chunk]
        scores = Xf[blk] @ Yf.T
        for row, u in enumerate(blk):
            lists[int(u)] = top_k(scores[row], seen_by_user[u], k, tie_seed=int(u))
    return lists

bpr_lists = factor_lists_for_all(X_bpr, Y_bpr, eval_users)
recommendation_lists["BPR"] = bpr_lists
display(evaluate_and_record("BPR", bpr_lists).round(4))

unbiased_results = pd.DataFrame({
    "Item-kNN (weighted sum)": evaluate_reranking(knn_scores_for),
    "Implicit ALS":            evaluate_reranking(lambda u, c: Y_als[c] @ X_als[u]),
    "BPR":                     evaluate_reranking(lambda u, c: Y_bpr[c] @ X_bpr[u]),
    **({f"BPR, lr={BPR_LR_P2}, reg={BPR_REG_P2} (P2-shape choice)":
        evaluate_reranking(lambda u, c: Y_bpr_p2[c] @ X_bpr_p2[u])}
       if (BPR_LR_P2, BPR_REG_P2) != (BPR_LR, BPR_REG) else {}),
}).T
unbiased_results.round(4)


### 8.6 Coverage for the hybrid

Rung 7 is the most accurate method on Track A and one of the most concentrated: 834 distinct
paintings across 10,448 top-10 lists, an average recommended popularity of 1,174 impressions
against a catalogue mean of 72. This cell does the simplest thing that acts on every list at once: subtract
γ × standardised log-popularity from the hybrid's score, sweep γ, and report the frontier —
test accuracy on both protocols against unique items, coverage, popularity and novelty.

γ is *not* chosen for accuracy, which would pick zero. The rule, fixed before the sweep: the
largest γ whose protocol-2-shape validation NDCG stays within 0.005 of the undiscounted
hybrid's. The row it selects is registered next to the other rungs so §9.5 can put an interval
on what the discount costs.


In [ ]:
# §8.6 — coverage for the hybrid: a popularity discount, and the frontier it buys ==========
# Rung 7 wins on accuracy and takes ALS's concentration: 834 distinct paintings across
# 10,448 top-10 lists. The simplest lever that acts on every list at once is a discount on
# log-popularity, applied to the hybrid's z-score: s'(u, i) = s(u, i) - gamma * logpop_z[i],
# where logpop_z is the training log's impression count, logged and standardised. gamma is
# NOT chosen for accuracy — that would just pick 0 — but by a rule fixed before the sweep:
# the largest gamma whose P2-shape validation NDCG stays within 0.005 of the undiscounted
# hybrid's. Everything else about the lists is unchanged, so the whole table is the price
# of coverage in accuracy, read off the same instruments as every other rung.
logpop_z = np.log1p(popularity)
logpop_z = ((logpop_z - logpop_z.mean()) / logpop_z.std()).astype(np.float32)
COVERAGE_TOL = 0.005
GAMMA_GRID   = [0.0, 0.1, 0.2, 0.3, 0.5, 0.75, 1.0, 1.5, 2.0]

def discounted_block(users, gamma):
    return hybrid_block(users, HYB_ALPHA) - gamma * logpop_z[None, :]

# validation side: the P2-shape holdout, vectorised as in §8.4
Zh_v = HYB_ALPHA * Zr_v + (1 - HYB_ALPHA) * Za_v
_f   = unbiased_val.sort_values("user_id", kind="stable")
I_v  = _f.item_id.to_numpy().reshape(R_v.shape)
val_curve = {g: ndcg_blocks(R_v, Zh_v - g * logpop_z[I_v]) for g in GAMMA_GRID}
GAMMA = max(g for g in GAMMA_GRID if val_curve[g] >= val_curve[0.0] - COVERAGE_TOL)
print(f"gamma chosen by the pre-committed rule: {GAMMA}   "
      f"(largest with P2-shape val NDCG >= {val_curve[0.0]:.4f} - {COVERAGE_TOL})\n")

rows = []
for g in GAMMA_GRID:
    lists = {}
    for start in range(0, len(eval_users), 500):
        blk = eval_users[start:start + 500]
        sc  = discounted_block(blk, g)
        for row, u in enumerate(blk):
            lists[int(u)] = top_k(sc[row], seen_by_user[u], K, tie_seed=int(u))
    m   = pd.DataFrame([evaluate_user(r, test_positives[u]) for u, r in lists.items()]).mean()
    cov, uniq = catalogue_coverage(lists)
    p2  = evaluate_reranking(lambda u, c, g=g: discounted_block(np.array([u]), g)[0, c])
    rows.append({"gamma": g, "val NDCG@10 (P2 shape)": val_curve[g],
                 "test P2 NDCG@10": p2.ndcg, "test P1 NDCG@10": m.ndcg,
                 "unique items": uniq, "coverage": cov,
                 "avg popularity": m.average_popularity, "novelty (bits)": m.novelty_bits})
    if g == GAMMA:
        recommendation_lists[f"Hybrid, popularity discount {GAMMA:g}"] = lists
        evaluate_and_record(f"Hybrid, popularity discount {GAMMA:g}", lists)
coverage_frontier = pd.DataFrame(rows).set_index("gamma")
display(coverage_frontier.round(4))


## 9. CLIP visual similarity (I1)

Every method so far has ranked paintings from *labels about* them — style, emotion proportions,
who else engaged with them. This is the one that looks at the painting. It is the natural end of the
item-to-item ladder and the one method whose inputs no other method here shares:

| | B1 | B2 | B3 | **I1** |
|---|---|---|---|---|
| sees | style | style + emotion | co-engagement behaviour | **pixels** |

### Why it slots in without changing anything else

- **The images are already prepared for it.** §1.2 resized all 81,444 files with the *short*
  side at 256 and no cropping, chosen so that CLIP's own `Resize(224) → CenterCrop(224)` never
  has to upscale and never sees blur the model was not trained on. That decision was made for
  this section.
- **The protocol is untouched.** Same 2,000 queries (`SEED_I2I`), same same-artist relevance,
  same `top_k` with the same seeded tie-breaking, same `evaluate_and_record_i2i`. The
  comparison against B0/B1/B2/B3 is like-for-like by construction rather than by argument.
- **Artist stays hidden.** CLIP receives pixels only. This is the constraint that makes the
  research question non-circular: *can visual features recover authorship that categorical
  metadata cannot see?* A method that could see the artist label would score near 1.0 by
  definition and the comparison would be unfalsifiable.
- **The corpus was not built with it.** §1.5 deduplicates by perceptual hash and title, never by
  embedding distance — precisely so that this evaluation is not handed a corpus from which its
  own hardest cases were removed.

### Checkpoint pinning

"CLIP ViT-B/32" is ambiguous, and this project measured how much that costs. OpenAI's original
CLIP uses **QuickGELU** activations; in `open_clip` 3.x the obvious-looking tag `ViT-B-32` +
`pretrained="openai"` builds a *non*-QuickGELU model and loads the QuickGELU-trained weights
into it. It warns, and then works. On a 64-image sample from this
corpus the two tags produce embeddings at **cosine 0.957** of one another, yet their nearest
neighbours agree only **60.9%** of the time.

The pin is therefore `ViT-B-32-quickgelu` / `openai`, and the cell prints a SHA-256 of the
resulting embedding file so a later run can be shown to be the same run.

### What would count as a result

The expected outcome is that it wins Precision@10 and NDCG@10 *and* loses
badly on catalogue coverage and novelty, because visual nearest neighbours of a painting are
disproportionately other works in the same series — §10.D. Three numbers should be read
together:

| read | against | tells you |
|---|---|---|
| Precision@10, NDCG@10 | B2 (metadata), B3 (behaviour) | does the visual signal recover authorship? |
| same-style share | B1's 1.000 | is it finding the artist, or just the style? |
| coverage@10, novelty | B2's, B3's | is it a browsing module or a series-duplicator? |

§9.3 replaces coverage@10 and
novelty
 with intra-list distance and distinct artists per list, and §10.D records why.

The embeddings are cached to `clip_vitb32.npy`, so it is a
one-time cost.

In [ ]:
# I1 — frozen CLIP image embeddings ==================
# Run once; cached to clip_vitb32.npy in item_id order. Inference only, no training.
# OpenAI's original CLIP uses QuickGELU
# activations. In open_clip 3.x, ("ViT-B-32", "openai") builds a model with quick_gelu=False
# and then loads QuickGELU-trained weights into it — open_clip warns, and the result is a
# quietly degraded encoder. Measured on a 64-image sample from this corpus: embeddings from
# the two tags sit at cosine 0.957 of each other, but their nearest neighbours agree only
# 60.9% of the time. For a retrieval evaluation that is not a rounding difference, it is a
# different experiment. "-quickgelu" is the faithful reproduction of the original release.
CLIP_TAG   = ("ViT-B-32-quickgelu", "openai")
CLIP_CACHE = Path("clip_vitb32.npy")
CLIP_BATCH = 64

def build_clip_embeddings(files, out=CLIP_CACHE, batch=CLIP_BATCH):
    """Encode every corpus painting with a frozen CLIP image tower, L2-normalised so that a
    dot product is the cosine. `preprocess` is CLIP's own transform, which is why §1.2 stored
    the images with the short side at 256 rather than the long side."""
    import torch, open_clip
    device = ("cuda" if torch.cuda.is_available()
              else "mps" if torch.backends.mps.is_available() else "cpu")
    model, _, preprocess = open_clip.create_model_and_transforms(
        CLIP_TAG[0], pretrained=CLIP_TAG[1])
    model = model.to(device).eval()
    print(f"encoding {len(files):,} images on {device} with {CLIP_TAG[0]}/{CLIP_TAG[1]}")

    chunks = []
    with torch.no_grad():
        for start in range(0, len(files), batch):
            px = torch.stack([preprocess(Image.open(IMG_DIR / f).convert("RGB"))
                              for f in files[start:start + batch]]).to(device)
            v = model.encode_image(px).float()
            chunks.append((v / v.norm(dim=-1, keepdim=True)).cpu().numpy())
            if (start // batch) % 50 == 0:
                print(f"  {start:,} / {len(files):,}", flush=True)
    E = np.vstack(chunks).astype(np.float32)
    np.save(out, E)
    return E


# content_meta is the §1.5 corpus in item_id order, so row i of E_clip IS item i
clip_files = content_meta.file.to_numpy()
try:
    import torch, open_clip                                      # noqa: F401
    E_clip = np.load(CLIP_CACHE) if CLIP_CACHE.exists() else build_clip_embeddings(clip_files)
    assert len(E_clip) == n_items, "embeddings are not aligned to the frozen corpus"
    print(f"CLIP embeddings : {E_clip.shape}  ({CLIP_TAG[0]} / {CLIP_TAG[1]})")
    print(f"  file sha256   : {hashlib.sha256(CLIP_CACHE.read_bytes()).hexdigest()[:16]}...")
    print(f"  norm check    : {np.linalg.norm(E_clip, axis=1).mean():.4f} (expect 1.0)")
except ImportError:
    E_clip = None
    print("I1 NOT RUN — torch / open_clip_torch are not installed in this environment.")
    print("    pip install torch open_clip_torch")
    print("    then re-run this cell and the next one; nothing else changes.")

/Users/diazm/Documents/HSLU/07_SS2026/Recommender/Code/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CLIP embeddings : (20833, 512)  (ViT-B-32-quickgelu / openai)
  file sha256   : 26ad6637b77349f8...
  norm check    : 1.0000 (expect 1.0)


In [56]:
# I1 — CLIP cosine on the item-to-item track =========
# Same queries, same relevance, same tie-breaking as B0-B3, so the row lands in the same table.
if E_clip is None:
    print("skipped — no embeddings; see the cell above.")
else:
    def clip_i2i_lists(queries, k=K):
        scores = (E_clip[queries] @ E_clip.T).astype(np.float32)   # 2,000 x 20,833 cosines
        scores[np.arange(len(queries)), queries] = -np.inf         # never return the query
        ties = (scores == np.partition(scores, -k, axis=1)[:, -k][:, None]).sum(1)
        print(f"I1 candidates tied at rank {k}: mean {ties.mean():,.1f} "
              f"(continuous scores — expect ~1, unlike B2)")
        return {int(q): top_k(scores[row], np.array([q]), k, tie_seed=int(q))
                for row, q in enumerate(queries)}

    i2i_clip = clip_i2i_lists(query_ids)
    i2i_lists["I1 CLIP ViT-B/32"] = i2i_clip

    # §10.D — the series-collapse check. Precision alone cannot distinguish "found the artist"
    # from "returned ten scans of the same series", so report the pair together.
    dup_share = np.mean([
        (item_artist[r] == item_artist[q]).mean() for q, r in i2i_clip.items()])
    print(f"same-artist share of returned lists: {dup_share:.3f}  "
          f"— read against coverage and novelty below, not on its own")

    display(evaluate_and_record_i2i("I1 CLIP ViT-B/32", i2i_clip).round(4))

I1 candidates tied at rank 10: mean 1.0 (continuous scores — expect ~1, unlike B2)


same-artist share of returned lists: 0.298  — read against coverage and novelty below, not on its own


,algorithm,Hit Rate@10,Precision@10,Recall@10,MRR@10,NDCG@10,Same-style share,Novelty (bits),Catalogue coverage
5,I1 CLIP ViT-B/32,0.7525,0.2984,0.0486,0.5546,0.3306,0.6298,15.6455,0.5152
2,B2 Content (emotion+style),0.2960,0.0434,0.0059,0.1022,0.0431,1.0000,15.5456,0.5915
1,B1 Random within style,0.2240,0.0261,0.0035,0.0710,0.0262,1.0000,15.6036,0.6162
3,B3 Item-kNN (collaborative),0.0860,0.0095,0.0012,0.0288,0.0100,0.3554,13.2179,0.4255
4,B4 Implicit ALS (latent factors),0.0455,0.0050,0.0006,0.0140,0.0049,0.1708,15.0551,0.5859
0,B0 Random,0.0405,0.0041,0.0005,0.0127,0.0042,0.1498,15.6152,0.6145


### 9.1 CLIP on Track A — pixels as a *user* profile

§9 asks whether pixels beat metadata at finding another work by the same artist. This asks a
different question with the same embeddings and no extra encoding pass:

> **Do pixels help *personalisation*, or only item-to-item similarity?**

The construction is §5's content profile with one substitution. A user's profile is the mean of
the **CLIP embeddings** of the paintings they engaged with in the training split, and candidates
are ranked by cosine against it — same `top_k`, same tie-breaking, same protocols, same
`evaluate_and_record`. Only the item representation changes, from 489 metadata dimensions to
512 visual ones.

That makes it a clean comparison on ladder A: everything else is held fixed, so the difference
is attributable to the representation and nothing else. It is not a *rung*, because it lands below the rung it was meant to climb from.

Metadata content reaches NDCG@10 = 0.5187 on the
unbiased holdout. §2.1's utility is built from emotion, style, artist and a hidden latent
factor — **none of which is a pixel statistic.** CLIP can only reach the simulated user's taste
insofar as visual appearance correlates with style and emotion labels, which it does, but
indirectly. So the honest prediction is that CLIP profiles land *below* metadata on Track A
while doing well on Track B, and if that is what happens it is a statement about the simulator
rather than about CLIP. Track B is where a CLIP result carries real evidence, because there the
labels are real.

### 9.2 The hybrid

Content and collaborative signals fail differently, which is the standard argument for
combining them. Here the two content representations also fail differently:
metadata knows the artist and the crowd's emotional reading; CLIP knows composition, palette
and texture. A convex blend of the two cosine scores,

$$s_\alpha(u, i) \;=\; \alpha \cdot \cos(\mathbf{u}_{\text{meta}}, \mathbf{w}_i)
\;+\; (1-\alpha)\cdot \cos(\mathbf{u}_{\text{clip}}, \mathbf{e}_i),$$

costs one line and sweeps in one loop. Both terms are cosines on L2-normalised vectors, so they
share a scale and the blend needs no calibration. $\alpha = 1$ recovers §5 exactly and
$\alpha = 0$ recovers §9.1, so the sweep contains both endpoints as a correctness check.

The same blend runs on Track B between `W_i2i` (17 metadata dimensions) and CLIP.

In [57]:
# §9.1 — CLIP as a Track A user profile (ladder A — the failed rung, §5.0) ====
if E_clip is None:
    print("skipped — no embeddings; see §9 above.")
else:
    E_clip_n = l2_normalize(E_clip)          # CLIP already returns unit vectors; explicit anyway

    # Profile = sum of the CLIP embeddings of the items the user engaged with in train, then
    # L2-normalised — identical construction to §5's metadata profile (cell above), so the
    # only thing that differs between the two rungs is the item representation.
    clip_profiles = np.zeros((n_users, E_clip_n.shape[1]), np.float32)
    edges = np.searchsorted(u_sorted, np.arange(0, n_users + 1, CHUNK))
    for lo, hi in zip(edges[:-1], edges[1:]):
        if lo == hi:
            continue
        block = u_sorted[lo:hi]
        first = block[0]
        P = np.zeros((block[-1] - first + 1, n_items), np.float32)
        P[block - first, i_sorted[lo:hi]] = 1.0
        clip_profiles[first:block[-1] + 1] = P @ E_clip_n
    clip_profiles_n = l2_normalize(clip_profiles)

    def clip_top_k_for_all(users, k=K, chunk=500):
        """§5's content_top_k_for_all with W_user_n swapped for the CLIP embeddings."""
        lists = {}
        for start in range(0, len(users), chunk):
            blk = users[start:start + chunk]
            scores = clip_profiles_n[blk] @ E_clip_n.T
            for row, user_id in enumerate(blk):
                lists[int(user_id)] = top_k(scores[row], seen_by_user[user_id], k,
                                            tie_seed=int(user_id))
        return lists

    print(f"CLIP user profiles: {clip_profiles_n.shape}, "
          f"{(np.abs(clip_profiles_n).sum(1) > 0).sum():,} non-empty")
    clip_user_lists = clip_top_k_for_all(eval_users)
    recommendation_lists["Content (CLIP pixels)"] = clip_user_lists
    display(evaluate_and_record("Content (CLIP pixels)", clip_user_lists).round(4))

CLIP user profiles: (12000, 512), 11,722 non-empty


,algorithm,Hit Rate@10,Precision@10,Recall@10,MRR@10,NDCG@10,Average popularity,Novelty (bits),Catalogue coverage,Unique items recommended
8,Implicit ALS,0.5199,0.0951,0.0615,0.2891,0.1206,1212.6448,10.4576,0.0322,671
6,Item-kNN (weighted sum),0.4453,0.0776,0.0489,0.2432,0.0986,995.2536,10.8584,0.0736,1533
0,Popularity,0.2998,0.0396,0.0264,0.1313,0.0488,2070.2538,9.5421,0.0026,54
2,Content (metadata),0.0703,0.0080,0.0049,0.0256,0.0092,120.6574,14.7265,0.5159,10747
9,Content (ridge),0.0621,0.0070,0.0034,0.0215,0.0076,101.6772,15.0887,0.2844,5925
5,Quality (lower bound),0.0322,0.0035,0.0018,0.0079,0.0033,167.3336,13.3468,0.0012,24
4,Quality (posterior mean),0.0218,0.0023,0.0012,0.0053,0.0021,113.9964,13.9806,0.0009,19
7,Item-kNN (normalized),0.0127,0.0013,0.0010,0.0041,0.0015,94.5352,14.9596,0.8555,17823
1,Random,0.0075,0.0008,0.0004,0.0020,0.0008,69.5521,15.6187,0.9930,20688
10,Content (CLIP pixels),0.0072,0.0007,0.0005,0.0021,0.0007,62.4589,15.8232,0.0648,1349


In [58]:
# §9.1 — and on the unbiased holdout, next to every other Track A rung ==
if E_clip is None:
    print("skipped — no embeddings; see §9 above.")
else:
    unbiased_results = pd.DataFrame({
        "Random":     evaluate_reranking(
                          lambda u, c: np.random.default_rng(SEED_EVAL + u).random(len(c))),
        "Popularity": evaluate_reranking(lambda u, c: popularity[c]),
        "Content (CLIP pixels)":  evaluate_reranking(
                          lambda u, c: clip_profiles_n[u] @ E_clip_n[c].T),
        "Content (metadata)": evaluate_reranking(
                          lambda u, c: user_profiles_n[u] @ W_user_n[c].T),
        "Quality (lower bound)": evaluate_reranking(lambda u, c: lower_bound[c]),
        "Item-kNN (weighted sum)": evaluate_reranking(knn_scores_for),
    }).T
    print(f"oracle ceiling on this protocol: {ORACLE_NDCG_P2:.4f}\n")
    display(unbiased_results.round(4))

oracle ceiling on this protocol: 0.8919



,hit_rate,precision,recall,mrr,ndcg,average_popularity
Random,0.9303,0.2895,0.4992,0.4932,0.4015,69.4848
Popularity,0.9445,0.3180,0.5530,0.5545,0.4552,125.6273
Content (CLIP pixels),0.9326,0.2978,0.5137,0.5078,0.4159,68.9857
Content (metadata),0.9666,0.3551,0.6263,0.6065,0.5187,72.2733
Quality (lower bound),0.9601,0.3581,0.6250,0.6446,0.5385,107.2878
Item-kNN (weighted sum),0.9656,0.3660,0.6417,0.7061,0.5717,99.3154


In [59]:
# §9.2 — hybrid: metadata (+) pixels, on both tracks ==================
# alpha = 1 recovers §5 exactly and alpha = 0 recovers §9.1, so the endpoints are a
# correctness check on the blend rather than results in their own right.
if E_clip is None:
    print("skipped — no embeddings; see §9 above.")
else:
    ALPHAS = [0.0, 0.25, 0.50, 0.75, 1.0]

    rows = []
    for a in ALPHAS:                                   # Track A, protocol 2
        r = evaluate_reranking(
            lambda u, c, a=a: a * (user_profiles_n[u] @ W_user_n[c].T)
                            + (1 - a) * (clip_profiles_n[u] @ E_clip_n[c].T))
        rows.append({"alpha (metadata weight)": a, **r.to_dict()})
    print("Track A — protocol 2, blended content scores")
    display(pd.DataFrame(rows).set_index("alpha (metadata weight)").round(4))

    rows = []
    for a in ALPHAS:                                   # Track B, protocol 3
        s = (a * (W_i2i_n[query_ids] @ W_i2i_n.T)
             + (1 - a) * (E_clip_n[query_ids] @ E_clip_n.T)).astype(np.float32)
        s[np.arange(len(query_ids)), query_ids] = -np.inf
        lists = {int(q): top_k(s[row], np.array([q]), K, tie_seed=int(q))
                 for row, q in enumerate(query_ids)}
        m = pd.DataFrame([evaluate_query(r, q) for q, r in lists.items()]).mean()
        rows.append({"alpha (metadata weight)": a, f"NDCG@{K}": m.ndcg,
                     f"Precision@{K}": m.precision, f"Hit Rate@{K}": m.hit_rate,
                     "Same-style share": m.same_style_share,
                     "Novelty (bits)": m.novelty_bits,
                     "Catalogue coverage": len({int(i) for r in lists.values() for i in r}) / n_items})
        if a == 0.5:
            i2i_lists["H1 Hybrid (metadata+CLIP)"] = lists
            evaluate_and_record_i2i("H1 Hybrid (metadata+CLIP)", lists)
    print("\nTrack B — protocol 3, blended item-to-item scores")
    display(pd.DataFrame(rows).set_index("alpha (metadata weight)").round(4))

Track A — protocol 2, blended content scores


,hit_rate,precision,recall,mrr,ndcg,average_popularity
alpha (metadata weight),,,,,,
0.00,0.9326,0.2978,0.5137,0.5078,0.4159,68.9857
0.25,0.9457,0.3190,0.5540,0.5726,0.4633,69.7364
0.50,0.9553,0.3376,0.5895,0.5946,0.4937,70.6034
0.75,0.9611,0.3477,0.6112,0.6033,0.5088,71.1838
1.00,0.9666,0.3551,0.6263,0.6065,0.5187,72.2733



Track B — protocol 3, blended item-to-item scores


,NDCG@10,Precision@10,Hit Rate@10,Same-style share,Novelty (bits),Catalogue coverage
alpha (metadata weight),,,,,,
0.00,0.3306,0.2984,0.7525,0.6298,15.6455,0.5152
0.25,0.4086,0.3762,0.8255,1.0000,15.6263,0.5234
0.50,0.4000,0.3698,0.8205,1.0000,15.6203,0.5198
0.75,0.3626,0.3341,0.7890,1.0000,15.6087,0.5257
1.00,0.0431,0.0433,0.2960,1.0000,15.5456,0.5915


### 9.3 Testing the failure mode 

Was predicted that a strong visual model would win Precision@10 by returning ten
near-identical works from one series, and it named **coverage@10 and novelty** as the
diagnostics. Running I1 exposed a flaw in that plan: those two metrics *cannot detect the thing
they were chosen to detect.* Both are computed across the 2,000 queries, and series collapse is
a property *within* a single list. A method can return ten near-duplicates for every query and
still cover half the catalogue, because different queries collapse onto different series.

So the pre-registered check needs a pre-registered metric it does not have. This cell adds two
within-list measures on the same queries:

- **intra-list distance** — mean pairwise $1 - \cos$ among the ten returned items in CLIP
  space. Low means the list is visually monotonous.
- **distinct artists per list** — how many different artists appear in a top-10. This one is
  independent of any embedding, which matters because intra-list distance measured in CLIP
  space judges I1 by its own geometry and is biased against it.

In [60]:
# §9.3 — the §10.D series-collapse test, done with the right metric ===
# §10.D predicted that a strong visual model would return ten near-identical works from one
# series, and named coverage@10 and novelty as the diagnostics. **Those cannot detect it.**
# Both are CROSS-QUERY metrics: a method can return ten near-duplicates for every single query
# and still cover half the catalogue, because different queries collapse onto different series.
# Series collapse is a WITHIN-LIST property and needs a within-list metric.
if E_clip is None:
    print("skipped — no embeddings; see §9 above.")
else:
    def intra_list_distance(ranked, k=K):
        """Mean pairwise (1 - cosine) among the returned items, in CLIP space."""
        V = E_clip_n[ranked[:k]]
        S = V @ V.T
        iu = np.triu_indices(len(V), k=1)
        return float((1.0 - S[iu]).mean())

    def distinct_artists(ranked, k=K):
        return len(np.unique(item_artist[ranked[:k]]))

    rows = []
    for name, lists in i2i_lists.items():
        ild = np.mean([intra_list_distance(r) for r in lists.values()])
        art = np.mean([distinct_artists(r) for r in lists.values()])
        rows.append({"method": name, "intra-list distance": ild,
                     "distinct artists / 10": art,
                     f"Precision@{K}": np.mean([evaluate_query(r, q)["precision"]
                                                for q, r in lists.items()])})
    ild_table = pd.DataFrame(rows).set_index("method").sort_values("intra-list distance")
    print("Within-list diversity on the same 2,000 queries "
          "(higher intra-list distance = less collapsed)\n")
    display(ild_table.round(4))
    print("\nCaveat: intra-list distance is measured in CLIP space, which is the space I1 ranks")
    print("in — so I1 is judged by its own metric and is expected to look least diverse. The")
    print("distinct-artist column is representation-independent and is the honest cross-check.")

Within-list diversity on the same 2,000 queries (higher intra-list distance = less collapsed)



,intra-list distance,distinct artists / 10,Precision@10
method,,,
I1 CLIP ViT-B/32,0.1723,6.2450,0.2984
H1 Hybrid (metadata+CLIP),0.1803,5.3410,0.3698
B2 Content (emotion+style),0.3328,8.4975,0.0433
B1 Random within style,0.3465,8.9680,0.0261
B3 Item-kNN (collaborative),0.3766,9.5805,0.0095
B4 Implicit ALS (latent factors),0.3846,9.7425,0.0050
B0 Random,0.3921,9.8445,0.0041



Caveat: intra-list distance is measured in CLIP space, which is the space I1 ranks
in — so I1 is judged by its own metric and is expected to look least diverse. The
distinct-artist column is representation-independent and is the honest cross-check.


### 9.4 The hybrid weight, chosen on queries it is not scored on

§9.2's sweep peaks at α = 0.25, but on the same 2,000 queries it is scored on, which makes
0.4086 a description of that sweep rather than a result. §6.3 fixes the rule: split the queries
once into a *tuning* half and a *scoring* half (`SEED_I2I + 1`), choose α on the first, and
re-score **every** Track B method on the second, so the tuned blend meets I1, B2 and the
pre-committed α = 0.5 on queries it has never seen. A 0.05 grid is fine enough — the §9.2 curve
is smooth and single-peaked — and the table below is on 1,000 queries, so its numbers are not on
the same scale as the 2,000-query ladder.


In [61]:
# §9.4 — the hybrid weight on held-out queries =========================
if E_clip is None:
    print("skipped — no embeddings; see §9 above.")
else:
    rng_split = np.random.default_rng(SEED_I2I + 1)
    perm      = rng_split.permutation(len(query_ids))
    tune_q    = np.sort(query_ids[perm[:N_QUERIES // 2]])
    score_q   = np.sort(query_ids[perm[N_QUERIES // 2:]])
    assert not set(tune_q.tolist()) & set(score_q.tolist())

    def blended_lists(queries, alpha, k=K):
        """§9.2's Track B blend, for any query subset and any alpha."""
        s = (alpha * (W_i2i_n[queries] @ W_i2i_n.T)
             + (1 - alpha) * (E_clip_n[queries] @ E_clip_n.T)).astype(np.float32)
        s[np.arange(len(queries)), queries] = -np.inf
        return {int(q): top_k(s[row], np.array([q]), k, tie_seed=int(q))
                for row, q in enumerate(queries)}

    def mean_ndcg(lists):
        return float(np.mean([evaluate_query(r, q)["ndcg"] for q, r in lists.items()]))

    ALPHA_GRID = np.round(np.arange(0.0, 1.0001, 0.05), 2)
    tune_curve = pd.Series({a: mean_ndcg(blended_lists(tune_q, a)) for a in ALPHA_GRID},
                           name=f"NDCG@{K}, tuning half")
    ALPHA_STAR = float(tune_curve.idxmax())
    print(f"tuning half : {len(tune_q):,} queries   scoring half : {len(score_q):,} queries")
    print(f"alpha*      : {ALPHA_STAR:.2f}   (tuning-half NDCG@{K} {tune_curve.max():.4f})\n")
    with pd.option_context("display.max_columns", 30):
        display(tune_curve.round(4).to_frame().T)

    # every Track B method on the scoring half only, so the tuned row is like-for-like
    i2i_heldout = {name: {int(q): lists[int(q)] for q in score_q}
                   for name, lists in i2i_lists.items()}
    i2i_heldout[f"H1* Hybrid (alpha={ALPHA_STAR:.2f}, tuned)"] = blended_lists(score_q, ALPHA_STAR)

    rows = []
    for name, lists in i2i_heldout.items():
        m = pd.DataFrame([evaluate_query(r, q) for q, r in lists.items()]).mean()
        rows.append({"algorithm": name, f"NDCG@{K}": m.ndcg, f"Precision@{K}": m.precision,
                     f"MRR@{K}": m.mrr, "Same-style share": m.same_style_share})
    heldout_table = (pd.DataFrame(rows).set_index("algorithm")
                     .sort_values(f"NDCG@{K}", ascending=False))
    print(f"\nTrack B on the {len(score_q):,} scoring queries only — "
          "not comparable to the 2,000-query tables above")
    display(heldout_table.round(4))


tuning half : 1,000 queries   scoring half : 1,000 queries
alpha*      : 0.25   (tuning-half NDCG@10 0.4060)



,0.00,0.05,0.10,0.15,0.20,0.25,0.30,0.35,0.40,0.45,0.50,0.55,0.60,0.65,0.70,0.75,0.80,0.85,0.90,0.95,1.00
"NDCG@10, tuning half",0.3315,0.403,0.4046,0.4053,0.4053,0.406,0.4053,0.405,0.4051,0.4031,0.4001,0.396,0.3918,0.3853,0.3785,0.3647,0.3502,0.3264,0.2896,0.2202,0.0433



Track B on the 1,000 scoring queries only — not comparable to the 2,000-query tables above


,NDCG@10,Precision@10,MRR@10,Same-style share
algorithm,,,,
"H1* Hybrid (alpha=0.25, tuned)",0.4112,0.3810,0.6287,1.0000
H1 Hybrid (metadata+CLIP),0.3999,0.3724,0.6036,1.0000
I1 CLIP ViT-B/32,0.3296,0.3001,0.5415,0.6287
B2 Content (emotion+style),0.0429,0.0428,0.1014,1.0000
B1 Random within style,0.0260,0.0262,0.0704,1.0000
B3 Item-kNN (collaborative),0.0107,0.0097,0.0323,0.3554
B4 Implicit ALS (latent factors),0.0056,0.0055,0.0166,0.1675
B0 Random,0.0038,0.0040,0.0106,0.1471


### 9.5 Confidence intervals and paired tests on every rung

§7.7 called this the most valuable addition left. Two numbers per rung, both computed from the
per-user (Track A, protocol 2) and per-query (Track B, protocol 3) NDCG@10 frames:

- a **95% percentile-bootstrap interval** on the mean — 2,000 resamples of users or queries,
  `SEED_BOOT`;
- a **paired Wilcoxon signed-rank test** against the rung below — paired, because every method
  is scored on the same users and the same queries, so the between-user variance that dominates
  the intervals cancels in the differences. The share of users (queries) on which the rung wins
  and loses is printed next to the p-value, because a p-value of 10⁻³⁰ at n = 11,695 says
  nothing about *how many* people the rung helped.

The test is written in numpy: zero differences are
dropped, absolute differences are ranked with average ranks, and the statistic is compared to
its normal approximation with the tie correction — which is what scipy does above n = 50.

The reading rule from §5.0: a rung whose interval overlaps the rung below's is *reported*, not
*claimed* — the last column says which, and the sign of Δ says whether a separated rung is a
claimed gain or a claimed loss. The fourth table applies the same test to §9.4's tuned hybrid on
its scoring half; the last three repeat the exercise on protocol 1 and on §9.3's two within-list
diversity measures, where the pairing is the ladder's and a negative Δ means *less* diverse than
the rung below. Coverage gets no interval — it is a union over users, with no per-unit
decomposition to resample.


In [62]:
# §9.5 — bootstrap CIs and paired Wilcoxon tests on every rung ===========
import math
SEED_BOOT, N_BOOT = 2026, 2_000

def bootstrap_ci(x, n_boot=N_BOOT, seed=SEED_BOOT, chunk=250):
    """95% percentile interval for the mean of x, resampling users / queries with replacement."""
    x = np.asarray(x, float)
    rng_bs = np.random.default_rng(seed)
    means = np.concatenate([
        x[rng_bs.integers(0, len(x), size=(min(chunk, n_boot - s), len(x)))].mean(axis=1)
        for s in range(0, n_boot, chunk)])
    return float(np.quantile(means, 0.025)), float(np.quantile(means, 0.975))

def average_ranks(x):
    """1-based ranks with ties averaged — scipy.stats.rankdata(method='average')."""
    order = np.argsort(x, kind="stable")
    _, inverse, counts = np.unique(x[order], return_inverse=True, return_counts=True)
    first = np.cumsum(counts) - counts + 1
    ranks = np.empty(len(x)); ranks[order] = (first + (counts - 1) / 2)[inverse]
    return ranks

def wilcoxon_paired(a, b):
    """Two-sided paired Wilcoxon signed-rank test of a vs b. Zeros dropped, average ranks,
    normal approximation with tie correction (scipy's defaults for n > 50). Returns (z, p)."""
    d = np.asarray(a, float) - np.asarray(b, float)
    d = d[d != 0]
    n = len(d)
    if n == 0:
        return 0.0, 1.0
    r = average_ranks(np.abs(d))
    w_plus = r[d > 0].sum()
    mu = n * (n + 1) / 4
    _, t = np.unique(np.abs(d), return_counts=True)
    sigma = math.sqrt(n * (n + 1) * (2 * n + 1) / 24 - (t ** 3 - t).sum() / 48)
    z = (w_plus - mu) / sigma
    # erfc underflows to 0.0 beyond |z| ~ 38; floor it so a table never shows a literal zero
    return float(z), max(float(math.erfc(abs(z) / math.sqrt(2))), 1e-300)

def ladder_with_uncertainty(frames, ladder, label=f"NDCG@{K}"):
    """`frames`: name -> per-unit NDCG Series indexed by user / query id.
    `ladder`: [(rung, rung below or None)]. One row per rung."""
    rows, ci = [], {}
    for name, below in ladder:
        x = frames[name]
        ci[name] = bootstrap_ci(x)
        row = {"rung": name, label: x.mean(),
               "CI low": ci[name][0], "CI high": ci[name][1], "vs": below or "—"}
        if below is not None:
            y = frames[below].loc[x.index]
            z, p = wilcoxon_paired(x, y)
            row.update({"Δ": x.mean() - y.mean(), "Wilcoxon p": p,
                        "wins": float((x > y).mean()), "losses": float((x < y).mean()),
                        "CIs vs below": "separate" if (ci[name][0] > ci[below][1] or
                                                      ci[name][1] < ci[below][0]) else "overlap"})
        rows.append(row)
    return pd.DataFrame(rows).set_index("rung")

pd.set_option("display.float_format", lambda v: f"{v:.4f}" if abs(v) >= 1e-4 or v == 0 else f"{v:.1e}")

# ---- Track A, protocol 2: per-user NDCG@10 for every rung -------------------------------
P2_SCORERS = {
    "Random":                   lambda u, c: np.random.default_rng(SEED_EVAL + u).random(len(c)),
    "Popularity":               lambda u, c: popularity[c],
    "Content (metadata)":       lambda u, c: user_profiles_n[u] @ W_user_n[c].T,
    "Content (ridge)":          lambda u, c: W_ridge[c] @ B_ridge[u],
    "Quality (raw rate)":       lambda u, c: raw_rate[c],
    "Quality (posterior mean)": lambda u, c: posterior_mean[c],
    "Quality (lower bound)":    lambda u, c: lower_bound[c],
    "Item-kNN (weighted sum)":  knn_scores_for,
    "Implicit ALS":             lambda u, c: Y_als[c] @ X_als[u],
    "Hybrid (ridge+ALS)":       hybrid_score_fn(HYB_ALPHA),
    "BPR":                      lambda u, c: Y_bpr[c] @ X_bpr[u],
}
LADDER_A = [("Random", None), ("Popularity", "Random"), ("Content (metadata)", "Popularity"),
            ("Content (ridge)", "Content (metadata)"),            # 2b: same signal, other estimator
            ("Quality (raw rate)", "Content (metadata)"),
            ("Quality (posterior mean)", "Quality (raw rate)"),
            ("Quality (lower bound)", "Quality (posterior mean)"),
            ("Item-kNN (weighted sum)", "Quality (lower bound)"),
            ("Implicit ALS", "Item-kNN (weighted sum)"),          # 6b: same signal, other estimator
            ("Hybrid (ridge+ALS)", "Implicit ALS"),                # 7: 2b and 6b blended
            ("BPR", "Implicit ALS")]                                # 6c: same signal, pairwise objective
if E_clip is not None:
    P2_SCORERS["Content (CLIP pixels)"] = lambda u, c: clip_profiles_n[u] @ E_clip_n[c].T
    LADDER_A.insert(1, ("Content (CLIP pixels)", "Random"))     # outside the ladder, vs Random

p2_frames = {name: evaluate_reranking(fn, frame=True)["ndcg"] for name, fn in P2_SCORERS.items()}
ladder_a = ladder_with_uncertainty(p2_frames, LADDER_A)
top_rung   = max(p2_frames, key=lambda n: p2_frames[n].mean())
gap_closed = ((p2_frames[top_rung].mean() - p2_frames["Random"].mean())
              / (ORACLE_NDCG_P2 - p2_frames["Random"].mean()))
print(f"Track A — protocol 2, {len(p2_frames['Random']):,} users, "
      f"oracle ceiling {ORACLE_NDCG_P2:.4f}; the top rung ({top_rung}) closes "
      f"{gap_closed:.1%} of the random-to-oracle gap\n")
display(ladder_a)

# ---- the three fitted estimators head to head, each against the ridge ------------------
# The ladder pairs adjacent rungs only; rung 2b's tie with rung 6b needs the direct test.
HEAD_TO_HEAD = [("Content (ridge)", None), ("Item-kNN (weighted sum)", "Content (ridge)"),
                ("Implicit ALS", "Content (ridge)"), ("Hybrid (ridge+ALS)", "Content (ridge)"),
                ("BPR", "Content (ridge)")]
print("\nTrack A — protocol 2, behaviour against the metadata ridge, paired on the same users\n")
display(ladder_with_uncertainty(p2_frames, HEAD_TO_HEAD))

# ---- Track B, protocol 3: per-query NDCG@10 for every rung ------------------------------
p3_frames = {name: pd.Series({q: evaluate_query(r, q)["ndcg"] for q, r in lists.items()})
             for name, lists in i2i_lists.items()}
LADDER_B = [("B0 Random", None), ("B1 Random within style", "B0 Random"),
            ("B2 Content (emotion+style)", "B1 Random within style"),
            ("B3 Item-kNN (collaborative)", "B2 Content (emotion+style)"),
            ("B4 Implicit ALS (latent factors)", "B3 Item-kNN (collaborative)")]   # 3b
if E_clip is not None:
    LADDER_B += [("I1 CLIP ViT-B/32", "B2 Content (emotion+style)"),
                 ("H1 Hybrid (metadata+CLIP)", "I1 CLIP ViT-B/32")]
ladder_b = ladder_with_uncertainty(p3_frames, LADDER_B)
print(f"\nTrack B — protocol 3, {len(p3_frames['B0 Random']):,} queries\n")
display(ladder_b)

# ---- §9.4's tuned hybrid, on its scoring half only -------------------------------------
if E_clip is not None:
    h1_star = [n for n in i2i_heldout if n.startswith("H1*")][0]
    ho_frames = {name: pd.Series({q: evaluate_query(r, q)["ndcg"] for q, r in lists.items()})
                 for name, lists in i2i_heldout.items()}
    ladder_h = ladder_with_uncertainty(
        ho_frames, [("B2 Content (emotion+style)", None), ("I1 CLIP ViT-B/32", None),
                    ("H1 Hybrid (metadata+CLIP)", "I1 CLIP ViT-B/32"),
                    (h1_star, "H1 Hybrid (metadata+CLIP)")])
    print(f"\nTrack B — the {len(ho_frames[h1_star]):,} scoring queries of §9.4 only\n")
    display(ladder_h)

# ---- P1 (full catalogue), from the stored top-10 lists -------------------------------
# The same rungs, the same pairing, on the exposure-biased protocol. Coverage has no per-user
# decomposition — it is a union over users — so it gets no interval here.
pd.set_option("display.float_format", lambda v: f"{v:.4f}" if abs(v) >= 1e-4 or v == 0 else f"{v:.1e}")
p1_frames = {name: pd.Series({u: evaluate_user(r, test_positives[u])["ndcg"]
                              for u, r in lists.items()})
             for name, lists in recommendation_lists.items()}
ladder_a_p1 = ladder_with_uncertainty(p1_frames, [(n, b) for n, b in LADDER_A if n in p1_frames])
print(f"\nTrack A — protocol 1, {len(p1_frames['Random']):,} users, full-catalogue ranking\n")
display(ladder_a_p1)
print("\nTrack A — protocol 1, the same head-to-head\n")
display(ladder_with_uncertainty(p1_frames, HEAD_TO_HEAD))

# ---- §9.3's within-list diversity, per query --------------------------------------------
if E_clip is not None:
    ild_frames = {name: pd.Series({q: intra_list_distance(r) for q, r in lists.items()})
                  for name, lists in i2i_lists.items()}
    art_frames = {name: pd.Series({q: distinct_artists(r) for q, r in lists.items()})
                  for name, lists in i2i_lists.items()}
    print("\nTrack B — intra-list distance (CLIP space), same pairing as the ladder; "
          "Δ < 0 means the rung is *less* diverse\n")
    display(ladder_with_uncertainty(ild_frames, LADDER_B, label="intra-list distance"))
    print("\nTrack B — distinct artists per top-10\n")
    display(ladder_with_uncertainty(art_frames, LADDER_B, label="distinct artists / 10"))
pd.reset_option("display.float_format")


Track A — protocol 2, 11,695 users, oracle ceiling 0.8919; the top rung (Implicit ALS) closes 40.2% of the random-to-oracle gap



,NDCG@10,CI low,CI high,vs,Δ,Wilcoxon p,wins,losses,CIs vs below
rung,,,,,,,,,
Random,0.4015,0.3981,0.4054,—,NaN,NaN,NaN,NaN,NaN
Content (CLIP pixels),0.4159,0.4120,0.4197,Random,0.0144,1.2e-08,0.5077,0.4652,separate
Popularity,0.4552,0.4513,0.4592,Random,0.0537,3.8e-106,0.5753,0.3991,separate
Content (metadata),0.5187,0.5147,0.5227,Popularity,0.0635,2.3e-139,0.5857,0.3961,separate
Content (ridge),0.5972,0.5935,0.6012,Content (metadata),0.0786,1.0e-300,0.6526,0.2864,separate
Quality (raw rate),0.5244,0.5201,0.5286,Content (metadata),0.0057,0.0149,0.4970,0.4833,overlap
Quality (posterior mean),0.5317,0.5276,0.5360,Quality (raw rate),0.0073,3.6e-41,0.2650,0.1951,overlap
Quality (lower bound),0.5385,0.5343,0.5430,Quality (posterior mean),0.0068,5.9e-17,0.4899,0.4091,overlap
Item-kNN (weighted sum),0.5717,0.5675,0.5761,Quality (lower bound),0.0332,1.1e-63,0.5347,0.4203,separate



Track A — protocol 2, behaviour against the metadata ridge, paired on the same users



,NDCG@10,CI low,CI high,vs,Δ,Wilcoxon p,wins,losses,CIs vs below
rung,,,,,,,,,
Content (ridge),0.5972,0.5935,0.6012,—,NaN,NaN,NaN,NaN,NaN
Item-kNN (weighted sum),0.5717,0.5675,0.5761,Content (ridge),-0.0255,6.4e-25,0.4482,0.5328,separate
Implicit ALS,0.5984,0.5944,0.6025,Content (ridge),0.0012,0.5801,0.4751,0.4835,overlap



Track B — protocol 3, 2,000 queries



...

,NDCG@10,CI low,CI high,vs,Δ,Wilcoxon p,wins,losses,CIs vs below
rung,,,,,,,,,
Content (ridge),0.0076,0.0070,0.0083,—,NaN,NaN,NaN,NaN,NaN
Item-kNN (weighted sum),0.0986,0.0958,0.1015,Content (ridge),0.0909,1.0e-300,0.4375,0.0184,separate
Implicit ALS,0.1206,0.1174,0.1238,Content (ridge),0.1129,1.0e-300,0.5120,0.0158,separate



Track B — intra-list distance (CLIP space), same pairing as the ladder; Δ < 0 means the rung is *less* diverse



,intra-list distance,CI low,CI high,vs,Δ,Wilcoxon p,wins,losses,CIs vs below
rung,,,,,,,,,
B0 Random,0.3921,0.3908,0.3934,—,NaN,NaN,NaN,NaN,NaN
B1 Random within style,0.3465,0.3450,0.3480,B0 Random,-0.0456,9.6e-241,0.1460,0.8540,separate
B2 Content (emotion+style),0.3328,0.3312,0.3345,B1 Random within style,-0.0137,3.0e-50,0.3620,0.6380,separate
B3 Item-kNN (collaborative),0.3766,0.3750,0.3780,B2 Content (emotion+style),0.0438,2.8e-212,0.8215,0.1785,separate
B4 Implicit ALS (latent factors),0.3846,0.3833,0.3860,B3 Item-kNN (collaborative),0.0081,2.2e-13,0.5700,0.4300,separate
I1 CLIP ViT-B/32,0.1723,0.1707,0.1739,B2 Content (emotion+style),-0.1606,1.0e-300,0.0015,0.9985,separate
H1 Hybrid (metadata+CLIP),0.1803,0.1787,0.1821,I1 CLIP ViT-B/32,0.0081,7.6e-91,0.6915,0.3025,separate



Track B — distinct artists per top-10



,distinct artists / 10,CI low,CI high,vs,Δ,Wilcoxon p,wins,losses,CIs vs below
rung,,,,,,,,,
B0 Random,9.8445,9.8275,9.8615,—,NaN,NaN,NaN,NaN,NaN
B1 Random within style,8.9680,8.9240,9.0130,B0 Random,-0.8765,4.9e-182,0.0495,0.5890,separate
B2 Content (emotion+style),8.4975,8.4440,8.5550,B1 Random within style,-0.4705,2.9e-43,0.2390,0.4615,separate
B3 Item-kNN (collaborative),9.5805,9.5480,9.6105,B2 Content (emotion+style),1.0830,8.4e-174,0.6360,0.0930,separate
B4 Implicit ALS (latent factors),9.7425,9.7190,9.7655,B3 Item-kNN (collaborative),0.1620,1.0e-15,0.2530,0.1635,separate
I1 CLIP ViT-B/32,6.2450,6.1340,6.3550,B2 Content (emotion+style),-2.2525,1.6e-199,0.1405,0.7240,separate
H1 Hybrid (metadata+CLIP),5.3410,5.2385,5.4425,I1 CLIP ViT-B/32,-0.9040,7.3e-108,0.1540,0.5495,separate
